In [ ]:
# ============================================================
# ЯЧЕЙКА 1. Установка зависимостей
# ============================================================

!pip install -q \
    pandas \
    numpy \
    scipy \
    statsmodels \
    scikit-learn \
    sentence-transformers \
    pymorphy3 \
    pymorphy3-dicts-ru \
    razdel \
    plotly \
    matplotlib \
    seaborn \
    openpyxl \
    beautifulsoup4 \
    pyarrow \
    fastparquet \
    tqdm \
    scikit-posthocs \
    kaleido

print("✅ Зависимости установлены.")
print("Следующая ячейка: импорты, настройки путей и создание структуры /content/output.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 3.5 MB/s eta 0:00:00
✅ Зависимости установлены.
Следующая ячейка: импорты, настройки путей и создание структуры /content/output.


In [ ]:
# ============================================================
# ЯЧЕЙКА 2. Импорты, настройки путей и структура проекта
# ============================================================

import os
import re
import json
import math
import hashlib
import unicodedata
import warnings
import shutil
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from bs4 import BeautifulSoup
from tqdm.auto import tqdm

import scipy.stats as stats
from scipy.spatial.distance import cosine

import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore")
tqdm.pandas()

# Белый фон для графиков, без серой seaborn-стилизации по умолчанию
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "axes.grid": False,
    "font.size": 11,
})

# ------------------------------------------------------------
# Пути
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content")
INPUT_FILE = PROJECT_ROOT / "hh_big_creative_vacancies_html.xlsx"

OUTPUT_DIR = PROJECT_ROOT / "output"
DATA_DIR = OUTPUT_DIR / "data"
STATS_DIR = OUTPUT_DIR / "stats"
VALIDATION_DIR = OUTPUT_DIR / "validation"
FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR = OUTPUT_DIR / "tables_for_thesis"
REFERENCE_DIR = OUTPUT_DIR / "reference"

for folder in [
    OUTPUT_DIR,
    DATA_DIR,
    STATS_DIR,
    VALIDATION_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    REFERENCE_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Базовые константы исследования
# ------------------------------------------------------------

RANDOM_STATE = 42
BOOTSTRAP_B = 5000

EXPECTED_COLUMNS = [
    "creative_segment",
    "query_keyword",
    "vacancy_id",
    "url",
    "name",
    "company",
    "salary_text",
    "area_text",
    "experience_text",
    "schedule_text",
    "description",
]

SUPPORTIVE_INDEXES = [
    "autonomy",
    "competence",
    "relatedness",
    "meaning",
    "authorship",
]

CONTROL_INDEXES = [
    "output_control",
    "behavior_control",
    "intensity",
]

ALL_CONSTRUCTS = [
    "autonomy",
    "competence",
    "relatedness",
    "meaning",
    "authorship",
    "output_control",
    "behavior_control",
    "intensity",
    "selfbrand",
    "ai_empowerment",
    "ai_intensification",
]

print("✅ Импорты выполнены.")
print(f"📁 INPUT_FILE: {INPUT_FILE}")
print(f"📁 OUTPUT_DIR: {OUTPUT_DIR}")
print(f"📁 Созданные папки: data, stats, validation, figures, tables_for_thesis, reference")

✅ Импорты выполнены.
📁 INPUT_FILE: /content/hh_big_creative_vacancies_html.xlsx
📁 OUTPUT_DIR: /content/output
📁 Созданные папки: data, stats, validation, figures, tables_for_thesis, reference


In [ ]:
# ============================================================
# ЯЧЕЙКА 3. Загрузка Excel, базовая валидация и отчёт о пропусках
# ============================================================

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Файл не найден: {INPUT_FILE}\n"
        "Загрузите hh_big_creative_vacancies_html.xlsx в левую панель Colab "
        "или через кнопку Files → Upload."
    )

df_raw = pd.read_excel(INPUT_FILE)

# Нормализуем названия колонок
df_raw.columns = [str(c).strip() for c in df_raw.columns]

missing_columns = [c for c in EXPECTED_COLUMNS if c not in df_raw.columns]
extra_columns = [c for c in df_raw.columns if c not in EXPECTED_COLUMNS]

if missing_columns:
    raise ValueError(f"В файле отсутствуют обязательные столбцы: {missing_columns}")

# Оставляем ожидаемые колонки в фиксированном порядке
df_raw = df_raw[EXPECTED_COLUMNS].copy()

# Приводим текстовые поля к строковому типу, но NaN пока сохраняем как пустые строки только там, где нужно
for col in EXPECTED_COLUMNS:
    if col != "vacancy_id":
        df_raw[col] = df_raw[col].astype("string")

# Отчёт о пропусках
missing_report = (
    pd.DataFrame({
        "column": df_raw.columns,
        "missing_count": df_raw.isna().sum().values,
        "missing_share": df_raw.isna().mean().values,
        "non_missing_count": df_raw.notna().sum().values,
        "dtype": [str(df_raw[c].dtype) for c in df_raw.columns],
    })
    .sort_values("missing_share", ascending=False)
    .reset_index(drop=True)
)

# Базовые частотные таблицы
segment_counts = (
    df_raw["creative_segment"]
    .fillna("не указано")
    .value_counts(dropna=False)
    .rename_axis("creative_segment")
    .reset_index(name="n")
)

keyword_counts = (
    df_raw["query_keyword"]
    .fillna("не указано")
    .value_counts(dropna=False)
    .rename_axis("query_keyword")
    .reset_index(name="n")
)

area_counts = (
    df_raw["area_text"]
    .fillna("не указано")
    .value_counts(dropna=False)
    .head(50)
    .rename_axis("area_text")
    .reset_index(name="n")
)

# Сохраняем один Excel-файл для раздела первичной валидации
validation_report_path = VALIDATION_DIR / "01_data_quality_report.xlsx"

with pd.ExcelWriter(validation_report_path, engine="openpyxl") as writer:
    missing_report.to_excel(writer, sheet_name="missing_report", index=False)
    segment_counts.to_excel(writer, sheet_name="creative_segment", index=False)
    keyword_counts.to_excel(writer, sheet_name="query_keyword", index=False)
    area_counts.to_excel(writer, sheet_name="area_top50", index=False)

# Сохраняем исходный parquet для воспроизводимости
raw_parquet_path = DATA_DIR / "00_raw_loaded.parquet"
df_raw.to_parquet(raw_parquet_path, index=False)

print("✅ Файл загружен и проверен.")
print(f"df_raw shape: {df_raw.shape}")
print(f"Количество уникальных vacancy_id: {df_raw['vacancy_id'].nunique(dropna=True)}")
print(f"Лишние колонки в исходном файле: {extra_columns if extra_columns else 'нет'}")
print(f"Отчёт о пропусках сохранён: {validation_report_path}")
print(f"Raw parquet сохранён: {raw_parquet_path}")
display(missing_report)

✅ Файл загружен и проверен.
df_raw shape: (3305, 11)
Количество уникальных vacancy_id: 3305
Лишние колонки в исходном файле: нет
Отчёт о пропусках сохранён: /content/output/validation/01_data_quality_report.xlsx
Raw parquet сохранён: /content/output/data/00_raw_loaded.parquet


,column,missing_count,missing_share,non_missing_count,dtype
0,schedule_text,3305,1.000000,0,string
1,salary_text,3175,0.960666,130,string
2,name,3127,0.946142,178,string
3,area_text,622,0.188200,2683,string
4,experience_text,54,0.016339,3251,string
5,company,5,0.001513,3300,string
6,description,3,0.000908,3302,string
7,vacancy_id,0,0.000000,3305,int64
8,query_keyword,0,0.000000,3305,string
9,creative_segment,0,0.000000,3305,string


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=missing_report)

https://docs.google.com/spreadsheets/d/1NkGZfSEfhAayZHE8RIRxIX7EFUiTbeObq3De1SMJcwU/edit#gid=0


In [ ]:
# ============================================================
# ЯЧЕЙКА 4. Дедупликация по vacancy_id и near-duplicate по компании + первым 500 символам description
# ============================================================

df = df_raw.copy()

# ------------------------------------------------------------
# 1. Дедупликация по vacancy_id
# ------------------------------------------------------------

before_vacancy_id = len(df)

# Если vacancy_id пустой, такие строки не схлопываем между собой.
# Для непустых vacancy_id оставляем первое вхождение.
df["_vacancy_id_is_missing"] = df["vacancy_id"].isna() | (df["vacancy_id"].astype(str).str.strip() == "")

df_with_id = df[~df["_vacancy_id_is_missing"]].drop_duplicates(subset=["vacancy_id"], keep="first")
df_without_id = df[df["_vacancy_id_is_missing"]].copy()

df = pd.concat([df_with_id, df_without_id], ignore_index=True)

after_vacancy_id = len(df)
removed_by_vacancy_id = before_vacancy_id - after_vacancy_id

# ------------------------------------------------------------
# 2. Near-duplicate: одна компания + хэш первых 500 символов description
# ------------------------------------------------------------

def normalize_for_hash(text):
    """Минимальная нормализация текста для поиска near-duplicate."""
    if pd.isna(text):
        return ""
    text = str(text)
    text = BeautifulSoup(text, "html.parser").get_text(" ")
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text[:500]

def make_hash(text):
    """MD5-хэш строки для компактного сравнения."""
    return hashlib.md5(text.encode("utf-8")).hexdigest()

before_near_dup = len(df)

df["_company_norm"] = (
    df["company"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df["_description_head_norm"] = df["description"].progress_apply(normalize_for_hash)
df["_description_head_hash"] = df["_description_head_norm"].apply(make_hash)

# Если description пустой, near-duplicate не применяем
df["_near_dup_key_is_valid"] = (
    (df["_company_norm"] != "") &
    (df["_description_head_norm"].str.len() > 0)
)

df_valid_key = (
    df[df["_near_dup_key_is_valid"]]
    .drop_duplicates(subset=["_company_norm", "_description_head_hash"], keep="first")
)

df_invalid_key = df[~df["_near_dup_key_is_valid"]].copy()

df_dedup = pd.concat([df_valid_key, df_invalid_key], ignore_index=True)

after_near_dup = len(df_dedup)
removed_by_near_dup = before_near_dup - after_near_dup

# ------------------------------------------------------------
# 3. Отчёт по дедупликации
# ------------------------------------------------------------

dedup_report = pd.DataFrame({
    "step": [
        "initial_rows",
        "after_vacancy_id_dedup",
        "after_company_description_head_hash_dedup",
    ],
    "rows": [
        before_vacancy_id,
        after_vacancy_id,
        after_near_dup,
    ],
    "removed_at_step": [
        0,
        removed_by_vacancy_id,
        removed_by_near_dup,
    ]
})

dedup_report_path = VALIDATION_DIR / "02_deduplication_report.xlsx"
with pd.ExcelWriter(dedup_report_path, engine="openpyxl") as writer:
    dedup_report.to_excel(writer, sheet_name="dedup_summary", index=False)

# Удаляем технические поля, кроме тех, что могут понадобиться для аудита
technical_cols_to_drop = [
    "_vacancy_id_is_missing",
    "_company_norm",
    "_description_head_norm",
    "_near_dup_key_is_valid",
]

df_dedup = df_dedup.drop(columns=technical_cols_to_drop)

dedup_parquet_path = DATA_DIR / "01_deduplicated.parquet"
df_dedup.to_parquet(dedup_parquet_path, index=False)

print("✅ Дедупликация выполнена.")
print(f"До дедупликации: {before_vacancy_id}")
print(f"Удалено по vacancy_id: {removed_by_vacancy_id}")
print(f"Удалено near-duplicate по company + hash первых 500 символов description: {removed_by_near_dup}")
print(f"df_dedup shape: {df_dedup.shape}")
print(f"Отчёт сохранён: {dedup_report_path}")
print(f"Parquet сохранён: {dedup_parquet_path}")
display(dedup_report)

In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=dedup_report)

In [ ]:
# ============================================================
# ЯЧЕЙКА 5. Очистка description и удаление вакансий короче 200 символов
# ============================================================

# Автономность ячейки: если df_dedup не в памяти, загружаем с диска
if "df_dedup" not in globals():
    dedup_parquet_path = DATA_DIR / "01_deduplicated.parquet"
    if not dedup_parquet_path.exists():
        raise FileNotFoundError("Не найден 01_deduplicated.parquet. Сначала запустите ячейки 1–4.")
    df_dedup = pd.read_parquet(dedup_parquet_path)

df_clean = df_dedup.copy()

def clean_html_text(text, lower=False):
    """
    Очистка HTML-описания вакансии:
    1) BeautifulSoup убирает HTML-разметку;
    2) NFKC-нормализация приводит символы к стандартному виду;
    3) сжатие пробелов;
    4) lower=True — версия для лексиконного матчинга.
    """
    if pd.isna(text):
        return ""

    text = str(text)
    text = BeautifulSoup(text, "html.parser").get_text(" ")
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text).strip()

    if lower:
        text = text.lower()

    return text

df_clean["description_original"] = df_clean["description"].progress_apply(
    lambda x: clean_html_text(x, lower=False)
)

df_clean["description_lower"] = df_clean["description"].progress_apply(
    lambda x: clean_html_text(x, lower=True)
)

df_clean["description_len_chars"] = df_clean["description_original"].str.len()
df_clean["description_len_words_rough"] = df_clean["description_original"].str.split().str.len()

before_len_filter = len(df_clean)

df_clean = df_clean[df_clean["description_len_chars"] >= 200].copy()

after_len_filter = len(df_clean)
removed_short = before_len_filter - after_len_filter

# Для анализа используем query_keyword как базу профессии, потому что name почти пустой
df_clean["profession_source"] = df_clean["query_keyword"].fillna("не указано").astype(str).str.strip()

# Сохраняем отчёт
cleaning_report = pd.DataFrame({
    "metric": [
        "rows_before_length_filter",
        "rows_after_length_filter",
        "removed_short_descriptions",
        "min_description_len_chars_after_filter",
        "median_description_len_chars_after_filter",
        "max_description_len_chars_after_filter",
    ],
    "value": [
        before_len_filter,
        after_len_filter,
        removed_short,
        int(df_clean["description_len_chars"].min()),
        float(df_clean["description_len_chars"].median()),
        int(df_clean["description_len_chars"].max()),
    ]
})

cleaning_report_path = VALIDATION_DIR / "03_description_cleaning_report.xlsx"
with pd.ExcelWriter(cleaning_report_path, engine="openpyxl") as writer:
    cleaning_report.to_excel(writer, sheet_name="cleaning_summary", index=False)

clean_parquet_path = DATA_DIR / "02_cleaned_descriptions.parquet"
df_clean.to_parquet(clean_parquet_path, index=False)

print("✅ Очистка description выполнена.")
print(f"До фильтра длины: {before_len_filter}")
print(f"Удалено вакансий короче 200 символов: {removed_short}")
print(f"df_clean shape: {df_clean.shape}")
print(f"Отчёт сохранён: {cleaning_report_path}")
print(f"Parquet сохранён: {clean_parquet_path}")
display(cleaning_report)

In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=cleaning_report)

https://docs.google.com/spreadsheets/d/1_TlG-dP7DppnzKIMNO9fj4AXSRYUxZMAefg_SDb2_y4/edit#gid=0


In [ ]:
# ============================================================
# ЯЧЕЙКА 6. Парсинг salary_text
# ============================================================

# Автономность ячейки
if "df_clean" not in globals():
    clean_parquet_path = DATA_DIR / "02_cleaned_descriptions.parquet"
    if not clean_parquet_path.exists():
        raise FileNotFoundError("Не найден 02_cleaned_descriptions.parquet. Сначала запустите ячейки 1–5.")
    df_clean = pd.read_parquet(clean_parquet_path)

df_salary = df_clean.copy()

def parse_salary_text(s):
    """
    Парсинг зарплатного поля hh.ru.
    Важно: salary_text заполнен только у небольшой доли вакансий,
    поэтому зарплатный анализ будет разведочным.
    """
    result = {
        "salary_from": np.nan,
        "salary_to": np.nan,
        "salary_currency": np.nan,
        "salary_gross": np.nan,
        "salary_mid": np.nan,
        "salary_parse_note": "missing",
    }

    if pd.isna(s) or str(s).strip() == "":
        return pd.Series(result)

    text = str(s).lower()
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text).strip()

    # Валюта
    if re.search(r"₽|руб|rur|rub", text):
        currency = "RUB"
    elif re.search(r"\$|usd|доллар", text):
        currency = "USD"
    elif re.search(r"€|eur|евро", text):
        currency = "EUR"
    elif re.search(r"kzt|₸|тенге", text):
        currency = "KZT"
    else:
        currency = np.nan

    # Gross / net
    if re.search(r"до вычета|gross|гросс", text):
        gross = True
    elif re.search(r"на руки|net|нетто|после вычета", text):
        gross = False
    else:
        gross = np.nan

    # Убираем валютные обозначения и слова, оставляем числа
    # Поддержка форматов: 100 000, 100000, 100.000, 100,000
    number_pattern = r"(\d[\d\s.,]{2,})"
    numbers_raw = re.findall(number_pattern, text)

    def normalize_number(x):
        x = re.sub(r"[^\d]", "", x)
        if x == "":
            return np.nan
        return float(x)

    numbers = [normalize_number(x) for x in numbers_raw]
    numbers = [x for x in numbers if not pd.isna(x)]

    salary_from = np.nan
    salary_to = np.nan
    note = "parsed_no_bounds"

    # Явные маркеры от / до
    from_match = re.search(r"(?:от|from)\s*(\d[\d\s.,]{2,})", text)
    to_match = re.search(r"(?:до|to)\s*(\d[\d\s.,]{2,})", text)

    if from_match:
        salary_from = normalize_number(from_match.group(1))
    if to_match:
        salary_to = normalize_number(to_match.group(1))

    # Диапазон вида 100 000 - 150 000
    range_match = re.search(r"(\d[\d\s.,]{2,})\s*[-–—]\s*(\d[\d\s.,]{2,})", text)
    if pd.isna(salary_from) and pd.isna(salary_to) and range_match:
        salary_from = normalize_number(range_match.group(1))
        salary_to = normalize_number(range_match.group(2))

    # Если есть два числа, но явных маркеров не нашли — трактуем как диапазон
    if pd.isna(salary_from) and pd.isna(salary_to) and len(numbers) >= 2:
        salary_from = min(numbers[0], numbers[1])
        salary_to = max(numbers[0], numbers[1])

    # Если одно число и есть "от"
    if pd.isna(salary_from) and pd.isna(salary_to) and len(numbers) == 1:
        if re.search(r"\bот\b|from", text):
            salary_from = numbers[0]
        elif re.search(r"\bдо\b|to", text):
            salary_to = numbers[0]
        else:
            # Не считаем mid, потому что неизвестно: это нижняя, верхняя граница или фикс
            salary_from = numbers[0]
            note = "single_number_interpreted_as_from"

    if not pd.isna(salary_from) and not pd.isna(salary_to):
        salary_mid = (salary_from + salary_to) / 2
        note = "both_bounds"
    else:
        salary_mid = np.nan
        if note == "parsed_no_bounds":
            if not pd.isna(salary_from):
                note = "from_only"
            elif not pd.isna(salary_to):
                note = "to_only"
            else:
                note = "not_parsed"

    result.update({
        "salary_from": salary_from,
        "salary_to": salary_to,
        "salary_currency": currency,
        "salary_gross": gross,
        "salary_mid": salary_mid,
        "salary_parse_note": note,
    })

    return pd.Series(result)

salary_parsed = df_salary["salary_text"].progress_apply(parse_salary_text)
df_salary = pd.concat([df_salary, salary_parsed], axis=1)

salary_report = pd.DataFrame({
    "metric": [
        "rows_total",
        "salary_text_non_missing",
        "salary_text_non_missing_share",
        "salary_mid_available",
        "salary_mid_available_share",
        "rub_salary_mid_available",
    ],
    "value": [
        len(df_salary),
        int(df_salary["salary_text"].notna().sum()),
        float(df_salary["salary_text"].notna().mean()),
        int(df_salary["salary_mid"].notna().sum()),
        float(df_salary["salary_mid"].notna().mean()),
        int(((df_salary["salary_currency"] == "RUB") & df_salary["salary_mid"].notna()).sum()),
    ]
})

salary_parse_counts = (
    df_salary["salary_parse_note"]
    .value_counts(dropna=False)
    .rename_axis("salary_parse_note")
    .reset_index(name="n")
)

salary_report_path = VALIDATION_DIR / "04_salary_parsing_report.xlsx"
with pd.ExcelWriter(salary_report_path, engine="openpyxl") as writer:
    salary_report.to_excel(writer, sheet_name="salary_summary", index=False)
    salary_parse_counts.to_excel(writer, sheet_name="parse_notes", index=False)

salary_parquet_path = DATA_DIR / "03_salary_parsed.parquet"
df_salary.to_parquet(salary_parquet_path, index=False)

print("✅ Парсинг salary_text выполнен.")
print(f"df_salary shape: {df_salary.shape}")
print(f"Заполнено salary_text: {df_salary['salary_text'].notna().sum()} из {len(df_salary)}")
print(f"Доступен salary_mid: {df_salary['salary_mid'].notna().sum()} из {len(df_salary)}")
print("⚠️ Зарплатный анализ далее трактовать как разведочный: поле salary_text заполнено у малой доли вакансий.")
print(f"Отчёт сохранён: {salary_report_path}")
print(f"Parquet сохранён: {salary_parquet_path}")
display(salary_report)

  0%|          | 0/3121 [00:00<?, ?it/s]

✅ Парсинг salary_text выполнен.
df_salary shape: (3121, 23)
Заполнено salary_text: 126 из 3121
Доступен salary_mid: 58 из 3121
⚠️ Зарплатный анализ далее трактовать как разведочный: поле salary_text заполнено у малой доли вакансий.
Отчёт сохранён: /content/output/validation/04_salary_parsing_report.xlsx
Parquet сохранён: /content/output/data/03_salary_parsed.parquet


,metric,value
0,rows_total,3121.000000
1,salary_text_non_missing,126.000000
2,salary_text_non_missing_share,0.040372
3,salary_mid_available,58.000000
4,salary_mid_available_share,0.018584
5,rub_salary_mid_available,58.000000


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=salary_report)

https://docs.google.com/spreadsheets/d/1MtnONPo_dhiO_qqhbXq-z7vBB69rjMlV99ShenyDgV4/edit#gid=0


In [ ]:
# ============================================================
# ЯЧЕЙКА 7. Парсинг area_text, городов, удалёнки и city_group
# ============================================================

# Автономность ячейки
if "df_salary" not in globals():
    salary_parquet_path = DATA_DIR / "03_salary_parsed.parquet"
    if not salary_parquet_path.exists():
        raise FileNotFoundError("Не найден 03_salary_parsed.parquet. Сначала запустите ячейки 1–6.")
    df_salary = pd.read_parquet(salary_parquet_path)

df_geo = df_salary.copy()

million_plus_cities = {
    "новосибирск",
    "екатеринбург",
    "казань",
    "нижний новгород",
    "красноярск",
    "челябинск",
    "самара",
    "уфа",
    "ростов-на-дону",
    "омск",
    "краснодар",
    "воронеж",
    "пермь",
    "волгоград",
}

def extract_city(area_text):
    """
    По ТЗ: первый токен до запятой — город.
    """
    if pd.isna(area_text) or str(area_text).strip() == "":
        return "не указано"
    city = str(area_text).split(",")[0].strip()
    city = re.sub(r"\s+", " ", city)
    return city if city else "не указано"

def detect_remote(row):
    """
    Флаг удалёнки: ищем в area_text и description_lower.
    schedule_text по ТЗ игнорируем, потому что оно пустое.
    """
    area = "" if pd.isna(row.get("area_text")) else str(row.get("area_text")).lower()
    desc = "" if pd.isna(row.get("description_lower")) else str(row.get("description_lower")).lower()

    remote_patterns = [
        r"удален",
        r"удалён",
        r"remote",
        r"дистанцион",
        r"из дома",
        r"home office",
        r"можно работать из любой точки",
        r"работа из любой точки",
    ]

    combined = area + " " + desc
    return bool(any(re.search(p, combined) for p in remote_patterns))

def classify_city_group(city, is_remote):
    """
    Приоритет: удалённо → Москва → Санкт-Петербург → миллионники → прочие → не указано.
    """
    if is_remote:
        return "удалённо"

    if pd.isna(city) or str(city).strip() == "" or str(city).strip().lower() == "не указано":
        return "не указано"

    city_norm = str(city).strip().lower().replace("ё", "е")

    if city_norm == "москва":
        return "Москва"

    if city_norm in {"санкт-петербург", "санкт петербург", "спб", "петербург"}:
        return "Санкт-Петербург"

    million_norm = {c.replace("ё", "е") for c in million_plus_cities}
    if city_norm in million_norm:
        return "миллионники"

    return "прочие"

df_geo["city"] = df_geo["area_text"].apply(extract_city)
df_geo["is_remote"] = df_geo.progress_apply(detect_remote, axis=1)
df_geo["city_group"] = df_geo.apply(
    lambda row: classify_city_group(row["city"], row["is_remote"]),
    axis=1
)

city_group_counts = (
    df_geo["city_group"]
    .value_counts(dropna=False)
    .rename_axis("city_group")
    .reset_index(name="n")
)

city_counts = (
    df_geo["city"]
    .value_counts(dropna=False)
    .head(100)
    .rename_axis("city")
    .reset_index(name="n")
)

geo_report = pd.DataFrame({
    "metric": [
        "rows_total",
        "remote_count",
        "remote_share",
        "unique_cities",
    ],
    "value": [
        len(df_geo),
        int(df_geo["is_remote"].sum()),
        float(df_geo["is_remote"].mean()),
        int(df_geo["city"].nunique(dropna=True)),
    ]
})

geo_report_path = VALIDATION_DIR / "05_geo_parsing_report.xlsx"
with pd.ExcelWriter(geo_report_path, engine="openpyxl") as writer:
    geo_report.to_excel(writer, sheet_name="geo_summary", index=False)
    city_group_counts.to_excel(writer, sheet_name="city_group", index=False)
    city_counts.to_excel(writer, sheet_name="city_top100", index=False)

geo_parquet_path = DATA_DIR / "04_geo_parsed.parquet"
df_geo.to_parquet(geo_parquet_path, index=False)

print("✅ Парсинг area_text и city_group выполнен.")
print(f"df_geo shape: {df_geo.shape}")
print(f"Удалённые вакансии: {df_geo['is_remote'].sum()} из {len(df_geo)}")
print(f"Уникальных городов: {df_geo['city'].nunique(dropna=True)}")
print(f"Отчёт сохранён: {geo_report_path}")
print(f"Parquet сохранён: {geo_parquet_path}")
display(city_group_counts)

  0%|          | 0/3121 [00:00<?, ?it/s]

✅ Парсинг area_text и city_group выполнен.
df_geo shape: (3121, 26)
Удалённые вакансии: 892 из 3121
Уникальных городов: 285
Отчёт сохранён: /content/output/validation/05_geo_parsing_report.xlsx
Parquet сохранён: /content/output/data/04_geo_parsed.parquet


,city_group,n
0,удалённо,892
1,Москва,711
2,прочие,672
3,миллионники,375
4,Санкт-Петербург,255
5,не указано,216


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=city_group_counts)

In [ ]:
# ============================================================
# ЯЧЕЙКА 8. Маппинг query_keyword в profession_group
# ============================================================

# Автономность ячейки
if "df_geo" not in globals():
    geo_parquet_path = DATA_DIR / "04_geo_parsed.parquet"
    if not geo_parquet_path.exists():
        raise FileNotFoundError("Не найден 04_geo_parsed.parquet. Сначала запустите ячейки 1–7.")
    df_geo = pd.read_parquet(geo_parquet_path)

df_prof = df_geo.copy()

# Явный словарь правил.
# Важно: более специфичные паттерны стоят раньше общих.
profession_rules = [
    # architecture_design
    ("architecture_design", [
        "ландшафтный дизайнер",
        "дизайнер интерьера",
        "дизайнер одежды",
        "промышленный дизайнер",
        "архитектор",
    ]),

    # production_design
    ("production_design", [
        "художник-постановщик",
        "художник постановщик",
        "декоратор",
        "костюмер",
        "гример",
        "гримёр",
    ]),

    # art_director
    ("art_director", [
        "креативный директор",
        "арт-директор",
        "арт директор",
    ]),

    # game_design
    ("game_design", [
        "game designer",
        "геймдизайнер",
        "game-дизайнер",
    ]),

    # design_visual
    ("design_visual", [
        "графический дизайнер",
        "бренд-дизайнер",
        "бренд дизайнер",
        "ux/ui дизайнер",
        "ui/ux дизайнер",
        "ux дизайнер",
        "ui дизайнер",
        "motion designer",
        "моушн дизайнер",
        "motion-дизайнер",
        "3d художник",
        "3d artist",
        "concept artist",
        "концепт-художник",
        "game artist",
        "иллюстратор",
        "дизайнер",
    ]),

    # writing_editorial
    ("writing_editorial", [
        "копирайтер",
        "редактор",
        "сценарист",
        "драматург",
    ]),

    # smm_content
    ("smm_content", [
        "smm-специалист",
        "smm специалист",
        "smm",
        "контент-менеджер",
        "контент менеджер",
        "контент-креатор",
        "контент креатор",
    ]),

    # marketing_pr
    ("marketing_pr", [
        "pr-менеджер",
        "pr менеджер",
        "пиар-менеджер",
        "пиар менеджер",
        "маркетолог",
    ]),

    # performing_arts
    ("performing_arts", [
        "аранжировщик",
        "хореограф",
        "композитор",
        "вокалист",
        "музыкант",
        "режиссёр",
        "режиссер",
        "танцор",
        "актёр",
        "актер",
    ]),

    # sound
    ("sound", [
        "звукорежиссёр",
        "звукорежиссер",
        "саунд-дизайнер",
        "саунд дизайнер",
    ]),

    # visual_arts
    ("visual_arts", [
        "живописец",
        "художник",
    ]),

    # photo_video
    ("photo_video", [
        "видеооператор",
        "монтажёр",
        "монтажер",
        "аниматор",
        "фотограф",
    ]),

    # event_production
    ("event_production", [
        "театральный продюсер",
        "кино-продюсер",
        "кинопродюсер",
        "креативный продюсер",
        "event-менеджер",
        "event менеджер",
        "организатор мероприятий",
        "продюсер",
    ]),
]

def normalize_keyword(text):
    if pd.isna(text):
        return ""
    text = str(text).lower().replace("ё", "е")
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def map_profession_group(keyword):
    keyword_norm = normalize_keyword(keyword)

    if keyword_norm == "":
        return "other"

    for group, patterns in profession_rules:
        patterns_norm = [p.lower().replace("ё", "е") for p in patterns]
        if any(p in keyword_norm for p in patterns_norm):
            return group

    return "other"

df_prof["query_keyword_norm"] = df_prof["query_keyword"].apply(normalize_keyword)
df_prof["profession_group"] = df_prof["query_keyword_norm"].apply(map_profession_group)

# Экспортируем маппинг всех уникальных query_keyword
profession_mapping = (
    df_prof[["query_keyword", "query_keyword_norm", "profession_group"]]
    .drop_duplicates()
    .sort_values(["profession_group", "query_keyword_norm"])
    .reset_index(drop=True)
)

profession_mapping_path = REFERENCE_DIR / "profession_mapping.csv"
profession_mapping.to_csv(profession_mapping_path, index=False, encoding="utf-8-sig")

# Экспортируем также правила маппинга в JSON
profession_rules_path = REFERENCE_DIR / "profession_mapping_rules.json"
profession_rules_json = {
    group: patterns
    for group, patterns in profession_rules
}
with open(profession_rules_path, "w", encoding="utf-8") as f:
    json.dump(profession_rules_json, f, ensure_ascii=False, indent=2)

profession_counts = (
    df_prof["profession_group"]
    .value_counts(dropna=False)
    .rename_axis("profession_group")
    .reset_index(name="n")
)

profession_report_path = VALIDATION_DIR / "06_profession_mapping_report.xlsx"
with pd.ExcelWriter(profession_report_path, engine="openpyxl") as writer:
    profession_counts.to_excel(writer, sheet_name="profession_counts", index=False)
    profession_mapping.to_excel(writer, sheet_name="keyword_mapping", index=False)

prof_parquet_path = DATA_DIR / "05_profession_mapped.parquet"
df_prof.to_parquet(prof_parquet_path, index=False)

print("✅ Маппинг профессий выполнен.")
print(f"df_prof shape: {df_prof.shape}")
print(f"Файл маппинга сохранён: {profession_mapping_path}")
print(f"Правила маппинга сохранены: {profession_rules_path}")
print(f"Отчёт сохранён: {profession_report_path}")
print(f"Parquet сохранён: {prof_parquet_path}")
display(profession_counts)

In [ ]:
# ============================================================
# ЯЧЕЙКА 9. Лемматизация description через pymorphy3 и токенизация через razdel
# ============================================================

# Автономность ячейки
if "df_prof" not in globals():
    prof_parquet_path = DATA_DIR / "05_profession_mapped.parquet"
    if not prof_parquet_path.exists():
        raise FileNotFoundError("Не найден 05_profession_mapped.parquet. Сначала запустите ячейки 1–8.")
    df_prof = pd.read_parquet(prof_parquet_path)

import pymorphy3
from razdel import tokenize
from functools import lru_cache

df_lemma = df_prof.copy()

morph = pymorphy3.MorphAnalyzer()

TOKEN_RE = re.compile(r"[A-Za-zА-Яа-яЁё0-9][A-Za-zА-Яа-яЁё0-9\-]*")

@lru_cache(maxsize=300_000)
def lemmatize_token(token):
    """
    Лемматизация одного токена.
    Английские токены и числа оставляем в нижнем регистре без морфологического разбора.
    """
    token = token.lower().replace("ё", "е")

    if not re.search(r"[а-я]", token):
        return token

    try:
        return morph.parse(token)[0].normal_form.replace("ё", "е")
    except Exception:
        return token

def tokenize_text(text):
    """
    Токенизация через razdel + фильтр на реальные словесные токены.
    """
    if pd.isna(text):
        return []

    text = str(text).lower().replace("ё", "е")
    tokens = []

    for t in tokenize(text):
        token = t.text.strip().lower().replace("ё", "е")
        if TOKEN_RE.fullmatch(token):
            tokens.append(token)

    return tokens

def lemmatize_text(text):
    """
    Возвращает:
    - lemma_text: строка лемм через пробел;
    - token_count: число токенов;
    - unique_token_count: число уникальных токенов.
    """
    tokens = tokenize_text(text)
    lemmas = [lemmatize_token(tok) for tok in tokens]

    return pd.Series({
        "tokens": tokens,
        "lemma_text": " ".join(lemmas),
        "token_count": len(tokens),
        "unique_token_count": len(set(tokens)),
    })

lemma_features = df_lemma["description_lower"].progress_apply(lemmatize_text)
df_lemma = pd.concat([df_lemma, lemma_features], axis=1)

lemma_report = pd.DataFrame({
    "metric": [
        "rows_total",
        "median_token_count",
        "mean_token_count",
        "min_token_count",
        "max_token_count",
        "median_unique_token_count",
    ],
    "value": [
        len(df_lemma),
        float(df_lemma["token_count"].median()),
        float(df_lemma["token_count"].mean()),
        int(df_lemma["token_count"].min()),
        int(df_lemma["token_count"].max()),
        float(df_lemma["unique_token_count"].median()),
    ]
})

lemma_report_path = VALIDATION_DIR / "07_lemmatization_report.xlsx"
with pd.ExcelWriter(lemma_report_path, engine="openpyxl") as writer:
    lemma_report.to_excel(writer, sheet_name="lemmatization_summary", index=False)

lemma_parquet_path = DATA_DIR / "06_lemmatized.parquet"
df_lemma.to_parquet(lemma_parquet_path, index=False)

print("✅ Лемматизация выполнена.")
print(f"df_lemma shape: {df_lemma.shape}")
print(f"Медианное число токенов: {df_lemma['token_count'].median():.0f}")
print(f"Среднее число токенов: {df_lemma['token_count'].mean():.1f}")
print(f"Отчёт сохранён: {lemma_report_path}")
print(f"Parquet сохранён: {lemma_parquet_path}")
display(lemma_report)

In [ ]:
# ============================================================
# ЯЧЕЙКА 10. Стартовые лексиконы и сохранение lexicons.json
# ============================================================

lexicons = {
    "autonomy": [
        "автономия", "самостоятельность", "самостоятельно", "свобода действий",
        "свобода выбора", "свобода творчества", "творческая свобода",
        "твои идеи", "ваши идеи", "собственные идеи", "простор для творчества",
        "самовыражение", "авторский подход", "своё видение", "право голоса",
        "доверие", "без микроменеджмента", "без жёсткого контроля",
        "гибкий график", "гибкость", "выбор инструментов", "выбор стека",
        "инициатива",
    ],

    "competence": [
        "развитие", "профессиональный рост", "карьерный рост", "обучение",
        "повышение квалификации", "тренинги", "курсы", "конференции",
        "корпоративное обучение", "наставничество", "mentorship", "обмен опытом",
        "новые навыки", "hard skills", "soft skills", "экспертиза",
        "профессионализм", "мастерство", "расти как специалист",
        "прокачивать навыки", "развиваться",
    ],

    "relatedness": [
        "команда", "дружная команда", "сильная команда", "коллектив", "коллеги",
        "сообщество", "поддержка коллег", "открытая культура", "открытость",
        "доверительная атмосфера", "неформальное общение", "тимбилдинг",
        "общие ценности", "плоская структура",
    ],

    "meaning": [
        "миссия", "влиять на", "делать жизнь лучше", "миллионы пользователей",
        "наша аудитория", "помогать людям", "социальная значимость",
        "общественная польза", "культурный вклад", "наследие",
        "влияние на индустрию", "продукт меняет", "делать значимое",
        "осмысленная работа",
    ],

    "authorship": [
        "портфолио", "авторство", "твой стиль", "узнаваемый стиль",
        "кейс в портфолио", "твоё имя", "признание", "награды", "премии",
        "фестивали", "конкурсы", "ревью работ", "презентация результатов",
        "авторская работа", "индивидуальный почерк",
    ],

    "output_control": [
        "kpi", "кипиай", "метрики", "показатели", "выполнение плана",
        "план продаж", "конверсия", "ctr", "охваты", "roi", "roas",
        "отчётность", "отчёты", "цифры", "измеримый результат",
        "ориентация на результат", "таргет", "целевые показатели", "нормативы",
    ],

    "behavior_control": [
        "тз", "техническое задание", "бриф", "брифинг", "бренд-бук",
        "брендбук", "гайдлайн", "гайдлайны", "стандарты бренда",
        "корпоративный стиль", "регламент", "процедура", "согласование",
        "утверждение", "чёткое следование", "строгое соблюдение", "дедлайн",
        "дедлайны", "в установленные сроки", "по гайдам",
    ],

    "intensity": [
        "многозадачность", "многозадачный", "в режиме многозадачности",
        "быстрый темп", "высокий темп", "стрессоустойчивость",
        "стрессоустойчивый", "работа под давлением", "сжатые сроки",
        "горящие сроки", "готовность работать сверхурочно",
        "ненормированный график", "плотный график", "большой объём",
        "поток задач", "много задач одновременно", "готовность к переработкам",
    ],

    "selfbrand": [
        "личный бренд", "активные соцсети", "ведение соцсетей",
        "развитые соцсети", "телеграм-канал", "своё медиа", "экспертный блог",
        "личный сайт", "инфлюенсер", "экспертный контент", "медийность",
        "публичность",
    ],

    "ai_terms": [
        "ии", "искусственный интеллект", "нейросет", "нейронн", "chatgpt",
        "midjourney", "stable diffusion", "dall-e", "llm", "генеративн",
        "prompt-", "prompting", "copilot", "ai-",
    ],

    "ai_empowerment": [
        "ии помогает", "нейросеть как помощник", "освобождает время для творчества",
        "расширяет возможности", "расширяет инструментарий", "экспериментируем с",
        "исследуем нейросети", "новые инструменты ии", "изучаем ии",
        "готовы вместе осваивать", "использовать как ассистента", "ии-копайлот",
    ],

    "ai_intensification": [
        "ускорить с помощью ии", "автоматизировать рутину",
        "повысить производительность с помощью", "оптимизация процессов с ии",
        "выпускать больше контента", "увеличить объём", "масштабировать с помощью",
        "prompt-инженер", "prompt-инжиниринг", "владение нейросетями обязательно",
        "продвинутый prompting", "обязательное знание midjourney",
        "обязательное знание chatgpt", "опыт работы с нейросетями обязателен",
    ],
}

lexicons_path = REFERENCE_DIR / "lexicons.json"

with open(lexicons_path, "w", encoding="utf-8") as f:
    json.dump(lexicons, f, ensure_ascii=False, indent=2)

lexicon_summary = pd.DataFrame([
    {"lexicon": key, "n_markers": len(value)}
    for key, value in lexicons.items()
]).sort_values("lexicon")

lexicon_summary_path = REFERENCE_DIR / "lexicon_summary.csv"
lexicon_summary.to_csv(lexicon_summary_path, index=False, encoding="utf-8-sig")

print("✅ Стартовые лексиконы созданы и сохранены.")
print(f"lexicons.json: {lexicons_path}")
print(f"lexicon_summary.csv: {lexicon_summary_path}")
print(f"Всего лексиконов: {len(lexicons)}")
display(lexicon_summary)

In [ ]:
# ============================================================
# ЯЧЕЙКА 11. Расчёт лексиконных индексов на 1000 токенов
# ============================================================

# Автономность ячейки
if "df_lemma" not in globals():
    lemma_parquet_path = DATA_DIR / "06_lemmatized.parquet"
    if not lemma_parquet_path.exists():
        raise FileNotFoundError("Не найден 06_lemmatized.parquet. Сначала запустите ячейки 1–9.")
    df_lemma = pd.read_parquet(lemma_parquet_path)

if "lexicons" not in globals():
    lexicons_path = REFERENCE_DIR / "lexicons.json"
    if not lexicons_path.exists():
        raise FileNotFoundError("Не найден lexicons.json. Сначала запустите ячейку 10.")
    with open(lexicons_path, "r", encoding="utf-8") as f:
        lexicons = json.load(f)

df_lex = df_lemma.copy()

def normalize_marker_raw(marker):
    marker = str(marker).lower().replace("ё", "е")
    marker = unicodedata.normalize("NFKC", marker)
    marker = re.sub(r"\s+", " ", marker).strip()
    return marker

def lemmatize_marker(marker):
    """
    Лемматизация маркера: нужна для сопоставления с lemma_text.
    Многословные маркеры превращаются в последовательность лемм.
    """
    marker = normalize_marker_raw(marker)
    tokens = tokenize_text(marker)
    lemmas = [lemmatize_token(tok) for tok in tokens]
    return " ".join(lemmas)

def count_occurrences_safe(text, marker):
    """
    Подсчёт вхождений маркера.
    Для обычных слов и фраз используем границы слова.
    Для специальных маркеров вроде ai-, prompt- допускаем substring-поиск.
    """
    if not text or not marker:
        return 0

    text = str(text)
    marker = str(marker)

    # Специальные кусочные маркеры, где границы слова могут мешать
    if marker.endswith("-") or "-" in marker or marker in {"нейросет", "нейронн", "генеративн"}:
        return len(re.findall(re.escape(marker), text, flags=re.IGNORECASE))

    pattern = r"(?<!\w)" + re.escape(marker) + r"(?!\w)"
    return len(re.findall(pattern, text, flags=re.IGNORECASE))

# Предобработка лексиконов: raw-маркеры + лемматизированные маркеры
processed_lexicons = {}

for lex_name, markers in lexicons.items():
    processed_lexicons[lex_name] = []
    for marker in markers:
        raw_marker = normalize_marker_raw(marker)
        lemma_marker = lemmatize_marker(marker)
        processed_lexicons[lex_name].append({
            "marker": marker,
            "raw_marker": raw_marker,
            "lemma_marker": lemma_marker,
        })

processed_lexicons_path = REFERENCE_DIR / "lexicons_processed.json"
with open(processed_lexicons_path, "w", encoding="utf-8") as f:
    json.dump(processed_lexicons, f, ensure_ascii=False, indent=2)

def compute_lexicon_counts(row):
    """
    Для каждого конструкта считаем число найденных маркеров.
    Берём максимум между raw-поиском и lemma-поиском по каждому маркеру,
    чтобы не задваивать одно и то же вхождение.
    """
    desc_raw = row["description_lower"] if pd.notna(row["description_lower"]) else ""
    lemma_text = row["lemma_text"] if pd.notna(row["lemma_text"]) else ""
    token_count = row["token_count"] if pd.notna(row["token_count"]) else 0

    results = {}

    for lex_name, markers in processed_lexicons.items():
        total_count = 0

        for item in markers:
            raw_count = count_occurrences_safe(desc_raw, item["raw_marker"])
            lemma_count = count_occurrences_safe(lemma_text, item["lemma_marker"])
            total_count += max(raw_count, lemma_count)

        results[f"{lex_name}_lex_count"] = total_count

        if token_count and token_count > 0:
            results[f"{lex_name}_lex_freq_1000"] = total_count / token_count * 1000
        else:
            results[f"{lex_name}_lex_freq_1000"] = np.nan

    return pd.Series(results)

lexicon_features = df_lex.progress_apply(compute_lexicon_counts, axis=1)
df_lex = pd.concat([df_lex, lexicon_features], axis=1)

# Бинарный флаг упоминания ИИ
df_lex["ai_mention"] = df_lex["ai_terms_lex_count"] > 0

# Для единообразия создаём короткие имена лексиконных индексов
for construct in ALL_CONSTRUCTS:
    df_lex[f"{construct}_lex"] = df_lex[f"{construct}_lex_freq_1000"]

lex_index_cols = [f"{c}_lex" for c in ALL_CONSTRUCTS]

lex_summary = (
    df_lex[lex_index_cols + ["ai_mention"]]
    .agg(["mean", "median", "std", "min", "max"])
    .T
    .reset_index()
    .rename(columns={"index": "variable"})
)

ai_mention_report = pd.DataFrame({
    "metric": [
        "rows_total",
        "ai_mention_count",
        "ai_mention_share",
    ],
    "value": [
        len(df_lex),
        int(df_lex["ai_mention"].sum()),
        float(df_lex["ai_mention"].mean()),
    ]
})

lex_report_path = STATS_DIR / "01_lexicon_indices_summary.xlsx"
with pd.ExcelWriter(lex_report_path, engine="openpyxl") as writer:
    lex_summary.to_excel(writer, sheet_name="lexicon_summary", index=False)
    ai_mention_report.to_excel(writer, sheet_name="ai_mention", index=False)

lex_parquet_path = DATA_DIR / "07_lexicon_indices.parquet"
df_lex.to_parquet(lex_parquet_path, index=False)

print("✅ Лексиконные индексы рассчитаны.")
print(f"df_lex shape: {df_lex.shape}")
print(f"AI mention: {df_lex['ai_mention'].sum()} из {len(df_lex)} ({df_lex['ai_mention'].mean():.2%})")
print(f"Обработанные лексиконы сохранены: {processed_lexicons_path}")
print(f"Сводка сохранена: {lex_report_path}")
print(f"Parquet сохранён: {lex_parquet_path}")
display(lex_summary)

In [ ]:
# ============================================================
# ЯЧЕЙКА 12. Якорные фразы для эмбеддинговых центроидов и сохранение anchor_phrases.json
# ============================================================

anchor_phrases = {
    "autonomy": [
        "Мы доверяем специалисту самостоятельный выбор решений и даём пространство для собственных идей в рамках общей задачи проекта.",
        "В работе будет много свободы действий, возможности предлагать свои подходы и влиять на финальный творческий результат.",
        "Команда ценит инициативу, авторское видение и готовность самостоятельно выбирать инструменты для решения нестандартных задач.",
        "Мы не практикуем микроменеджмент и ожидаем, что специалист сможет сам выстраивать процесс работы над креативными задачами.",
        "У кандидата будет право голоса в обсуждении концепций, визуальных решений и способов реализации проектных идей.",
        "Работа предполагает гибкость, творческую свободу и возможность пробовать разные форматы без жёсткого давления сверху.",
        "Нам важен человек, который умеет брать ответственность за собственные идеи и превращать их в сильные проектные решения.",
        "Мы открыты к новым предложениям и готовы обсуждать нестандартные решения, если они усиливают качество продукта.",
        "Специалист сможет самостоятельно планировать часть задач, выбирать рабочий ритм и предлагать улучшения в процессе.",
        "Вакансия подойдёт тем, кто хочет не просто выполнять инструкции, а влиять на содержание и форму результата.",
        "Мы ищем человека с собственным взглядом, которому важно иметь пространство для эксперимента и профессионального самовыражения.",
        "Внутри команды есть свобода выбора инструментов, подходов и визуального языка при сохранении общего направления проекта.",
    ],

    "competence": [
        "Мы поддерживаем профессиональный рост, обучение новым инструментам и развитие экспертизы через сложные практические задачи.",
        "В компании есть наставничество, обмен опытом с сильными коллегами и возможность регулярно прокачивать профессиональные навыки.",
        "Работа позволит развиваться как специалисту, расширять портфолио компетенций и осваивать современные методы креативного производства.",
        "Мы оплачиваем профильные курсы, внутренние тренинги и участие в конференциях для развития hard skills и soft skills.",
        "Кандидат будет расти через обратную связь, разбор проектов и участие в задачах разного уровня сложности.",
        "Нам важно, чтобы специалист усиливал свою экспертизу, учился новым подходам и постепенно брал более сложные задачи.",
        "Команда помогает быстрее адаптироваться, делится опытом и поддерживает развитие профессионального мастерства в ежедневной работе.",
        "Вакансия подойдёт тем, кто хочет системно развиваться, изучать новые инструменты и повышать качество своих решений.",
        "Мы создаём среду, где можно учиться у опытных специалистов и постепенно расширять зону профессиональной ответственности.",
        "Проекты требуют высокого уровня профессионализма, но одновременно дают возможность расти через практику и содержательную обратную связь.",
        "Компания заинтересована в долгосрочном развитии сотрудников, повышении квалификации и формировании сильной внутренней экспертизы.",
        "У нас можно развивать креативные и аналитические навыки, работая с реальными задачами бизнеса и аудитории.",
    ],

    "relatedness": [
        "Мы предлагаем работу в дружной команде, где коллеги помогают друг другу и открыто обсуждают рабочие вопросы.",
        "В коллективе важны доверие, уважительное общение и готовность поддерживать других участников проекта в сложные периоды.",
        "Команда объединена общими ценностями, открытой культурой и желанием создавать качественный продукт вместе.",
        "У нас неформальная атмосфера, горизонтальное взаимодействие и регулярный обмен идеями между специалистами разных направлений.",
        "Кандидат будет работать с сильными коллегами, которые готовы делиться опытом и включаться в совместный поиск решений.",
        "Мы ценим командность, взаимную поддержку и способность договариваться при работе над общим творческим результатом.",
        "Внутри компании развита культура открытой коммуникации, где можно спокойно задавать вопросы и обсуждать трудности.",
        "Специалист станет частью профессионального сообщества, где важны совместная ответственность и уважение к вкладу каждого.",
        "Наша команда работает в атмосфере доверия, где ценятся честная обратная связь и поддержка коллег.",
        "Вакансия подойдёт тем, кому важно чувствовать себя частью коллектива и вместе двигаться к результату.",
        "Мы проводим командные встречи, обсуждения проектов и неформальные активности, чтобы сохранять живую рабочую связь.",
        "Работа предполагает постоянное взаимодействие с коллегами, совместное обсуждение идей и согласование решений внутри команды.",
    ],

    "meaning": [
        "Наш продукт помогает людям решать важные задачи и даёт специалисту возможность видеть реальный смысл своей работы.",
        "Проекты компании влияют на большую аудиторию, поэтому вклад каждого сотрудника заметен за пределами внутренней команды.",
        "Мы создаём решения, которые меняют пользовательский опыт и помогают делать повседневную жизнь людей удобнее.",
        "Работа связана с культурным вкладом, развитием индустрии и созданием проектов, которые имеют общественную значимость.",
        "Нам важно делать не просто контент, а осмысленные продукты, которые находят отклик у аудитории.",
        "Кандидат сможет участвовать в проектах, которые влияют на рынок, пользователей и восприятие бренда в индустрии.",
        "Мы верим в миссию продукта и ищем человека, которому важно понимать, зачем создаётся каждый проект.",
        "Результаты работы будут видеть тысячи и миллионы пользователей, поэтому качество и смысл решений особенно важны.",
        "Вакансия подойдёт тем, кто хочет создавать значимые вещи, а не просто закрывать потоковые задачи.",
        "Компания развивает проекты с долгосрочным эффектом, где творческие решения помогают усиливать ценность продукта для людей.",
        "Мы ищем специалиста, которому важно чувствовать влияние своей работы на аудиторию, культуру или профессиональное сообщество.",
        "Каждая задача связана с более широкой целью: улучшать продукт, помогать людям и создавать устойчивую ценность.",
    ],

    "authorship": [
        "У специалиста будет возможность создавать авторские работы, развивать узнаваемый стиль и добавлять сильные кейсы в портфолио.",
        "Мы ценим индивидуальный почерк, самостоятельное творческое решение и готовность презентовать результаты своей работы команде.",
        "Проекты можно будет использовать как заметные портфолио-кейсы, показывающие авторство и профессиональный уровень кандидата.",
        "Компания поддерживает участие в конкурсах, фестивалях и профессиональных премиях, если проект соответствует требованиям площадки.",
        "Нам важен специалист, который хочет развивать собственный стиль и получать признание за качественно выполненную работу.",
        "Работа предполагает ревью творческих решений, обсуждение авторского подхода и презентацию финальных результатов заказчикам.",
        "Кандидат сможет формировать визуальный или текстовый язык проектов, сохраняя авторский вклад в итоговом результате.",
        "Мы открыты к тому, чтобы сильные работы публиковались с указанием роли специалиста и попадали в портфолио.",
        "Вакансия подойдёт тем, кому важно не только выполнять задачи, но и видеть своё имя за результатом.",
        "Проекты дают возможность проявить индивидуальность, создать узнаваемые решения и получить профессиональное признание внутри команды.",
        "Мы ищем человека, который умеет защищать свою идею и превращать её в убедительную авторскую работу.",
        "В компании ценится не безличное производство, а личный вклад специалиста в качество и выразительность результата.",
    ],

    "output_control": [
        "Работа предполагает ориентацию на измеримый результат, выполнение KPI и регулярный анализ показателей эффективности проектов.",
        "Специалист будет отвечать за метрики, отчётность, достижение целевых показателей и понятную связь действий с результатом.",
        "Нам важно, чтобы кандидат умел работать с цифрами, конверсией, охватами и другими показателями эффективности контента.",
        "Задачи оцениваются через выполнение плана, достижение KPI и регулярную отчётность перед руководителем проекта.",
        "Кандидат должен понимать влияние своих решений на ROI, охваты, вовлечённость и другие бизнес-метрики.",
        "Работа требует системного контроля результатов, анализа данных и готовности корректировать действия по фактическим показателям.",
        "Мы ожидаем, что специалист сможет планировать результат, отслеживать метрики и объяснять эффективность выбранных решений.",
        "Важна ориентация на результат, достижение измеримых целей и соблюдение установленных нормативов качества работы.",
        "Каждый проект сопровождается понятными показателями, регулярными отчётами и оценкой вклада в бизнес-цели компании.",
        "Специалист будет работать с планами, целевыми значениями, аналитикой и постоянным контролем выполнения задач.",
        "Нам нужен человек, который умеет переводить креативные решения в понятные показатели эффективности и роста.",
        "Результат работы оценивается не только визуально или содержательно, но и через цифры, метрики и выполнение плана.",
    ],

    "behavior_control": [
        "Работа требует точного следования техническому заданию, брендбуку, гайдлайнам и утверждённым стандартам визуальной коммуникации.",
        "Специалист должен соблюдать регламенты, дедлайны, процедуры согласования и требования корпоративного стиля в каждом проекте.",
        "Все материалы создаются на основе брифа, проходят утверждение и должны соответствовать внутренним стандартам бренда.",
        "Кандидат будет работать по чётким ТЗ, установленным срокам и заранее согласованным правилам оформления материалов.",
        "Важно внимательно следовать гайдам, учитывать правки руководителя и соблюдать процедуру согласования перед публикацией.",
        "Процесс работы включает брифинг, подготовку материалов, промежуточные проверки и финальное утверждение результата.",
        "Мы ожидаем аккуратности, дисциплины, соблюдения дедлайнов и способности работать в рамках заданного регламента.",
        "Задачи требуют строгого соблюдения брендбука, корректного использования шаблонов и внимательности к деталям оформления.",
        "Специалист должен уметь работать в заданных рамках, не нарушая утверждённые стандарты и сроки проекта.",
        "Вакансия предполагает регулярные согласования, контроль соответствия техническому заданию и работу по установленным процессам.",
        "Кандидату важно комфортно чувствовать себя в структуре, где есть правила, дедлайны и обязательные этапы проверки.",
        "Мы ищем человека, который умеет точно выполнять бриф и качественно доводить материалы до утверждения.",
    ],

    "intensity": [
        "Работа проходит в высоком темпе, с большим потоком задач, сжатыми сроками и необходимостью быстро переключаться.",
        "Кандидат должен быть готов к многозадачности, плотному графику и периодическим задачам с горящими дедлайнами.",
        "Вакансия предполагает работу под давлением, быстрые итерации, большой объём коммуникаций и высокий уровень ответственности.",
        "Нужно уметь одновременно вести несколько проектов, быстро реагировать на изменения и сохранять качество в срок.",
        "Команда работает динамично, поэтому важны стрессоустойчивость, собранность и готовность справляться с интенсивной нагрузкой.",
        "В периоды запусков возможен высокий темп, большое количество задач и необходимость оперативно вносить правки.",
        "Мы ищем человека, которому комфортна многозадачная среда, быстрый ритм и постоянный поток проектных запросов.",
        "Рабочий процесс требует умения выдерживать сжатые сроки, приоритизировать задачи и быстро принимать решения.",
        "Кандидат будет регулярно сталкиваться с плотным графиком, срочными задачами и необходимостью держать несколько направлений.",
        "Вакансия подойдёт тем, кто умеет сохранять продуктивность при высокой нагрузке и частой смене задач.",
        "Работа требует готовности к интенсивным периодам, когда нужно выпускать много материалов в ограниченные сроки.",
        "Нам важна стрессоустойчивость, способность работать в быстром темпе и не терять качество при потоке задач.",
    ],

    "selfbrand": [
        "Нам важен специалист с развитым личным брендом, активными социальными сетями и опытом публичной экспертной коммуникации.",
        "Кандидат должен уметь вести профессиональные соцсети, создавать экспертный контент и усиливать доверие аудитории.",
        "Преимуществом будет собственный телеграм-канал, публичное портфолио и заметное присутствие в профессиональном сообществе.",
        "Мы ищем человека, который умеет развивать медийность, работать с личной репутацией и представлять экспертизу публично.",
        "Вакансия предполагает участие в публичных материалах, экспертных комментариях и продвижении профессионального образа специалиста.",
        "Наличие активных соцсетей, личного сайта или собственного медиа будет преимуществом для выполнения рабочих задач.",
        "Кандидат сможет использовать свой публичный образ, профессиональный блог и экспертность для усиления проектов компании.",
        "Нам интересны специалисты, которые понимают ценность личного бренда и умеют работать с публичной аудиторией.",
        "Работа связана с экспертным контентом, видимостью специалиста и регулярным взаимодействием с подписчиками или сообществом.",
        "Будет плюсом опыт развития личного бренда, ведения профессионального блога и участия в публичных обсуждениях.",
        "Мы ценим кандидатов, которые умеют говорить от своего имени и поддерживать профессиональную узнаваемость онлайн.",
        "Вакансия подойдёт тем, кто готов совмещать профессиональную работу с развитием собственной медийности и репутации.",
    ],

    "ai_empowerment": [
        "Мы используем нейросети как помощника, чтобы освобождать время для творчества и расширять инструментарий команды.",
        "Кандидат сможет экспериментировать с ИИ-инструментами, тестировать новые подходы и усиливать качество творческих решений.",
        "Компания изучает генеративный ИИ как дополнительный ресурс для идей, прототипирования и ускорения черновой работы.",
        "Нейросети помогают команде быстрее проверять гипотезы, искать референсы и создавать больше пространства для креатива.",
        "Мы готовы вместе осваивать ИИ-инструменты, обсуждать лучшие практики и применять их как ассистента специалиста.",
        "В работе можно использовать ChatGPT, Midjourney и другие инструменты для исследования, вдохновения и подготовки вариантов.",
        "ИИ рассматривается как копайлот, который расширяет возможности специалиста, но не заменяет авторское решение.",
        "Мы поддерживаем эксперименты с нейросетями, если они помогают находить новые идеи и повышать качество результата.",
        "Кандидат будет участвовать в поиске способов использовать ИИ для творческого развития, а не только автоматизации рутины.",
        "Нам интересен человек, который видит в нейросетях инструмент расширения профессиональных возможностей и проектного мышления.",
        "Использование ИИ помогает быстрее делать наброски, сравнивать варианты и оставлять больше времени для финальной доработки.",
        "Компания открыта к новым инструментам ИИ и готова поддерживать сотрудников в их аккуратном освоении.",
    ],

    "ai_intensification": [
        "Кандидат должен уверенно владеть нейросетями, чтобы ускорять производство контента и увеличивать объём выпускаемых материалов.",
        "Работа предполагает автоматизацию рутины с помощью ИИ, повышение производительности и масштабирование контентных процессов.",
        "Нужно использовать ChatGPT и Midjourney для быстрого создания большого количества вариантов в ограниченные сроки.",
        "Опыт работы с нейросетями обязателен, потому что команда рассчитывает быстрее закрывать потоковые задачи и дедлайны.",
        "Специалист будет отвечать за prompt-инжиниринг, оптимизацию процессов и ускорение регулярного производства контента.",
        "ИИ используется для сокращения времени на типовые операции, увеличения объёма задач и повышения скорости выполнения.",
        "Кандидат должен применять генеративные инструменты для масштабирования материалов, быстрого тестирования и постоянного роста производительности.",
        "Владение нейросетями обязательно, так как часть задач связана с массовым выпуском визуального или текстового контента.",
        "Мы ожидаем продвинутый prompting, умение автоматизировать процессы и быстро создавать материалы по заданным требованиям.",
        "Работа требует использовать ИИ для оптимизации процессов, ускорения согласований и выполнения большего количества задач.",
        "Кандидат будет настраивать связки инструментов ИИ, чтобы команда могла быстрее производить и адаптировать контент.",
        "Нейросети применяются для повышения эффективности, снижения ручной нагрузки и выполнения большего объёма работы теми же ресурсами.",
    ],
}

anchor_phrases_path = REFERENCE_DIR / "anchor_phrases.json"

with open(anchor_phrases_path, "w", encoding="utf-8") as f:
    json.dump(anchor_phrases, f, ensure_ascii=False, indent=2)

anchor_summary = pd.DataFrame([
    {"construct": construct, "n_phrases": len(phrases)}
    for construct, phrases in anchor_phrases.items()
]).sort_values("construct")

anchor_summary_path = REFERENCE_DIR / "anchor_phrases_summary.csv"
anchor_summary.to_csv(anchor_summary_path, index=False, encoding="utf-8-sig")

print("✅ Якорные фразы созданы и сохранены.")
print(f"anchor_phrases.json: {anchor_phrases_path}")
print(f"anchor_phrases_summary.csv: {anchor_summary_path}")
print(f"Всего конструктов: {len(anchor_phrases)}")
print(f"Всего якорных фраз: {sum(len(v) for v in anchor_phrases.values())}")
display(anchor_summary)

In [ ]:
# ============================================================
# ЯЧЕЙКА 13. Эмбеддинги вакансий и якорных центроидов через multilingual-e5-base
# ============================================================

# Автономность ячейки
if "df_lex" not in globals():
    lex_parquet_path = DATA_DIR / "07_lexicon_indices.parquet"
    if not lex_parquet_path.exists():
        raise FileNotFoundError("Не найден 07_lexicon_indices.parquet. Сначала запустите ячейки 1–11.")
    df_lex = pd.read_parquet(lex_parquet_path)

if "anchor_phrases" not in globals():
    anchor_phrases_path = REFERENCE_DIR / "anchor_phrases.json"
    if not anchor_phrases_path.exists():
        raise FileNotFoundError("Не найден anchor_phrases.json. Сначала запустите ячейку 12.")
    with open(anchor_phrases_path, "r", encoding="utf-8") as f:
        anchor_phrases = json.load(f)

from sentence_transformers import SentenceTransformer
import torch

df_embed = df_lex.copy()

PRIMARY_MODEL_NAME = "intfloat/multilingual-e5-base"
FALLBACK_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

MODEL_USED_PATH = REFERENCE_DIR / "embedding_model_used.json"
PASSAGE_EMB_PATH = DATA_DIR / "08_e5_passage_embeddings.parquet"
CENTROIDS_PATH = DATA_DIR / "08_e5_anchor_centroids.parquet"
EMBED_INDEX_PATH = DATA_DIR / "08_embedding_indices_e5.parquet"

device = "cuda" if torch.cuda.is_available() else "cpu"

def load_embedding_model():
    """
    Загружаем основную e5-модель.
    Если загрузка не удалась, используем резервную MiniLM.
    """
    try:
        model = SentenceTransformer(PRIMARY_MODEL_NAME, device=device)
        model_name = PRIMARY_MODEL_NAME
        model_type = "e5"
    except Exception as e:
        print("⚠️ Не удалось загрузить intfloat/multilingual-e5-base.")
        print(f"Причина: {repr(e)}")
        print("Перехожу к резервной модели MiniLM.")
        model = SentenceTransformer(FALLBACK_MODEL_NAME, device=device)
        model_name = FALLBACK_MODEL_NAME
        model_type = "minilm_fallback"

    return model, model_name, model_type

def encode_texts(model, texts, batch_size=32):
    """
    Кодирование текстов с L2-нормализацией.
    normalize_embeddings=True позволяет дальше считать cosine как dot product.
    """
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    return embeddings.astype("float32")

# ------------------------------------------------------------
# Если всё уже посчитано, подхватываем с диска
# ------------------------------------------------------------

if EMBED_INDEX_PATH.exists() and PASSAGE_EMB_PATH.exists() and CENTROIDS_PATH.exists():
    df_embed_indices = pd.read_parquet(EMBED_INDEX_PATH)
    passage_embeddings_df = pd.read_parquet(PASSAGE_EMB_PATH)
    centroids_df = pd.read_parquet(CENTROIDS_PATH)

    df_embed = df_embed.merge(
        df_embed_indices[["vacancy_id"] + [f"{c}_emb" for c in ALL_CONSTRUCTS]],
        on="vacancy_id",
        how="left",
    )

    print("✅ Найдены готовые эмбеддинги и индексы. Повторный расчёт не выполнялся.")
    print(f"df_embed shape: {df_embed.shape}")
    print(f"Загружено: {EMBED_INDEX_PATH}")

else:
    model, model_name, model_type = load_embedding_model()

    with open(MODEL_USED_PATH, "w", encoding="utf-8") as f:
        json.dump(
            {
                "model_name": model_name,
                "model_type": model_type,
                "device": device,
                "timestamp": datetime.now().isoformat(),
                "note": "Для e5 используются префиксы passage: для вакансий и query: для якорных фраз.",
            },
            f,
            ensure_ascii=False,
            indent=2,
        )

    # --------------------------------------------------------
    # 1. Эмбеддинги текстов вакансий
    # --------------------------------------------------------

    if model_type == "e5":
        passage_texts = [
            "passage: " + str(x) if pd.notna(x) else "passage: "
            for x in df_embed["description_original"].tolist()
        ]
    else:
        passage_texts = [
            str(x) if pd.notna(x) else ""
            for x in df_embed["description_original"].tolist()
        ]

    passage_embeddings = encode_texts(model, passage_texts, batch_size=64)

    emb_cols = [f"emb_{i:04d}" for i in range(passage_embeddings.shape[1])]
    passage_embeddings_df = pd.DataFrame(passage_embeddings, columns=emb_cols)
    passage_embeddings_df.insert(0, "vacancy_id", df_embed["vacancy_id"].values)
    passage_embeddings_df.to_parquet(PASSAGE_EMB_PATH, index=False)

    # --------------------------------------------------------
    # 2. Эмбеддинги якорных фраз и центроиды конструктов
    # --------------------------------------------------------

    centroid_rows = []

    for construct, phrases in anchor_phrases.items():
        if model_type == "e5":
            query_texts = ["query: " + str(p) for p in phrases]
        else:
            query_texts = [str(p) for p in phrases]

        anchor_emb = encode_texts(model, query_texts, batch_size=16)

        centroid = anchor_emb.mean(axis=0)
        centroid = centroid / np.linalg.norm(centroid)

        row = {"construct": construct}
        row.update({col: float(val) for col, val in zip(emb_cols, centroid)})
        centroid_rows.append(row)

    centroids_df = pd.DataFrame(centroid_rows)
    centroids_df.to_parquet(CENTROIDS_PATH, index=False)

    # --------------------------------------------------------
    # 3. Косинусная близость вакансии к каждому центроиду
    # --------------------------------------------------------

    centroid_matrix = (
        centroids_df
        .set_index("construct")[emb_cols]
        .loc[ALL_CONSTRUCTS]
        .to_numpy(dtype="float32")
    )

    # Так как embeddings нормализованы, cosine similarity = dot product
    sim_matrix = passage_embeddings @ centroid_matrix.T

    sim_df = pd.DataFrame(
        sim_matrix,
        columns=[f"{construct}_emb" for construct in ALL_CONSTRUCTS],
    )
    sim_df.insert(0, "vacancy_id", df_embed["vacancy_id"].values)

    sim_df.to_parquet(EMBED_INDEX_PATH, index=False)

    df_embed = df_embed.merge(sim_df, on="vacancy_id", how="left")

    print("✅ Эмбеддинги и эмбеддинговые индексы рассчитаны.")
    print(f"Модель: {model_name}")
    print(f"Устройство: {device}")
    print(f"Матрица эмбеддингов вакансий: {passage_embeddings.shape}")
    print(f"Индексы сохранены: {EMBED_INDEX_PATH}")

print(f"df_embed shape: {df_embed.shape}")
print(f"Passage embeddings parquet: {PASSAGE_EMB_PATH}")
print(f"Centroids parquet: {CENTROIDS_PATH}")
print(f"Model metadata: {MODEL_USED_PATH}")

In [ ]:
# ============================================================
# ЯЧЕЙКА 14. Составные переменные: support, control, semantic_balance, quadrant_type и ai_mode
# ============================================================

# Автономность ячейки
if "df_embed" not in globals():
    base_path = DATA_DIR / "07_lexicon_indices.parquet"
    emb_path = DATA_DIR / "08_embedding_indices_e5.parquet"

    if not base_path.exists() or not emb_path.exists():
        raise FileNotFoundError("Не найдены 07_lexicon_indices.parquet и/или 08_embedding_indices_e5.parquet. Сначала запустите ячейки 1–13.")

    df_embed = pd.read_parquet(base_path)
    df_embed_indices = pd.read_parquet(emb_path)
    df_embed = df_embed.merge(
        df_embed_indices[["vacancy_id"] + [f"{c}_emb" for c in ALL_CONSTRUCTS]],
        on="vacancy_id",
        how="left",
    )

df_final = df_embed.copy()

def zscore_series(s):
    """
    Z-score с защитой от нулевой дисперсии.
    """
    s = pd.to_numeric(s, errors="coerce")
    std = s.std(ddof=0)
    if pd.isna(std) or std == 0:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.mean()) / std

# ------------------------------------------------------------
# 1. Z-score для лексиконных и эмбеддинговых индексов
# ------------------------------------------------------------

for construct in ALL_CONSTRUCTS:
    df_final[f"{construct}_lex_z"] = zscore_series(df_final[f"{construct}_lex"])
    df_final[f"{construct}_emb_z"] = zscore_series(df_final[f"{construct}_emb"])

# ------------------------------------------------------------
# 2. Основные составные переменные
# ------------------------------------------------------------
# В качестве основной семантической меры берём эмбеддинговые индексы,
# потому что они менее зависят от буквального совпадения слов.
# Лексиконы остаются как прозрачная и валидируемая альтернатива.

df_final["support_total"] = df_final[[f"{c}_emb_z" for c in SUPPORTIVE_INDEXES]].sum(axis=1)
df_final["control_total"] = df_final[[f"{c}_emb_z" for c in CONTROL_INDEXES]].sum(axis=1)

df_final["support_total_lex"] = df_final[[f"{c}_lex_z" for c in SUPPORTIVE_INDEXES]].sum(axis=1)
df_final["control_total_lex"] = df_final[[f"{c}_lex_z" for c in CONTROL_INDEXES]].sum(axis=1)

df_final["semantic_balance"] = df_final["support_total"] - df_final["control_total"]
df_final["semantic_balance_lex"] = df_final["support_total_lex"] - df_final["control_total_lex"]

# Интенсивность как объединение лексиконного и эмбеддингового сигнала
df_final["combined_intensity"] = (
    df_final["intensity_emb_z"] + df_final["intensity_lex_z"]
) / 2

# ------------------------------------------------------------
# 3. Квадранты support × control по медианному разбиению
# ------------------------------------------------------------

support_median = df_final["support_total"].median()
control_median = df_final["control_total"].median()

def assign_quadrant(row):
    high_support = row["support_total"] >= support_median
    high_control = row["control_total"] >= control_median

    if high_support and high_control:
        return "enabling_bureaucracy"
    elif high_support and not high_control:
        return "creative_freedom"
    elif not high_support and high_control:
        return "production_line"
    else:
        return "thin_text"

df_final["quadrant_type"] = df_final.apply(assign_quadrant, axis=1)

# ------------------------------------------------------------
# 4. AI mode: empowerment / intensification / mixed / no_ai
# ------------------------------------------------------------

def assign_ai_mode(row):
    if not bool(row.get("ai_mention", False)):
        return "no_ai"

    emp = row.get("ai_empowerment_emb_z", np.nan)
    intens = row.get("ai_intensification_emb_z", np.nan)

    # Если оба сигнала примерно близки и явно присутствуют — mixed
    emp_lex = row.get("ai_empowerment_lex_count", 0)
    intens_lex = row.get("ai_intensification_lex_count", 0)

    if emp_lex > 0 and intens_lex > 0:
        return "mixed"

    if pd.isna(emp) or pd.isna(intens):
        return "mixed"

    diff = emp - intens

    if abs(diff) < 0.15:
        return "mixed"
    elif diff > 0:
        return "empowerment"
    else:
        return "intensification"

df_final["ai_mode"] = df_final.apply(assign_ai_mode, axis=1)

# ------------------------------------------------------------
# 5. Сохранение
# ------------------------------------------------------------

quadrant_counts = (
    df_final["quadrant_type"]
    .value_counts()
    .rename_axis("quadrant_type")
    .reset_index(name="n")
)

ai_mode_counts = (
    df_final["ai_mode"]
    .value_counts()
    .rename_axis("ai_mode")
    .reset_index(name="n")
)

composite_summary = pd.DataFrame({
    "metric": [
        "rows_total",
        "support_median",
        "control_median",
        "semantic_balance_median",
        "semantic_balance_mean",
        "ai_mention_count",
        "ai_mention_share",
    ],
    "value": [
        len(df_final),
        float(support_median),
        float(control_median),
        float(df_final["semantic_balance"].median()),
        float(df_final["semantic_balance"].mean()),
        int(df_final["ai_mention"].sum()),
        float(df_final["ai_mention"].mean()),
    ]
})

composite_report_path = STATS_DIR / "02_composite_variables_summary.xlsx"

with pd.ExcelWriter(composite_report_path, engine="openpyxl") as writer:
    composite_summary.to_excel(writer, sheet_name="summary", index=False)
    quadrant_counts.to_excel(writer, sheet_name="quadrant_type", index=False)
    ai_mode_counts.to_excel(writer, sheet_name="ai_mode", index=False)

final_parquet_path = DATA_DIR / "09_final_indices.parquet"
df_final.to_parquet(final_parquet_path, index=False)

print("✅ Составные переменные рассчитаны.")
print(f"df_final shape: {df_final.shape}")
print(f"Медиана support_total: {support_median:.4f}")
print(f"Медиана control_total: {control_median:.4f}")
print(f"Отчёт сохранён: {composite_report_path}")
print(f"Parquet сохранён: {final_parquet_path}")
display(composite_summary)
display(quadrant_counts)
display(ai_mode_counts)

In [ ]:
# ============================================================
# ЯЧЕЙКА 15. Описательная статистика и bootstrap 95% CI для медианы
# ============================================================

# Автономность ячейки
if "df_final" not in globals():
    final_parquet_path = DATA_DIR / "09_final_indices.parquet"
    if not final_parquet_path.exists():
        raise FileNotFoundError("Не найден 09_final_indices.parquet. Сначала запустите ячейки 1–14.")
    df_final = pd.read_parquet(final_parquet_path)

np.random.seed(RANDOM_STATE)

# ------------------------------------------------------------
# Основные переменные для описательной статистики
# ------------------------------------------------------------

index_vars_emb = [f"{c}_emb" for c in ALL_CONSTRUCTS]
index_vars_lex = [f"{c}_lex" for c in ALL_CONSTRUCTS]

composite_vars = [
    "support_total",
    "control_total",
    "semantic_balance",
    "combined_intensity",
    "support_total_lex",
    "control_total_lex",
    "semantic_balance_lex",
]

analysis_vars = index_vars_emb + index_vars_lex + composite_vars

group_vars = [
    "profession_group",
    "creative_segment",
    "city_group",
    "experience_text",
    "ai_mention",
    "quadrant_type",
]

def bootstrap_median_ci(x, B=BOOTSTRAP_B, random_state=RANDOM_STATE):
    """
    Bootstrap 95% CI для медианы.
    Возвращает нижнюю и верхнюю границу доверительного интервала.
    """
    x = pd.Series(x).dropna().astype(float).values

    if len(x) < 5:
        return np.nan, np.nan

    rng = np.random.default_rng(random_state)
    boot_medians = np.empty(B)

    for b in range(B):
        sample = rng.choice(x, size=len(x), replace=True)
        boot_medians[b] = np.median(sample)

    ci_low, ci_high = np.percentile(boot_medians, [2.5, 97.5])
    return float(ci_low), float(ci_high)

def descriptive_table(data, variables, group_col=None, min_group_n=10):
    """
    Описательная статистика:
    mean, median, std, IQR, bootstrap 95% CI для медианы.
    """
    rows = []

    if group_col is None:
        iterable = [("ALL", data)]
    else:
        iterable = []
        for group_value, group_df in data.groupby(group_col, dropna=False):
            if len(group_df) >= min_group_n:
                iterable.append((group_value, group_df))

    for group_value, group_df in tqdm(iterable, desc=f"descriptive: {group_col or 'overall'}"):
        for var in variables:
            x = pd.to_numeric(group_df[var], errors="coerce").dropna()

            if len(x) == 0:
                continue

            q1 = x.quantile(0.25)
            q3 = x.quantile(0.75)
            ci_low, ci_high = bootstrap_median_ci(x, B=BOOTSTRAP_B, random_state=RANDOM_STATE)

            rows.append({
                "group_variable": group_col if group_col is not None else "overall",
                "group_value": group_value,
                "variable": var,
                "n": int(x.shape[0]),
                "mean": float(x.mean()),
                "median": float(x.median()),
                "std": float(x.std(ddof=1)) if len(x) > 1 else np.nan,
                "q1": float(q1),
                "q3": float(q3),
                "iqr": float(q3 - q1),
                "min": float(x.min()),
                "max": float(x.max()),
                "median_ci_low": ci_low,
                "median_ci_high": ci_high,
            })

    return pd.DataFrame(rows)

# ------------------------------------------------------------
# Расчёт таблиц
# ------------------------------------------------------------

overall_desc = descriptive_table(df_final, analysis_vars, group_col=None)

group_desc_tables = {}
for group_col in group_vars:
    group_desc_tables[group_col] = descriptive_table(
        df_final,
        analysis_vars,
        group_col=group_col,
        min_group_n=10,
    )

# ------------------------------------------------------------
# Разведочная зарплатная статистика
# ------------------------------------------------------------

salary_df = df_final[
    (df_final["salary_currency"] == "RUB") &
    (df_final["salary_mid"].notna())
].copy()

salary_vars = ["salary_mid"]

salary_overall_desc = descriptive_table(
    salary_df,
    salary_vars,
    group_col=None,
) if len(salary_df) > 0 else pd.DataFrame()

salary_group_tables = {}
for group_col in ["profession_group", "creative_segment", "city_group", "experience_text"]:
    if len(salary_df) > 0:
        salary_group_tables[group_col] = descriptive_table(
            salary_df,
            salary_vars,
            group_col=group_col,
            min_group_n=20,
        )
    else:
        salary_group_tables[group_col] = pd.DataFrame()

salary_warning = pd.DataFrame({
    "note": [
        "Зарплатный анализ является разведочным: salary_text заполнен только у небольшой доли вакансий.",
        "salary_mid считается только при наличии обеих границ salary_from и salary_to.",
        "Групповые зарплатные сравнения выводятся только для групп, где не менее 20 вакансий с salary_mid в RUB.",
        "salary_relative_to_city_median не рассчитывается, так как по ТЗ нельзя считать его при малых группах salary.",
    ],
    "value": [
        f"salary_text non-missing: {int(df_final['salary_text'].notna().sum())} из {len(df_final)}",
        f"salary_mid RUB available: {len(salary_df)} из {len(df_final)}",
        "min_group_n = 20",
        "not calculated",
    ],
})

# ------------------------------------------------------------
# Сохранение
# ------------------------------------------------------------

desc_stats_path = STATS_DIR / "03_descriptive_statistics.xlsx"

with pd.ExcelWriter(desc_stats_path, engine="openpyxl") as writer:
    overall_desc.to_excel(writer, sheet_name="overall", index=False)

    for group_col, table in group_desc_tables.items():
        sheet_name = group_col[:31]
        table.to_excel(writer, sheet_name=sheet_name, index=False)

    salary_warning.to_excel(writer, sheet_name="salary_warning", index=False)

    if not salary_overall_desc.empty:
        salary_overall_desc.to_excel(writer, sheet_name="salary_overall", index=False)

    for group_col, table in salary_group_tables.items():
        if not table.empty:
            sheet_name = f"salary_{group_col}"[:31]
            table.to_excel(writer, sheet_name=sheet_name, index=False)

# Отдельная таблица:
# медианы ключевых индексов по profession_group
thesis_medians_by_profession = (
    group_desc_tables["profession_group"]
    .query("variable in @index_vars_emb")
    .loc[:, ["group_value", "variable", "n", "median", "median_ci_low", "median_ci_high", "iqr"]]
    .rename(columns={"group_value": "profession_group"})
    .sort_values(["variable", "median"], ascending=[True, False])
)

thesis_table_path = TABLES_DIR / "table_medians_by_profession_group.xlsx"
with pd.ExcelWriter(thesis_table_path, engine="openpyxl") as writer:
    thesis_medians_by_profession.to_excel(writer, sheet_name="medians_by_profession", index=False)

desc_parquet_path = DATA_DIR / "10_descriptive_statistics_overall.parquet"
overall_desc.to_parquet(desc_parquet_path, index=False)

print("✅ Описательная статистика рассчитана.")
print(f"overall_desc shape: {overall_desc.shape}")
print(f"Количество групповых таблиц: {len(group_desc_tables)}")
print(f"salary_mid RUB доступен: {len(salary_df)} вакансий")
print(f"Excel сохранён: {desc_stats_path}")
print(f"Таблица сохранена: {thesis_table_path}")
display(overall_desc.head(20))

In [ ]:
# ============================================================
# ЯЧЕЙКА 16. Инференциальная статистика
# Mann–Whitney U + Cliff's delta
# Kruskal–Wallis + epsilon squared
# Dunn post-hoc с Holm
# Spearman rho + FDR Benjamini-Hochberg
# ============================================================

# Автономность ячейки
if "df_final" not in globals():
    final_parquet_path = DATA_DIR / "09_final_indices.parquet"
    if not final_parquet_path.exists():
        raise FileNotFoundError("Не найден 09_final_indices.parquet. Сначала запустите ячейки 1–14.")
    df_final = pd.read_parquet(final_parquet_path)

try:
    import scikit_posthocs as sp
except ImportError:
    raise ImportError("Пакет scikit-posthocs не найден. Запустите ячейку 1 с установкой зависимостей.")

# ------------------------------------------------------------
# Переменные анализа
# ------------------------------------------------------------

index_vars_emb = [f"{c}_emb" for c in ALL_CONSTRUCTS]

inferential_vars = index_vars_emb + [
    "support_total",
    "control_total",
    "semantic_balance",
    "combined_intensity",
]

binary_group_vars = [
    "ai_mention",
]

multi_group_vars = [
    "profession_group",
    "creative_segment",
    "city_group",
    "experience_text",
    "quadrant_type",
    "ai_mode",
]

# ------------------------------------------------------------
# Интерпретация эффектов
# ------------------------------------------------------------

def interpret_cliffs_delta(delta):
    """
    Пороговая интерпретация Cliff's delta:
    |d| < .147 negligible
    .147 small
    .33 medium
    .474 large
    """
    ad = abs(delta)
    if ad < 0.147:
        return "negligible"
    elif ad < 0.33:
        return "small"
    elif ad < 0.474:
        return "medium"
    else:
        return "large"

def interpret_epsilon_squared(eps):
    """
    Условная интерпретация epsilon squared для Kruskal-Wallis.
    """
    if pd.isna(eps):
        return np.nan
    if eps < 0.01:
        return "negligible"
    elif eps < 0.06:
        return "small"
    elif eps < 0.14:
        return "medium"
    else:
        return "large"

def interpret_spearman(rho):
    """
    Условная интерпретация силы корреляции.
    """
    ar = abs(rho)
    if ar < 0.10:
        return "negligible"
    elif ar < 0.30:
        return "small"
    elif ar < 0.50:
        return "medium"
    else:
        return "large"

def cliffs_delta(x, y):
    """
    Cliff's delta:
    P(x > y) - P(x < y)
    Реализовано через ранги, чтобы не строить огромную попарную матрицу.
    """
    x = pd.Series(x).dropna().astype(float).values
    y = pd.Series(y).dropna().astype(float).values

    if len(x) == 0 or len(y) == 0:
        return np.nan

    # Для n около 3305 можно считать напрямую, но сделаем аккуратно через сортировку
    y_sorted = np.sort(y)

    greater = np.searchsorted(y_sorted, x, side="left").sum()
    less = (len(y) - np.searchsorted(y_sorted, x, side="right")).sum()

    delta = (greater - less) / (len(x) * len(y))
    return float(delta)

# ------------------------------------------------------------
# 1. Mann–Whitney U для бинарных сравнений
# ------------------------------------------------------------

mw_rows = []

for group_col in binary_group_vars:
    temp = df_final[[group_col] + inferential_vars].dropna(subset=[group_col]).copy()

    group_values = sorted(temp[group_col].dropna().unique(), key=lambda x: str(x))

    if len(group_values) != 2:
        continue

    g1, g2 = group_values[0], group_values[1]

    for var in tqdm(inferential_vars, desc=f"Mann-Whitney: {group_col}"):
        x = pd.to_numeric(temp.loc[temp[group_col] == g1, var], errors="coerce").dropna()
        y = pd.to_numeric(temp.loc[temp[group_col] == g2, var], errors="coerce").dropna()

        if len(x) < 5 or len(y) < 5:
            continue

        stat, p_value = stats.mannwhitneyu(x, y, alternative="two-sided")
        delta = cliffs_delta(x, y)

        mw_rows.append({
            "group_variable": group_col,
            "group_1": g1,
            "group_2": g2,
            "variable": var,
            "n_1": len(x),
            "n_2": len(y),
            "median_1": float(x.median()),
            "median_2": float(y.median()),
            "mean_1": float(x.mean()),
            "mean_2": float(y.mean()),
            "mannwhitney_u": float(stat),
            "p_value": float(p_value),
            "cliffs_delta": delta,
            "effect_interpretation": interpret_cliffs_delta(delta),
        })

mw_results = pd.DataFrame(mw_rows)

if not mw_results.empty:
    mw_results["p_fdr_bh"] = multipletests(mw_results["p_value"], method="fdr_bh")[1]
    mw_results["significant_fdr_0_05"] = mw_results["p_fdr_bh"] < 0.05

# ------------------------------------------------------------
# 2. Kruskal–Wallis + epsilon squared
# ------------------------------------------------------------

kw_rows = []

for group_col in multi_group_vars:
    temp = df_final[[group_col] + inferential_vars].dropna(subset=[group_col]).copy()

    # Убираем слишком маленькие группы
    valid_groups = (
        temp[group_col]
        .value_counts(dropna=False)
        .loc[lambda s: s >= 10]
        .index
        .tolist()
    )

    temp = temp[temp[group_col].isin(valid_groups)].copy()

    if temp[group_col].nunique() < 2:
        continue

    for var in tqdm(inferential_vars, desc=f"Kruskal-Wallis: {group_col}"):
        groups = []
        group_labels = []

        for group_value, group_df in temp.groupby(group_col):
            x = pd.to_numeric(group_df[var], errors="coerce").dropna()
            if len(x) >= 10:
                groups.append(x.values)
                group_labels.append(group_value)

        if len(groups) < 2:
            continue

        H, p_value = stats.kruskal(*groups)
        n_total = sum(len(g) for g in groups)
        k = len(groups)

        # epsilon squared для Kruskal-Wallis
        # eps² = (H - k + 1) / (n - k), нижняя граница 0
        eps_sq = max((H - k + 1) / (n_total - k), 0) if n_total > k else np.nan

        kw_rows.append({
            "group_variable": group_col,
            "variable": var,
            "n_total": int(n_total),
            "n_groups": int(k),
            "groups_included": "; ".join(map(str, group_labels)),
            "kruskal_h": float(H),
            "p_value": float(p_value),
            "epsilon_squared": float(eps_sq),
            "effect_interpretation": interpret_epsilon_squared(eps_sq),
        })

kw_results = pd.DataFrame(kw_rows)

if not kw_results.empty:
    kw_results["p_fdr_bh"] = multipletests(kw_results["p_value"], method="fdr_bh")[1]
    kw_results["significant_fdr_0_05"] = kw_results["p_fdr_bh"] < 0.05

# ------------------------------------------------------------
# 3. Dunn post-hoc с Holm
# ------------------------------------------------------------
# Чтобы файл не стал слишком тяжёлым, делаем Dunn только для сравнений,
# где Kruskal-Wallis значим после FDR и эффект не negligible.

dunn_rows = []

kw_for_dunn = kw_results.copy()
if not kw_for_dunn.empty:
    kw_for_dunn = kw_for_dunn[
        (kw_for_dunn["significant_fdr_0_05"] == True) &
        (kw_for_dunn["effect_interpretation"] != "negligible")
    ].copy()

for _, row in tqdm(kw_for_dunn.iterrows(), total=len(kw_for_dunn), desc="Dunn post-hoc"):
    group_col = row["group_variable"]
    var = row["variable"]

    temp = df_final[[group_col, var]].dropna().copy()

    valid_groups = (
        temp[group_col]
        .value_counts(dropna=False)
        .loc[lambda s: s >= 10]
        .index
        .tolist()
    )
    temp = temp[temp[group_col].isin(valid_groups)].copy()

    if temp[group_col].nunique() < 2:
        continue

    try:
        p_matrix = sp.posthoc_dunn(
            temp,
            val_col=var,
            group_col=group_col,
            p_adjust="holm",
        )

        for g1 in p_matrix.index:
            for g2 in p_matrix.columns:
                if str(g1) >= str(g2):
                    continue

                x = pd.to_numeric(temp.loc[temp[group_col] == g1, var], errors="coerce").dropna()
                y = pd.to_numeric(temp.loc[temp[group_col] == g2, var], errors="coerce").dropna()
                delta = cliffs_delta(x, y)

                dunn_rows.append({
                    "group_variable": group_col,
                    "variable": var,
                    "group_1": g1,
                    "group_2": g2,
                    "n_1": len(x),
                    "n_2": len(y),
                    "median_1": float(x.median()),
                    "median_2": float(y.median()),
                    "median_diff_1_minus_2": float(x.median() - y.median()),
                    "p_holm": float(p_matrix.loc[g1, g2]),
                    "cliffs_delta": delta,
                    "effect_interpretation": interpret_cliffs_delta(delta),
                })
    except Exception as e:
        dunn_rows.append({
            "group_variable": group_col,
            "variable": var,
            "group_1": "ERROR",
            "group_2": "ERROR",
            "n_1": np.nan,
            "n_2": np.nan,
            "median_1": np.nan,
            "median_2": np.nan,
            "median_diff_1_minus_2": np.nan,
            "p_holm": np.nan,
            "cliffs_delta": np.nan,
            "effect_interpretation": f"error: {repr(e)}",
        })

dunn_results = pd.DataFrame(dunn_rows)

# ------------------------------------------------------------
# 4. Spearman correlation matrix + FDR
# Исправленная версия: безопасно приводит rho и p-value к скалярам
# ------------------------------------------------------------

corr_vars = index_vars_emb + [
    "support_total",
    "control_total",
    "semantic_balance",
    "combined_intensity",
]

corr_df = df_final[corr_vars].apply(pd.to_numeric, errors="coerce")

rho_matrix = pd.DataFrame(index=corr_vars, columns=corr_vars, dtype=float)
p_matrix = pd.DataFrame(index=corr_vars, columns=corr_vars, dtype=float)

corr_long_rows = []

def safe_spearman(x, y):
    """
    Безопасный Spearman:
    scipy.stats.spearmanr иногда возвращает матрицу, если вход интерпретирован как 2D.
    Здесь принудительно приводим результат к одному rho и одному p-value.
    """
    x = pd.Series(x).dropna().astype(float)
    y = pd.Series(y).dropna().astype(float)

    common_index = x.index.intersection(y.index)
    x = x.loc[common_index]
    y = y.loc[common_index]

    valid = pd.DataFrame({"x": x, "y": y}).dropna()

    if len(valid) < 5:
        return np.nan, np.nan, len(valid)

    # Если одна из переменных константная, корреляция не определена
    if valid["x"].nunique() < 2 or valid["y"].nunique() < 2:
        return np.nan, np.nan, len(valid)

    result = stats.spearmanr(valid["x"].to_numpy(), valid["y"].to_numpy())
    rho = result.statistic
    p_value = result.pvalue

    # Если вдруг вернулась матрица, берём внедиагональный элемент [0, 1]
    rho_arr = np.asarray(rho)
    p_arr = np.asarray(p_value)

    if rho_arr.ndim >= 2:
        rho = rho_arr[0, 1]
    else:
        rho = float(rho_arr)

    if p_arr.ndim >= 2:
        p_value = p_arr[0, 1]
    else:
        p_value = float(p_arr)

    return float(rho), float(p_value), len(valid)

for var1 in corr_vars:
    for var2 in corr_vars:
        rho, p_value, n_obs = safe_spearman(corr_df[var1], corr_df[var2])

        rho_matrix.loc[var1, var2] = rho
        p_matrix.loc[var1, var2] = p_value

        # В long-таблицу кладём только уникальные пары, без диагонали
        if corr_vars.index(var1) < corr_vars.index(var2):
            corr_long_rows.append({
                "variable_1": var1,
                "variable_2": var2,
                "n": n_obs,
                "spearman_rho": rho,
                "p_value": p_value,
                "effect_interpretation": interpret_spearman(rho) if not pd.isna(rho) else np.nan,
            })

corr_long = pd.DataFrame(corr_long_rows)

if not corr_long.empty:
    valid_p_mask = corr_long["p_value"].notna()
    corr_long["p_fdr_bh"] = np.nan
    corr_long["significant_fdr_0_05"] = False

    if valid_p_mask.sum() > 0:
        corr_long.loc[valid_p_mask, "p_fdr_bh"] = multipletests(
            corr_long.loc[valid_p_mask, "p_value"],
            method="fdr_bh",
        )[1]
        corr_long.loc[valid_p_mask, "significant_fdr_0_05"] = (
            corr_long.loc[valid_p_mask, "p_fdr_bh"] < 0.05
        )

# ------------------------------------------------------------
# 5. Сохранение всех результатов инференциальной статистики
# ------------------------------------------------------------

inferential_path = STATS_DIR / "04_inferential_statistics.xlsx"

with pd.ExcelWriter(inferential_path, engine="openpyxl") as writer:
    mw_results.to_excel(writer, sheet_name="mann_whitney", index=False)
    kw_results.to_excel(writer, sheet_name="kruskal_wallis", index=False)
    dunn_results.to_excel(writer, sheet_name="dunn_posthoc_holm", index=False)

    rho_matrix.reset_index().rename(columns={"index": "variable"}).to_excel(
        writer,
        sheet_name="spearman_rho_matrix",
        index=False,
    )

    p_matrix.reset_index().rename(columns={"index": "variable"}).to_excel(
        writer,
        sheet_name="spearman_p_matrix",
        index=False,
    )

    corr_long.to_excel(writer, sheet_name="spearman_long_fdr", index=False)

# ------------------------------------------------------------
# 6. Компактная таблица ключевых результатов
# ------------------------------------------------------------

important_kw = kw_results.copy()
if not important_kw.empty:
    important_kw = important_kw[
        (important_kw["significant_fdr_0_05"] == True) &
        (important_kw["effect_interpretation"].isin(["small", "medium", "large"]))
    ].copy()

    important_kw = important_kw.sort_values(
        ["epsilon_squared", "p_fdr_bh"],
        ascending=[False, True],
    )

important_mw = mw_results.copy()
if not important_mw.empty:
    important_mw = important_mw[
        (important_mw["significant_fdr_0_05"] == True) &
        (important_mw["effect_interpretation"].isin(["small", "medium", "large"]))
    ].copy()

    important_mw["abs_cliffs_delta"] = important_mw["cliffs_delta"].abs()
    important_mw = important_mw.sort_values(
        ["abs_cliffs_delta", "p_fdr_bh"],
        ascending=[False, True],
    )

corr_long_sorted = corr_long.copy()
if not corr_long_sorted.empty:
    corr_long_sorted["abs_spearman_rho"] = corr_long_sorted["spearman_rho"].abs()
    corr_long_sorted = corr_long_sorted.sort_values(
        ["abs_spearman_rho", "p_fdr_bh"],
        ascending=[False, True],
    )

thesis_inferential_path = TABLES_DIR / "table_key_inferential_results.xlsx"

with pd.ExcelWriter(thesis_inferential_path, engine="openpyxl") as writer:
    important_mw.to_excel(writer, sheet_name="binary_effects", index=False)
    important_kw.to_excel(writer, sheet_name="multigroup_effects", index=False)
    corr_long_sorted.to_excel(writer, sheet_name="correlations", index=False)

print("✅ Инференциальная статистика рассчитана.")
print(f"Mann–Whitney rows: {mw_results.shape[0]}")
print(f"Kruskal–Wallis rows: {kw_results.shape[0]}")
print(f"Dunn post-hoc rows: {dunn_results.shape[0]}")
print(f"Spearman pairs: {corr_long.shape[0]}")
print(f"Excel сохранён: {inferential_path}")
print(f"Компактная таблица сохранена: {thesis_inferential_path}")

if not kw_results.empty:
    display(kw_results.sort_values("epsilon_squared", ascending=False).head(15))
else:
    print("Kruskal–Wallis results пустой.")

In [ ]:
# ============================================================
# ЯЧЕЙКА 17. Валидация: top-20 положительных и отрицательных вакансий
# по каждому из 11 эмбеддинговых индексов
# ============================================================

# Автономность ячейки
if "df_final" not in globals():
    final_parquet_path = DATA_DIR / "09_final_indices.parquet"
    if not final_parquet_path.exists():
        raise FileNotFoundError("Не найден 09_final_indices.parquet. Сначала запустите ячейки 1–14.")
    df_final = pd.read_parquet(final_parquet_path)

validation_cols_base = [
    "vacancy_id",
    "url",
    "company",
    "query_keyword",
    "profession_group",
    "creative_segment",
    "city_group",
    "experience_text",
    "salary_text",
    "description_original",
]

top_rows = []

for construct in ALL_CONSTRUCTS:
    score_col = f"{construct}_emb"
    lex_col = f"{construct}_lex"

    temp = df_final[
        validation_cols_base + [score_col, lex_col, "support_total", "control_total", "semantic_balance"]
    ].copy()

    temp = temp.dropna(subset=[score_col])

    top_positive = (
        temp
        .sort_values(score_col, ascending=False)
        .head(20)
        .copy()
    )
    top_positive["construct"] = construct
    top_positive["pole"] = "positive_top20"
    top_positive["rank_within_pole"] = range(1, len(top_positive) + 1)

    top_negative = (
        temp
        .sort_values(score_col, ascending=True)
        .head(20)
        .copy()
    )
    top_negative["construct"] = construct
    top_negative["pole"] = "negative_top20"
    top_negative["rank_within_pole"] = range(1, len(top_negative) + 1)

    top_rows.append(top_positive)
    top_rows.append(top_negative)

top_validation = pd.concat(top_rows, ignore_index=True)

# Укорачиваем текст для удобного просмотра в Excel, но полное описание оставляем тоже
top_validation["description_preview_800"] = (
    top_validation["description_original"]
    .astype(str)
    .str.slice(0, 800)
)

# Переупорядочим колонки
front_cols = [
    "construct",
    "pole",
    "rank_within_pole",
    "vacancy_id",
    "url",
    "company",
    "query_keyword",
    "profession_group",
    "creative_segment",
    "city_group",
    "experience_text",
    "salary_text",
]

score_cols = [
    f"{c}_emb" for c in ALL_CONSTRUCTS
] + [
    f"{c}_lex" for c in ALL_CONSTRUCTS
] + [
    "support_total",
    "control_total",
    "semantic_balance",
]

available_score_cols = [c for c in score_cols if c in top_validation.columns]

top_validation = top_validation[
    front_cols + available_score_cols + ["description_preview_800", "description_original"]
]

# Один Excel-файл на логический раздел валидации
top_validation_path = VALIDATION_DIR / "08_top20_index_poles_validation.xlsx"

with pd.ExcelWriter(top_validation_path, engine="openpyxl") as writer:
    top_validation.to_excel(writer, sheet_name="all_top20", index=False)

    for construct in ALL_CONSTRUCTS:
        sheet = construct[:31]
        subset = top_validation[top_validation["construct"] == construct].copy()
        subset.to_excel(writer, sheet_name=sheet, index=False)

top_validation_parquet_path = DATA_DIR / "11_validation_top20_index_poles.parquet"
top_validation.to_parquet(top_validation_parquet_path, index=False)

print("✅ Top-20 положительных и отрицательных вакансий по индексам сохранены.")
print(f"top_validation shape: {top_validation.shape}")
print(f"Excel сохранён: {top_validation_path}")
print(f"Parquet сохранён: {top_validation_parquet_path}")
display(top_validation.head(10))

In [ ]:
# ============================================================
# ЯЧЕЙКА 18. Валидация: корреляции lexicon vs embedding
# ============================================================

# Автономность ячейки
if "df_final" not in globals():
    final_parquet_path = DATA_DIR / "09_final_indices.parquet"
    if not final_parquet_path.exists():
        raise FileNotFoundError("Не найден 09_final_indices.parquet. Сначала запустите ячейки 1–14.")
    df_final = pd.read_parquet(final_parquet_path)

lex_emb_rows = []

for construct in ALL_CONSTRUCTS:
    lex_col = f"{construct}_lex"
    emb_col = f"{construct}_emb"

    temp = df_final[[lex_col, emb_col]].apply(pd.to_numeric, errors="coerce").dropna()

    if len(temp) < 5:
        rho, p_value = np.nan, np.nan
        pearson_r, pearson_p = np.nan, np.nan
    else:
        rho, p_value = stats.spearmanr(temp[lex_col], temp[emb_col])
        pearson_r, pearson_p = stats.pearsonr(temp[lex_col], temp[emb_col])

    lex_emb_rows.append({
        "construct": construct,
        "n": len(temp),
        "spearman_rho": rho,
        "spearman_p": p_value,
        "pearson_r": pearson_r,
        "pearson_p": pearson_p,
        "interpretation": interpret_spearman(rho) if "interpret_spearman" in globals() else None,
    })

lex_emb_corr = pd.DataFrame(lex_emb_rows)

if not lex_emb_corr.empty:
    lex_emb_corr["spearman_p_fdr_bh"] = multipletests(
        lex_emb_corr["spearman_p"],
        method="fdr_bh"
    )[1]
    lex_emb_corr["spearman_significant_fdr_0_05"] = lex_emb_corr["spearman_p_fdr_bh"] < 0.05

# Матрица 11 × 11: строки — лексиконные индексы, столбцы — эмбеддинговые индексы
lex_cols = [f"{c}_lex" for c in ALL_CONSTRUCTS]
emb_cols = [f"{c}_emb" for c in ALL_CONSTRUCTS]

matrix_rows = []

for lex_construct in ALL_CONSTRUCTS:
    row = {"lexicon_construct": lex_construct}

    for emb_construct in ALL_CONSTRUCTS:
        lex_col = f"{lex_construct}_lex"
        emb_col = f"{emb_construct}_emb"

        temp = df_final[[lex_col, emb_col]].apply(pd.to_numeric, errors="coerce").dropna()

        if len(temp) < 5:
            rho = np.nan
        else:
            rho, _ = stats.spearmanr(temp[lex_col], temp[emb_col])

        row[emb_construct] = rho

    matrix_rows.append(row)

lex_emb_matrix = pd.DataFrame(matrix_rows)

lex_emb_validation_path = VALIDATION_DIR / "09_lexicon_embedding_validation.xlsx"

with pd.ExcelWriter(lex_emb_validation_path, engine="openpyxl") as writer:
    lex_emb_corr.to_excel(writer, sheet_name="same_construct_corr", index=False)
    lex_emb_matrix.to_excel(writer, sheet_name="matrix_11x11", index=False)

lex_emb_corr_path = DATA_DIR / "12_lexicon_embedding_correlations.parquet"
lex_emb_corr.to_parquet(lex_emb_corr_path, index=False)

print("✅ Корреляции lexicon vs embedding рассчитаны.")
print(f"lex_emb_corr shape: {lex_emb_corr.shape}")
print(f"lex_emb_matrix shape: {lex_emb_matrix.shape}")
print(f"Excel сохранён: {lex_emb_validation_path}")
print(f"Parquet сохранён: {lex_emb_corr_path}")
display(lex_emb_corr.sort_values("spearman_rho", ascending=False))

In [ ]:
# ============================================================
# ЯЧЕЙКА 19. Sensitivity check: MiniLM vs e5
# ============================================================
# Смысл: проверить, насколько выводы зависят от выбранной embedding-модели.
# Основная модель: intfloat/multilingual-e5-base.
# Резервная/проверочная: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2.

# Автономность ячейки
if "df_final" not in globals():
    final_parquet_path = DATA_DIR / "09_final_indices.parquet"
    if not final_parquet_path.exists():
        raise FileNotFoundError("Не найден 09_final_indices.parquet. Сначала запустите ячейки 1–14.")
    df_final = pd.read_parquet(final_parquet_path)

if "anchor_phrases" not in globals():
    anchor_phrases_path = REFERENCE_DIR / "anchor_phrases.json"
    if not anchor_phrases_path.exists():
        raise FileNotFoundError("Не найден anchor_phrases.json. Сначала запустите ячейку 12.")
    with open(anchor_phrases_path, "r", encoding="utf-8") as f:
        anchor_phrases = json.load(f)

from sentence_transformers import SentenceTransformer
import torch

MINILM_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

MINILM_PASSAGE_EMB_PATH = DATA_DIR / "13_minilm_passage_embeddings.parquet"
MINILM_CENTROIDS_PATH = DATA_DIR / "13_minilm_anchor_centroids.parquet"
MINILM_INDEX_PATH = DATA_DIR / "13_embedding_indices_minilm.parquet"
SENSITIVITY_PATH = VALIDATION_DIR / "10_sensitivity_minilm_vs_e5.xlsx"

device = "cuda" if torch.cuda.is_available() else "cpu"

def encode_texts_generic(model, texts, batch_size=64):
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    return embeddings.astype("float32")

# ------------------------------------------------------------
# 1. Если MiniLM-индексы уже посчитаны, подхватываем с диска
# ------------------------------------------------------------

if MINILM_INDEX_PATH.exists() and MINILM_PASSAGE_EMB_PATH.exists() and MINILM_CENTROIDS_PATH.exists():
    minilm_indices = pd.read_parquet(MINILM_INDEX_PATH)
    print("✅ Найдены готовые MiniLM-индексы. Повторный расчёт не выполнялся.")

else:
    minilm_model = SentenceTransformer(MINILM_MODEL_NAME, device=device)

    # Тексты вакансий без e5-префиксов
    texts = [
        str(x) if pd.notna(x) else ""
        for x in df_final["description_original"].tolist()
    ]

    minilm_embeddings = encode_texts_generic(minilm_model, texts, batch_size=64)

    emb_cols_minilm = [f"emb_{i:04d}" for i in range(minilm_embeddings.shape[1])]

    minilm_embeddings_df = pd.DataFrame(minilm_embeddings, columns=emb_cols_minilm)
    minilm_embeddings_df.insert(0, "vacancy_id", df_final["vacancy_id"].values)
    minilm_embeddings_df.to_parquet(MINILM_PASSAGE_EMB_PATH, index=False)

    # Центроиды
    centroid_rows = []

    for construct, phrases in anchor_phrases.items():
        phrase_embeddings = encode_texts_generic(minilm_model, phrases, batch_size=32)
        centroid = phrase_embeddings.mean(axis=0)
        centroid = centroid / np.linalg.norm(centroid)

        row = {"construct": construct}
        row.update({col: float(val) for col, val in zip(emb_cols_minilm, centroid)})
        centroid_rows.append(row)

    minilm_centroids_df = pd.DataFrame(centroid_rows)
    minilm_centroids_df.to_parquet(MINILM_CENTROIDS_PATH, index=False)

    centroid_matrix = (
        minilm_centroids_df
        .set_index("construct")[emb_cols_minilm]
        .loc[ALL_CONSTRUCTS]
        .to_numpy(dtype="float32")
    )

    sim_matrix = minilm_embeddings @ centroid_matrix.T

    minilm_indices = pd.DataFrame(
        sim_matrix,
        columns=[f"{construct}_minilm_emb" for construct in ALL_CONSTRUCTS],
    )
    minilm_indices.insert(0, "vacancy_id", df_final["vacancy_id"].values)
    minilm_indices.to_parquet(MINILM_INDEX_PATH, index=False)

# ------------------------------------------------------------
# 2. Сравнение e5 vs MiniLM
# ------------------------------------------------------------

df_sens = df_final.merge(minilm_indices, on="vacancy_id", how="left")

sensitivity_rows = []

for construct in ALL_CONSTRUCTS:
    e5_col = f"{construct}_emb"
    minilm_col = f"{construct}_minilm_emb"

    temp = df_sens[[e5_col, minilm_col]].apply(pd.to_numeric, errors="coerce").dropna()

    if len(temp) < 5:
        rho, p_value = np.nan, np.nan
        pearson_r, pearson_p = np.nan, np.nan
    else:
        rho, p_value = stats.spearmanr(temp[e5_col], temp[minilm_col])
        pearson_r, pearson_p = stats.pearsonr(temp[e5_col], temp[minilm_col])

    sensitivity_rows.append({
        "construct": construct,
        "n": len(temp),
        "spearman_rho_e5_minilm": rho,
        "spearman_p": p_value,
        "pearson_r_e5_minilm": pearson_r,
        "pearson_p": pearson_p,
        "interpretation": interpret_spearman(rho) if "interpret_spearman" in globals() else None,
    })

sensitivity_corr = pd.DataFrame(sensitivity_rows)

if not sensitivity_corr.empty:
    sensitivity_corr["spearman_p_fdr_bh"] = multipletests(
        sensitivity_corr["spearman_p"],
        method="fdr_bh"
    )[1]
    sensitivity_corr["spearman_significant_fdr_0_05"] = sensitivity_corr["spearman_p_fdr_bh"] < 0.05

# Сравнение составных переменных на MiniLM
for construct in ALL_CONSTRUCTS:
    df_sens[f"{construct}_minilm_z"] = zscore_series(df_sens[f"{construct}_minilm_emb"])

df_sens["support_total_minilm"] = df_sens[[f"{c}_minilm_z" for c in SUPPORTIVE_INDEXES]].sum(axis=1)
df_sens["control_total_minilm"] = df_sens[[f"{c}_minilm_z" for c in CONTROL_INDEXES]].sum(axis=1)
df_sens["semantic_balance_minilm"] = df_sens["support_total_minilm"] - df_sens["control_total_minilm"]

composite_sensitivity_rows = []

for var_e5, var_minilm in [
    ("support_total", "support_total_minilm"),
    ("control_total", "control_total_minilm"),
    ("semantic_balance", "semantic_balance_minilm"),
]:
    temp = df_sens[[var_e5, var_minilm]].apply(pd.to_numeric, errors="coerce").dropna()

    if len(temp) < 5:
        rho, p_value = np.nan, np.nan
    else:
        rho, p_value = stats.spearmanr(temp[var_e5], temp[var_minilm])

    composite_sensitivity_rows.append({
        "variable_e5": var_e5,
        "variable_minilm": var_minilm,
        "n": len(temp),
        "spearman_rho": rho,
        "p_value": p_value,
        "interpretation": interpret_spearman(rho) if "interpret_spearman" in globals() else None,
    })

composite_sensitivity = pd.DataFrame(composite_sensitivity_rows)

# Сохраняем один Excel-файл на раздел sensitivity
with pd.ExcelWriter(SENSITIVITY_PATH, engine="openpyxl") as writer:
    sensitivity_corr.to_excel(writer, sheet_name="construct_corr", index=False)
    composite_sensitivity.to_excel(writer, sheet_name="composite_corr", index=False)

sensitivity_parquet_path = DATA_DIR / "13_sensitivity_minilm_vs_e5.parquet"
df_sens.to_parquet(sensitivity_parquet_path, index=False)

print("✅ Sensitivity check MiniLM vs e5 выполнен.")
print(f"df_sens shape: {df_sens.shape}")
print(f"Средняя Spearman-корреляция по 11 конструктам: {sensitivity_corr['spearman_rho_e5_minilm'].mean():.3f}")
print(f"Excel сохранён: {SENSITIVITY_PATH}")
print(f"Parquet сохранён: {sensitivity_parquet_path}")
display(sensitivity_corr.sort_values("spearman_rho_e5_minilm", ascending=False))

In [ ]:
# ============================================================
# ЯЧЕЙКА 20. Sub-sampling stability: 500 вакансий × 100 повторов
# ============================================================
# Проверяем, насколько устойчивы медианные оценки индексов
# при случайном подвыборочном анализе.

# Автономность ячейки
if "df_final" not in globals():
    final_parquet_path = DATA_DIR / "09_final_indices.parquet"
    if not final_parquet_path.exists():
        raise FileNotFoundError("Не найден 09_final_indices.parquet. Сначала запустите ячейки 1–14.")
    df_final = pd.read_parquet(final_parquet_path)

STABILITY_PATH = VALIDATION_DIR / "11_subsampling_stability.xlsx"
STABILITY_PARQUET_PATH = DATA_DIR / "14_subsampling_stability.parquet"

SUBSAMPLE_N = 500
SUBSAMPLE_REPEATS = 100

stability_vars = [f"{c}_emb" for c in ALL_CONSTRUCTS] + [
    "support_total",
    "control_total",
    "semantic_balance",
    "combined_intensity",
]

if STABILITY_PARQUET_PATH.exists():
    stability_long = pd.read_parquet(STABILITY_PARQUET_PATH)
    print("✅ Найден готовый файл stability. Повторный расчёт не выполнялся.")
else:
    rng = np.random.default_rng(RANDOM_STATE)
    stability_rows = []

    if len(df_final) < SUBSAMPLE_N:
        raise ValueError(f"В данных меньше {SUBSAMPLE_N} строк. Невозможно выполнить sub-sampling stability.")

    for rep in tqdm(range(1, SUBSAMPLE_REPEATS + 1), desc="Sub-sampling stability"):
        sample_idx = rng.choice(df_final.index.values, size=SUBSAMPLE_N, replace=False)
        sample = df_final.loc[sample_idx].copy()

        for var in stability_vars:
            x = pd.to_numeric(sample[var], errors="coerce").dropna()

            if len(x) == 0:
                continue

            stability_rows.append({
                "repeat": rep,
                "sample_n": SUBSAMPLE_N,
                "variable": var,
                "median": float(x.median()),
                "mean": float(x.mean()),
                "std": float(x.std(ddof=1)) if len(x) > 1 else np.nan,
            })

    stability_long = pd.DataFrame(stability_rows)
    stability_long.to_parquet(STABILITY_PARQUET_PATH, index=False)

# Сравнение с полной выборкой
full_medians = []

for var in stability_vars:
    x = pd.to_numeric(df_final[var], errors="coerce").dropna()

    full_medians.append({
        "variable": var,
        "full_sample_n": len(x),
        "full_sample_median": float(x.median()),
        "full_sample_mean": float(x.mean()),
    })

full_medians = pd.DataFrame(full_medians)

stability_summary = (
    stability_long
    .groupby("variable", as_index=False)
    .agg(
        subsample_median_mean=("median", "mean"),
        subsample_median_std=("median", "std"),
        subsample_median_min=("median", "min"),
        subsample_median_max=("median", "max"),
        subsample_median_q025=("median", lambda x: np.quantile(x, 0.025)),
        subsample_median_q975=("median", lambda x: np.quantile(x, 0.975)),
        repeats=("repeat", "nunique"),
    )
    .merge(full_medians, on="variable", how="left")
)

stability_summary["abs_diff_full_vs_subsample_mean"] = (
    stability_summary["full_sample_median"] -
    stability_summary["subsample_median_mean"]
).abs()

stability_summary["relative_diff_full_vs_subsample_mean"] = (
    stability_summary["abs_diff_full_vs_subsample_mean"] /
    stability_summary["full_sample_median"].abs().replace(0, np.nan)
)

with pd.ExcelWriter(STABILITY_PATH, engine="openpyxl") as writer:
    stability_summary.to_excel(writer, sheet_name="stability_summary", index=False)
    stability_long.to_excel(writer, sheet_name="stability_long", index=False)

print("✅ Sub-sampling stability рассчитан.")
print(f"stability_long shape: {stability_long.shape}")
print(f"stability_summary shape: {stability_summary.shape}")
print(f"Excel сохранён: {STABILITY_PATH}")
print(f"Parquet сохранён: {STABILITY_PARQUET_PATH}")
display(stability_summary.sort_values("abs_diff_full_vs_subsample_mean", ascending=False).head(15))

In [ ]:
# ============================================================
# ЯЧЕЙКА 21. Общие функции для визуализаций и сохранения Plotly + PNG
# ============================================================

# Автономность ячейки
if "df_final" not in globals():
    final_parquet_path = DATA_DIR / "09_final_indices.parquet"
    if not final_parquet_path.exists():
        raise FileNotFoundError("Не найден 09_final_indices.parquet. Сначала запустите ячейки 1–14.")
    df_final = pd.read_parquet(final_parquet_path)

from scipy.stats import gaussian_kde

# Белая тема для Plotly
PLOTLY_TEMPLATE = "plotly_white"

# Переменные для графиков
index_vars_emb = [f"{c}_emb" for c in ALL_CONSTRUCTS]

construct_labels_ru = {
    "autonomy": "Автономия",
    "competence": "Компетентность / развитие",
    "relatedness": "Связанность / команда",
    "meaning": "Смысл",
    "authorship": "Авторство",
    "output_control": "Контроль результата",
    "behavior_control": "Контроль процесса",
    "intensity": "Интенсивность",
    "selfbrand": "Личный бренд",
    "ai_empowerment": "ИИ как расширение",
    "ai_intensification": "ИИ как интенсификация",
}

variable_labels_ru = {
    **{f"{k}_emb": v for k, v in construct_labels_ru.items()},
    "support_total": "Support total",
    "control_total": "Control total",
    "semantic_balance": "Semantic balance",
    "combined_intensity": "Combined intensity",
}

def safe_filename(name):
    """Безопасное имя файла."""
    name = str(name).lower()
    name = re.sub(r"[^a-zа-я0-9_\-]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name

def save_plotly_figure(fig, filename_base, width=1200, height=800, scale=2):
    """
    Сохраняет Plotly-график в HTML и PNG.
    Если PNG через kaleido не сохранился, HTML всё равно останется.
    """
    html_path = FIGURES_DIR / f"{filename_base}.html"
    png_path = FIGURES_DIR / f"{filename_base}_plotly.png"

    fig.write_html(str(html_path))

    try:
        fig.write_image(str(png_path), width=width, height=height, scale=scale)
        png_status = "PNG сохранён"
    except Exception as e:
        png_status = f"PNG не сохранён через kaleido: {repr(e)}"

    return html_path, png_path, png_status

def save_matplotlib_figure(fig, filename_base, dpi=300):
    """Сохраняет matplotlib-график в PNG."""
    png_path = FIGURES_DIR / f"{filename_base}_matplotlib.png"
    fig.savefig(png_path, dpi=dpi, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return png_path

def robust_group_order(data, group_col, value_col, min_n=20, top_n=12):
    """
    Выбирает читаемый набор групп:
    - минимум min_n наблюдений;
    - не больше top_n групп;
    - сортировка по медиане value_col.
    """
    temp = data[[group_col, value_col]].dropna().copy()

    counts = temp[group_col].value_counts()
    valid_groups = counts[counts >= min_n].index.tolist()

    medians = (
        temp[temp[group_col].isin(valid_groups)]
        .groupby(group_col)[value_col]
        .median()
        .sort_values(ascending=False)
    )

    return medians.head(top_n).index.tolist()

figures_index_rows = []

print("✅ Helper-функции для визуализаций загружены.")
print(f"df_final shape: {df_final.shape}")
print(f"Папка для графиков: {FIGURES_DIR}")

In [ ]:
# ============================================================
# ЯЧЕЙКА 22. Histogram semantic_balance с KDE и медианой
# ============================================================

# Автономность ячейки
if "df_final" not in globals():
    df_final = pd.read_parquet(DATA_DIR / "09_final_indices.parquet")

x = pd.to_numeric(df_final["semantic_balance"], errors="coerce").dropna()
median_x = x.median()

# ------------------------------------------------------------
# Plotly: histogram + KDE + median
# ------------------------------------------------------------

hist_counts, bin_edges = np.histogram(x, bins=40, density=True)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

kde = gaussian_kde(x)
x_grid = np.linspace(x.min(), x.max(), 400)
kde_values = kde(x_grid)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=bin_centers,
    y=hist_counts,
    width=np.diff(bin_edges),
    name="Плотность распределения",
    opacity=0.65,
))

fig.add_trace(go.Scatter(
    x=x_grid,
    y=kde_values,
    mode="lines",
    name="KDE",
    line=dict(width=3),
))

fig.add_vline(
    x=median_x,
    line_width=2,
    line_dash="dash",
    annotation_text=f"Медиана = {median_x:.3f}",
    annotation_position="top right",
)

fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Распределение semantic_balance",
    xaxis_title="semantic_balance = support_total − control_total",
    yaxis_title="Плотность",
    legend_title="",
    bargap=0.02,
)

html_path, png_path, png_status = save_plotly_figure(
    fig,
    "01_histogram_semantic_balance_kde",
    width=1200,
    height=750,
)

# ------------------------------------------------------------
# Matplotlib PNG-дубль
# ------------------------------------------------------------

fig_m, ax = plt.subplots(figsize=(10, 6))

ax.hist(x, bins=40, density=True, alpha=0.65, edgecolor="black", linewidth=0.3)
ax.plot(x_grid, kde_values, linewidth=2.5)
ax.axvline(median_x, linestyle="--", linewidth=2)
ax.text(
    median_x,
    max(kde_values) * 0.95,
    f"Медиана = {median_x:.3f}",
    rotation=90,
    va="top",
    ha="right",
)

ax.set_title("Распределение semantic_balance")
ax.set_xlabel("semantic_balance = support_total − control_total")
ax.set_ylabel("Плотность")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

mpl_path = save_matplotlib_figure(fig_m, "01_histogram_semantic_balance_kde")

# Сохраняем данные графика
hist_data = pd.DataFrame({
    "bin_center": bin_centers,
    "density": hist_counts,
})
kde_data = pd.DataFrame({
    "semantic_balance": x_grid,
    "kde_density": kde_values,
})
hist_data_path = FIGURES_DIR / "01_histogram_semantic_balance_data.xlsx"

with pd.ExcelWriter(hist_data_path, engine="openpyxl") as writer:
    hist_data.to_excel(writer, sheet_name="histogram", index=False)
    kde_data.to_excel(writer, sheet_name="kde", index=False)
    pd.DataFrame({"metric": ["median"], "value": [median_x]}).to_excel(
        writer,
        sheet_name="median",
        index=False,
    )

print("✅ Histogram semantic_balance построен.")
print(f"n: {len(x)}")
print(f"median semantic_balance: {median_x:.4f}")
print(f"Plotly HTML: {html_path}")
print(f"Plotly PNG: {png_path} | {png_status}")
print(f"Matplotlib PNG: {mpl_path}")
print(f"Данные графика: {hist_data_path}")

fig.show()

In [ ]:
# ============================================================
# ЯЧЕЙКА 23. Ridgeplot semantic_balance по profession_group
# ============================================================

# Автономность ячейки
if "df_final" not in globals():
    df_final = pd.read_parquet(DATA_DIR / "09_final_indices.parquet")

ridge_value = "semantic_balance"
ridge_group = "profession_group"

groups = robust_group_order(
    df_final,
    group_col=ridge_group,
    value_col=ridge_value,
    min_n=20,
    top_n=12,
)

ridge_df = (
    df_final[df_final[ridge_group].isin(groups)]
    [[ridge_group, ridge_value]]
    .dropna()
    .copy()
)

# Порядок снизу вверх: от меньшей медианы к большей
group_medians = ridge_df.groupby(ridge_group)[ridge_value].median().sort_values()
groups_ordered = group_medians.index.tolist()

x_all = ridge_df[ridge_value].values
x_grid = np.linspace(np.quantile(x_all, 0.01), np.quantile(x_all, 0.99), 300)

# ------------------------------------------------------------
# Plotly ridgeplot
# ------------------------------------------------------------

fig = go.Figure()

for i, group in enumerate(groups_ordered):
    vals = ridge_df.loc[ridge_df[ridge_group] == group, ridge_value].values

    if len(vals) < 5:
        continue

    kde = gaussian_kde(vals)
    y = kde(x_grid)

    # Масштабируем высоту, чтобы линии не наезжали друг на друга слишком сильно
    y_scaled = y / y.max() * 0.75
    baseline = i

    fig.add_trace(go.Scatter(
        x=np.concatenate([x_grid, x_grid[::-1]]),
        y=np.concatenate([baseline + y_scaled, np.full_like(x_grid, baseline)]),
        fill="toself",
        mode="lines",
        name=f"{group} | n={len(vals)}",
        line=dict(width=1.2),
        hoverinfo="name+x+y",
    ))

    fig.add_trace(go.Scatter(
        x=[np.median(vals), np.median(vals)],
        y=[baseline, baseline + 0.75],
        mode="lines",
        showlegend=False,
        line=dict(width=1, dash="dash"),
        hoverinfo="skip",
    ))

fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Ridgeplot semantic_balance по укрупнённым группам профессий",
    xaxis_title="semantic_balance",
    yaxis=dict(
        tickmode="array",
        tickvals=list(range(len(groups_ordered))),
        ticktext=groups_ordered,
        title="profession_group",
    ),
    showlegend=False,
    height=max(650, 70 * len(groups_ordered)),
)

html_path, png_path, png_status = save_plotly_figure(
    fig,
    "02_ridgeplot_semantic_balance_by_profession_group",
    width=1300,
    height=max(800, 90 * len(groups_ordered)),
)

# ------------------------------------------------------------
# Matplotlib PNG-дубль
# ------------------------------------------------------------

fig_m, ax = plt.subplots(figsize=(11, max(6, 0.55 * len(groups_ordered))))

for i, group in enumerate(groups_ordered):
    vals = ridge_df.loc[ridge_df[ridge_group] == group, ridge_value].values

    if len(vals) < 5:
        continue

    kde = gaussian_kde(vals)
    y = kde(x_grid)
    y_scaled = y / y.max() * 0.75
    baseline = i

    ax.fill_between(x_grid, baseline, baseline + y_scaled, alpha=0.65)
    ax.plot(x_grid, baseline + y_scaled, linewidth=1.2)
    ax.axvline(np.median(vals), ymin=baseline / max(1, len(groups_ordered)), ymax=(baseline + 0.75) / max(1, len(groups_ordered)), linestyle="--", linewidth=1)

ax.set_yticks(range(len(groups_ordered)))
ax.set_yticklabels(groups_ordered)
ax.set_title("Ridgeplot semantic_balance по profession_group")
ax.set_xlabel("semantic_balance")
ax.set_ylabel("profession_group")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

mpl_path = save_matplotlib_figure(
    fig_m,
    "02_ridgeplot_semantic_balance_by_profession_group",
)

# Сохраняем таблицу медиан
ridge_table = (
    ridge_df
    .groupby(ridge_group, as_index=False)
    .agg(
        n=(ridge_value, "size"),
        median=(ridge_value, "median"),
        mean=(ridge_value, "mean"),
        q1=(ridge_value, lambda s: s.quantile(0.25)),
        q3=(ridge_value, lambda s: s.quantile(0.75)),
    )
    .sort_values("median", ascending=False)
)

ridge_table_path = FIGURES_DIR / "02_ridgeplot_semantic_balance_by_profession_group_data.xlsx"
ridge_table.to_excel(ridge_table_path, index=False)

print("✅ Ridgeplot построен.")
print(f"Группы включены: {groups_ordered}")
print(f"ridge_df shape: {ridge_df.shape}")
print(f"Plotly HTML: {html_path}")
print(f"Plotly PNG: {png_path} | {png_status}")
print(f"Matplotlib PNG: {mpl_path}")
print(f"Данные графика: {ridge_table_path}")

fig.show()

In [ ]:
# ============================================================
# ЯЧЕЙКА 24. Lollipop-график медиан 11 индексов с bootstrap CI
# ============================================================

# Автономность ячейки
if "df_final" not in globals():
    df_final = pd.read_parquet(DATA_DIR / "09_final_indices.parquet")

if "bootstrap_median_ci" not in globals():
    def bootstrap_median_ci(x, B=5000, random_state=42):
        x = pd.Series(x).dropna().astype(float).values
        if len(x) < 5:
            return np.nan, np.nan
        rng = np.random.default_rng(random_state)
        boot_medians = np.empty(B)
        for b in range(B):
            sample = rng.choice(x, size=len(x), replace=True)
            boot_medians[b] = np.median(sample)
        return float(np.percentile(boot_medians, 2.5)), float(np.percentile(boot_medians, 97.5))

lollipop_rows = []

for construct in ALL_CONSTRUCTS:
    var = f"{construct}_emb"
    x = pd.to_numeric(df_final[var], errors="coerce").dropna()
    ci_low, ci_high = bootstrap_median_ci(x, B=BOOTSTRAP_B, random_state=RANDOM_STATE)

    lollipop_rows.append({
        "construct": construct,
        "label_ru": construct_labels_ru.get(construct, construct),
        "variable": var,
        "n": len(x),
        "median": float(x.median()),
        "ci_low": ci_low,
        "ci_high": ci_high,
        "mean": float(x.mean()),
    })

lollipop_df = pd.DataFrame(lollipop_rows).sort_values("median", ascending=True)

# ------------------------------------------------------------
# Plotly
# ------------------------------------------------------------

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=lollipop_df["median"],
    y=lollipop_df["label_ru"],
    mode="markers",
    marker=dict(size=11),
    error_x=dict(
        type="data",
        symmetric=False,
        array=lollipop_df["ci_high"] - lollipop_df["median"],
        arrayminus=lollipop_df["median"] - lollipop_df["ci_low"],
        thickness=1.5,
        width=5,
    ),
    text=[
        f"{row.label_ru}<br>median={row.median:.3f}<br>95% CI [{row.ci_low:.3f}; {row.ci_high:.3f}]"
        for row in lollipop_df.itertuples()
    ],
    hoverinfo="text",
    name="Медиана и 95% CI",
))

for row in lollipop_df.itertuples():
    fig.add_shape(
        type="line",
        x0=0,
        x1=row.median,
        y0=row.label_ru,
        y1=row.label_ru,
        line=dict(width=1),
    )

fig.add_vline(x=0, line_width=1, line_dash="dash")

fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Медианы 11 эмбеддинговых индексов с bootstrap 95% CI",
    xaxis_title="Косинусная близость к якорному центроиду",
    yaxis_title="Индекс",
    height=700,
    showlegend=False,
)

html_path, png_path, png_status = save_plotly_figure(
    fig,
    "03_lollipop_median_indices_ci",
    width=1200,
    height=800,
)

# ------------------------------------------------------------
# Matplotlib PNG-дубль
# ------------------------------------------------------------

fig_m, ax = plt.subplots(figsize=(10, 7))

y_pos = np.arange(len(lollipop_df))

ax.hlines(
    y=y_pos,
    xmin=0,
    xmax=lollipop_df["median"],
    linewidth=1.5,
)

ax.errorbar(
    x=lollipop_df["median"],
    y=y_pos,
    xerr=[
        lollipop_df["median"] - lollipop_df["ci_low"],
        lollipop_df["ci_high"] - lollipop_df["median"],
    ],
    fmt="o",
    capsize=4,
)

ax.axvline(0, linestyle="--", linewidth=1)
ax.set_yticks(y_pos)
ax.set_yticklabels(lollipop_df["label_ru"])
ax.set_title("Медианы 11 эмбеддинговых индексов с bootstrap 95% CI")
ax.set_xlabel("Косинусная близость к якорному центроиду")
ax.set_ylabel("Индекс")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

mpl_path = save_matplotlib_figure(fig_m, "03_lollipop_median_indices_ci")

# Сохраняем данные
lollipop_data_path = FIGURES_DIR / "03_lollipop_median_indices_ci_data.xlsx"
lollipop_df.to_excel(lollipop_data_path, index=False)

# Таблица
thesis_lollipop_path = TABLES_DIR / "table_overall_index_medians_ci.xlsx"
lollipop_df.to_excel(thesis_lollipop_path, index=False)

print("✅ Lollipop-график построен.")
print(f"lollipop_df shape: {lollipop_df.shape}")
print(f"Plotly HTML: {html_path}")
print(f"Plotly PNG: {png_path} | {png_status}")
print(f"Matplotlib PNG: {mpl_path}")
print(f"Данные графика: {lollipop_data_path}")
print(f"Таблица: {thesis_lollipop_path}")

fig.show()

In [ ]:
# ============================================================
# ЯЧЕЙКА 25. Heatmap profession_group × index
# ============================================================

# Автономность ячейки
if "df_final" not in globals():
    df_final = pd.read_parquet(DATA_DIR / "09_final_indices.parquet")

heatmap_group = "profession_group"
heatmap_vars = [f"{c}_emb" for c in ALL_CONSTRUCTS]

# Берём только группы с n >= 20, чтобы heatmap не был шумным
group_counts = df_final[heatmap_group].value_counts()
valid_groups = group_counts[group_counts >= 20].index.tolist()

heatmap_df = df_final[df_final[heatmap_group].isin(valid_groups)].copy()

heatmap_table = (
    heatmap_df
    .groupby(heatmap_group)[heatmap_vars]
    .median()
)

# Сортируем строки по semantic_balance медиане
row_order = (
    heatmap_df
    .groupby(heatmap_group)["semantic_balance"]
    .median()
    .sort_values(ascending=False)
    .index
    .tolist()
)

heatmap_table = heatmap_table.loc[row_order]

# Переименовываем столбцы для читаемости
heatmap_table_ru = heatmap_table.copy()
heatmap_table_ru.columns = [
    construct_labels_ru.get(c.replace("_emb", ""), c)
    for c in heatmap_table_ru.columns
]

# ------------------------------------------------------------
# Plotly heatmap
# ------------------------------------------------------------

fig = px.imshow(
    heatmap_table_ru,
    aspect="auto",
    text_auto=".2f",
    template=PLOTLY_TEMPLATE,
    title="Медианные значения индексов по profession_group",
    labels=dict(x="Индекс", y="profession_group", color="Медиана"),
)

fig.update_layout(
    height=max(650, 45 * len(heatmap_table_ru)),
    xaxis_tickangle=-35,
)

html_path, png_path, png_status = save_plotly_figure(
    fig,
    "04_heatmap_profession_group_by_index",
    width=1400,
    height=max(800, 70 * len(heatmap_table_ru)),
)

# ------------------------------------------------------------
# Matplotlib PNG-дубль
# ------------------------------------------------------------

fig_m, ax = plt.subplots(figsize=(13, max(6, 0.55 * len(heatmap_table_ru))))

im = ax.imshow(heatmap_table_ru.values, aspect="auto")

ax.set_xticks(np.arange(len(heatmap_table_ru.columns)))
ax.set_xticklabels(heatmap_table_ru.columns, rotation=35, ha="right")
ax.set_yticks(np.arange(len(heatmap_table_ru.index)))
ax.set_yticklabels(heatmap_table_ru.index)

for i in range(heatmap_table_ru.shape[0]):
    for j in range(heatmap_table_ru.shape[1]):
        ax.text(
            j,
            i,
            f"{heatmap_table_ru.values[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=8,
        )

ax.set_title("Медианные значения индексов по profession_group")
ax.set_xlabel("Индекс")
ax.set_ylabel("profession_group")
fig_m.colorbar(im, ax=ax, fraction=0.03, pad=0.02)

mpl_path = save_matplotlib_figure(fig_m, "04_heatmap_profession_group_by_index")

# Сохраняем данные
heatmap_data_path = FIGURES_DIR / "04_heatmap_profession_group_by_index_data.xlsx"

with pd.ExcelWriter(heatmap_data_path, engine="openpyxl") as writer:
    heatmap_table.reset_index().to_excel(writer, sheet_name="median_original", index=False)
    heatmap_table_ru.reset_index().to_excel(writer, sheet_name="median_ru", index=False)
    group_counts.reset_index().rename(columns={"index": heatmap_group, heatmap_group: "n"}).to_excel(
        writer,
        sheet_name="group_counts",
        index=False,
    )

# Таблица
thesis_heatmap_path = TABLES_DIR / "table_profession_group_index_medians.xlsx"
heatmap_table_ru.reset_index().to_excel(thesis_heatmap_path, index=False)

print("✅ Heatmap profession_group × index построен.")
print(f"heatmap_table shape: {heatmap_table.shape}")
print(f"Группы включены: {list(heatmap_table.index)}")
print(f"Plotly HTML: {html_path}")
print(f"Plotly PNG: {png_path} | {png_status}")
print(f"Matplotlib PNG: {mpl_path}")
print(f"Данные графика: {heatmap_data_path}")
print(f"Таблица: {thesis_heatmap_path}")

fig.show()

In [ ]:
# ============================================================
# ЯЧЕЙКА 26. Квадрантный scatter support × control
# Две версии окраски: по profession_group и по ai_mention
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# ------------------------------------------------------------
# Автономность ячейки
# ------------------------------------------------------------

if "DATA_DIR" not in globals():
    DATA_DIR = Path("/content/output/data")
if "FIGURES_DIR" not in globals():
    FIGURES_DIR = Path("/content/output/figures")
if "TABLES_DIR" not in globals():
    TABLES_DIR = Path("/content/output/tables_for_thesis")

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

if "df_final" not in globals():
    final_parquet_path = DATA_DIR / "09_final_indices.parquet"
    if not final_parquet_path.exists():
        raise FileNotFoundError("Не найден /content/output/data/09_final_indices.parquet. Сначала запустите ячейки до 14.")
    df_final = pd.read_parquet(final_parquet_path)

# ------------------------------------------------------------
# Безопасное сохранение Plotly-графиков
# Исправляет ошибку: Type is not JSON serializable: NAType
# ------------------------------------------------------------

def clean_plotly_json(obj):
    """
    Рекурсивно заменяет pd.NA / NaN / inf / numpy-типы на JSON-safe значения.
    """
    if isinstance(obj, dict):
        return {k: clean_plotly_json(v) for k, v in obj.items()}

    if isinstance(obj, list):
        return [clean_plotly_json(v) for v in obj]

    if isinstance(obj, tuple):
        return [clean_plotly_json(v) for v in obj]

    if isinstance(obj, np.ndarray):
        return clean_plotly_json(obj.tolist())

    if obj is pd.NA:
        return None

    if isinstance(obj, np.generic):
        obj = obj.item()

    if isinstance(obj, float):
        if np.isnan(obj) or np.isinf(obj):
            return None

    return obj

def make_plotly_safe(fig):
    fig_json = fig.to_plotly_json()
    fig_json_clean = clean_plotly_json(fig_json)
    return go.Figure(fig_json_clean)

def save_plotly_figure(fig, filename, width=1200, height=800, scale=2):
    """
    Сохраняет Plotly-график в HTML и PNG.
    Если PNG не сохранился из-за kaleido, HTML всё равно сохранится.
    """
    safe_fig = make_plotly_safe(fig)

    html_path = FIGURES_DIR / f"{filename}.html"
    png_path = FIGURES_DIR / f"{filename}.png"

    safe_fig.write_html(str(html_path))

    png_status = "not_saved"
    try:
        safe_fig.write_image(
            str(png_path),
            width=width,
            height=height,
            scale=scale,
        )
        png_status = "saved"
    except Exception as e:
        png_status = f"png_not_saved: {repr(e)}"

    return html_path, png_path, png_status

# ------------------------------------------------------------
# Подготовка данных
# ------------------------------------------------------------

needed_cols = [
    "vacancy_id",
    "url",
    "company",
    "query_keyword",
    "profession_group",
    "creative_segment",
    "city_group",
    "experience_text",
    "ai_mention",
    "quadrant_type",
    "support_total",
    "control_total",
    "semantic_balance",
]

missing_cols = [c for c in needed_cols if c not in df_final.columns]
if missing_cols:
    raise ValueError(f"В df_final отсутствуют нужные колонки: {missing_cols}")

scatter_df = df_final[needed_cols].copy()

# Чистим числовые поля
for col in ["support_total", "control_total", "semantic_balance"]:
    scatter_df[col] = pd.to_numeric(scatter_df[col], errors="coerce")
    scatter_df[col] = scatter_df[col].replace([np.inf, -np.inf], np.nan)

# Убираем строки без координат
scatter_df = scatter_df.dropna(subset=["support_total", "control_total"]).copy()

# Чистим категориальные поля от pd.NA
categorical_cols = [
    "vacancy_id",
    "url",
    "company",
    "query_keyword",
    "profession_group",
    "creative_segment",
    "city_group",
    "experience_text",
    "quadrant_type",
]

for col in categorical_cols:
    scatter_df[col] = (
        scatter_df[col]
        .astype("object")
        .where(scatter_df[col].notna(), "не указано")
        .astype(str)
    )

scatter_df["ai_mention_label"] = np.where(
    scatter_df["ai_mention"].fillna(False).astype(bool),
    "AI mentioned",
    "No AI mention"
)

support_median = scatter_df["support_total"].median()
control_median = scatter_df["control_total"].median()

quadrant_summary = (
    scatter_df
    .groupby("quadrant_type", dropna=False)
    .agg(
        n=("vacancy_id", "count"),
        support_median=("support_total", "median"),
        control_median=("control_total", "median"),
        semantic_balance_median=("semantic_balance", "median"),
        ai_share=("ai_mention", "mean"),
    )
    .reset_index()
    .sort_values("n", ascending=False)
)

# ------------------------------------------------------------
# Plotly scatter: окраска по profession_group
# ------------------------------------------------------------

fig_prof = px.scatter(
    scatter_df,
    x="support_total",
    y="control_total",
    color="profession_group",
    hover_data={
        "vacancy_id": True,
        "company": True,
        "query_keyword": True,
        "creative_segment": True,
        "city_group": True,
        "experience_text": True,
        "quadrant_type": True,
        "semantic_balance": ":.3f",
        "support_total": ":.3f",
        "control_total": ":.3f",
        "url": True,
    },
    title="Support × Control: квадрантная карта вакансий, окраска по профессии",
    labels={
        "support_total": "Support total",
        "control_total": "Control total",
        "profession_group": "Profession group",
    },
    opacity=0.65,
)

fig_prof.add_vline(x=support_median, line_dash="dash")
fig_prof.add_hline(y=control_median, line_dash="dash")
fig_prof.update_layout(
    template="plotly_white",
    height=850,
    legend_title_text="Profession group",
)

# ------------------------------------------------------------
# Plotly scatter: окраска по ai_mention
# ------------------------------------------------------------

fig_ai = px.scatter(
    scatter_df,
    x="support_total",
    y="control_total",
    color="ai_mention_label",
    hover_data={
        "vacancy_id": True,
        "company": True,
        "query_keyword": True,
        "profession_group": True,
        "creative_segment": True,
        "city_group": True,
        "quadrant_type": True,
        "semantic_balance": ":.3f",
        "support_total": ":.3f",
        "control_total": ":.3f",
        "url": True,
    },
    title="Support × Control: квадрантная карта вакансий, окраска по упоминанию AI",
    labels={
        "support_total": "Support total",
        "control_total": "Control total",
        "ai_mention_label": "AI mention",
    },
    opacity=0.65,
)

fig_ai.add_vline(x=support_median, line_dash="dash")
fig_ai.add_hline(y=control_median, line_dash="dash")
fig_ai.update_layout(
    template="plotly_white",
    height=850,
    legend_title_text="AI mention",
)

# ------------------------------------------------------------
# Сохранение Plotly
# ------------------------------------------------------------

html_prof, png_prof, png_prof_status = save_plotly_figure(
    fig_prof,
    "05_quadrant_scatter_support_control_by_profession",
)

html_ai, png_ai, png_ai_status = save_plotly_figure(
    fig_ai,
    "05_quadrant_scatter_support_control_by_ai",
)

# ------------------------------------------------------------
# Matplotlib PNG-дубли
# ------------------------------------------------------------

mpl_prof_path = FIGURES_DIR / "05_quadrant_scatter_support_control_by_profession_matplotlib.png"

plt.figure(figsize=(11, 8))
for group, group_df in scatter_df.groupby("profession_group"):
    plt.scatter(
        group_df["support_total"],
        group_df["control_total"],
        alpha=0.55,
        s=18,
        label=group,
    )

plt.axvline(support_median, linestyle="--")
plt.axhline(control_median, linestyle="--")
plt.xlabel("Support total")
plt.ylabel("Control total")
plt.title("Support × Control: окраска по profession_group")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(mpl_prof_path, dpi=300, bbox_inches="tight")
plt.close()

mpl_ai_path = FIGURES_DIR / "05_quadrant_scatter_support_control_by_ai_matplotlib.png"

plt.figure(figsize=(10, 8))
for group, group_df in scatter_df.groupby("ai_mention_label"):
    plt.scatter(
        group_df["support_total"],
        group_df["control_total"],
        alpha=0.6,
        s=18,
        label=group,
    )

plt.axvline(support_median, linestyle="--")
plt.axhline(control_median, linestyle="--")
plt.xlabel("Support total")
plt.ylabel("Control total")
plt.title("Support × Control: окраска по AI mention")
plt.legend()
plt.tight_layout()
plt.savefig(mpl_ai_path, dpi=300, bbox_inches="tight")
plt.close()

# ------------------------------------------------------------
# Сохранение данных для графика
# ------------------------------------------------------------

scatter_data_path = TABLES_DIR / "figure_05_quadrant_scatter_data.xlsx"

with pd.ExcelWriter(scatter_data_path, engine="openpyxl") as writer:
    scatter_df.to_excel(writer, sheet_name="scatter_data", index=False)
    quadrant_summary.to_excel(writer, sheet_name="quadrant_summary", index=False)

# ------------------------------------------------------------
# Sanity-check
# ------------------------------------------------------------

print("✅ Квадрантные scatter-графики построены.")
print(f"scatter_df shape: {scatter_df.shape}")
print(f"support_median: {support_median:.4f}")
print(f"control_median: {control_median:.4f}")
print(f"Plotly profession HTML: {html_prof}")
print(f"Plotly profession PNG: {png_prof} | {png_prof_status}")
print(f"Plotly AI HTML: {html_ai}")
print(f"Plotly AI PNG: {png_ai} | {png_ai_status}")
print(f"Matplotlib profession PNG: {mpl_prof_path}")
print(f"Matplotlib AI PNG: {mpl_ai_path}")
print(f"Данные графиков: {scatter_data_path}")

fig_prof.show()
fig_ai.show()

In [ ]:
# ============================================================
# ЯЧЕЙКА 27. Small multiples: AI vs non-AI по 11 индексам
# ============================================================

# Автономность ячейки
if "df_final" not in globals():
    df_final = pd.read_parquet(DATA_DIR / "09_final_indices.parquet")

if "save_plotly_figure" not in globals():
    raise RuntimeError("Сначала запустите ячейку 21 с helper-функциями для визуализаций.")

ai_long = df_final[
    ["vacancy_id", "ai_mention"] + [f"{c}_emb" for c in ALL_CONSTRUCTS]
].copy()

ai_long["ai_group"] = ai_long["ai_mention"].map({
    True: "AI mentioned",
    False: "No AI mention",
})

ai_long = ai_long.melt(
    id_vars=["vacancy_id", "ai_mention", "ai_group"],
    value_vars=[f"{c}_emb" for c in ALL_CONSTRUCTS],
    var_name="index_variable",
    value_name="index_value",
).dropna(subset=["index_value"])

ai_long["construct"] = ai_long["index_variable"].str.replace("_emb", "", regex=False)
ai_long["construct_label_ru"] = ai_long["construct"].map(construct_labels_ru)

# ------------------------------------------------------------
# Таблица медиан AI vs non-AI
# ------------------------------------------------------------

ai_summary = (
    ai_long
    .groupby(["construct", "construct_label_ru", "ai_group"], as_index=False)
    .agg(
        n=("index_value", "size"),
        median=("index_value", "median"),
        mean=("index_value", "mean"),
        q1=("index_value", lambda s: s.quantile(0.25)),
        q3=("index_value", lambda s: s.quantile(0.75)),
    )
)

# ------------------------------------------------------------
# Plotly small multiples: boxplots по индексам
# ------------------------------------------------------------

fig = px.box(
    ai_long,
    x="ai_group",
    y="index_value",
    facet_col="construct_label_ru",
    facet_col_wrap=4,
    points=False,
    template=PLOTLY_TEMPLATE,
    title="AI vs non-AI: распределения 11 эмбеддинговых индексов",
    labels={
        "ai_group": "",
        "index_value": "Значение индекса",
        "construct_label_ru": "",
    },
)

fig.update_layout(
    height=950,
    showlegend=False,
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

html_path, png_path, png_status = save_plotly_figure(
    fig,
    "07_small_multiples_ai_vs_non_ai_indices",
    width=1500,
    height=1000,
)

# ------------------------------------------------------------
# Matplotlib PNG-дубль
# ------------------------------------------------------------

n_constructs = len(ALL_CONSTRUCTS)
n_cols = 4
n_rows = math.ceil(n_constructs / n_cols)

fig_m, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3.2 * n_rows))
axes = axes.flatten()

for ax, construct in zip(axes, ALL_CONSTRUCTS):
    subset = ai_long[ai_long["construct"] == construct].copy()

    data_no_ai = subset.loc[subset["ai_group"] == "No AI mention", "index_value"].dropna()
    data_ai = subset.loc[subset["ai_group"] == "AI mentioned", "index_value"].dropna()

    ax.boxplot(
        [data_no_ai, data_ai],
        labels=["No AI", "AI"],
        showfliers=False,
    )

    ax.set_title(construct_labels_ru.get(construct, construct), fontsize=10)
    ax.set_ylabel("index")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# Удаляем пустые оси
for ax in axes[len(ALL_CONSTRUCTS):]:
    fig_m.delaxes(ax)

fig_m.suptitle("AI vs non-AI: распределения 11 эмбеддинговых индексов", y=1.02)
fig_m.tight_layout()

mpl_path = save_matplotlib_figure(
    fig_m,
    "07_small_multiples_ai_vs_non_ai_indices",
)

# ------------------------------------------------------------
# Сохранение данных
# ------------------------------------------------------------

ai_small_multiples_data_path = FIGURES_DIR / "07_small_multiples_ai_vs_non_ai_indices_data.xlsx"

with pd.ExcelWriter(ai_small_multiples_data_path, engine="openpyxl") as writer:
    ai_summary.to_excel(writer, sheet_name="summary", index=False)
    ai_long.to_excel(writer, sheet_name="long_data", index=False)

thesis_ai_summary_path = TABLES_DIR / "table_ai_vs_non_ai_index_medians.xlsx"
ai_summary.to_excel(thesis_ai_summary_path, index=False)

print("✅ Small multiples AI vs non-AI построены.")
print(f"ai_long shape: {ai_long.shape}")
print(f"AI mentioned vacancies: {int(df_final['ai_mention'].sum())} из {len(df_final)}")
print(f"Plotly HTML: {html_path}")
print(f"Plotly PNG: {png_path} | {png_status}")
print(f"Matplotlib PNG: {mpl_path}")
print(f"Данные графика: {ai_small_multiples_data_path}")
print(f"Таблица: {thesis_ai_summary_path}")

fig.show()

In [ ]:
# ============================================================
# ЯЧЕЙКА 28. Bar chart долей quadrant_type по creative_segment
# ============================================================

# Автономность ячейки
if "df_final" not in globals():
    df_final = pd.read_parquet(DATA_DIR / "09_final_indices.parquet")

if "save_plotly_figure" not in globals():
    raise RuntimeError("Сначала запустите ячейку 21 с helper-функциями для визуализаций.")

quad_segment = (
    df_final
    .groupby(["creative_segment", "quadrant_type"], as_index=False)
    .agg(n=("vacancy_id", "count"))
)

quad_segment["segment_total"] = quad_segment.groupby("creative_segment")["n"].transform("sum")
quad_segment["share"] = quad_segment["n"] / quad_segment["segment_total"]

quadrant_order = [
    "creative_freedom",
    "enabling_bureaucracy",
    "production_line",
    "thin_text",
]

quad_segment["quadrant_type"] = pd.Categorical(
    quad_segment["quadrant_type"],
    categories=quadrant_order,
    ordered=True,
)

quad_segment = quad_segment.sort_values(["creative_segment", "quadrant_type"])

# ------------------------------------------------------------
# Plotly stacked bar
# ------------------------------------------------------------

fig = px.bar(
    quad_segment,
    x="creative_segment",
    y="share",
    color="quadrant_type",
    text=quad_segment["share"].map(lambda x: f"{x:.1%}"),
    category_orders={"quadrant_type": quadrant_order},
    template=PLOTLY_TEMPLATE,
    title="Доли quadrant_type внутри creative_segment",
    labels={
        "creative_segment": "creative_segment",
        "share": "Доля внутри сегмента",
        "quadrant_type": "quadrant_type",
    },
)

fig.update_layout(
    yaxis_tickformat=".0%",
    height=700,
    barmode="stack",
)

fig.update_traces(textposition="inside")

html_path, png_path, png_status = save_plotly_figure(
    fig,
    "08_bar_quadrant_share_by_creative_segment",
    width=1200,
    height=750,
)

# ------------------------------------------------------------
# Matplotlib PNG-дубль
# ------------------------------------------------------------

pivot_share = (
    quad_segment
    .pivot(index="creative_segment", columns="quadrant_type", values="share")
    .fillna(0)
    .reindex(columns=quadrant_order)
)

fig_m, ax = plt.subplots(figsize=(10, 6))

bottom = np.zeros(len(pivot_share))

for col in pivot_share.columns:
    values = pivot_share[col].values
    ax.bar(pivot_share.index, values, bottom=bottom, label=col)

    for i, v in enumerate(values):
        if v >= 0.04:
            ax.text(
                i,
                bottom[i] + v / 2,
                f"{v:.0%}",
                ha="center",
                va="center",
                fontsize=9,
            )

    bottom += values

ax.set_title("Доли quadrant_type внутри creative_segment")
ax.set_xlabel("creative_segment")
ax.set_ylabel("Доля внутри сегмента")
ax.set_ylim(0, 1)
ax.legend(title="quadrant_type", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

mpl_path = save_matplotlib_figure(
    fig_m,
    "08_bar_quadrant_share_by_creative_segment",
)

# ------------------------------------------------------------
# Сохранение данных
# ------------------------------------------------------------

quad_segment_data_path = FIGURES_DIR / "08_bar_quadrant_share_by_creative_segment_data.xlsx"

with pd.ExcelWriter(quad_segment_data_path, engine="openpyxl") as writer:
    quad_segment.to_excel(writer, sheet_name="long_data", index=False)
    pivot_share.reset_index().to_excel(writer, sheet_name="pivot_share", index=False)

thesis_quad_segment_path = TABLES_DIR / "table_quadrant_share_by_creative_segment.xlsx"
quad_segment.to_excel(thesis_quad_segment_path, index=False)

print("✅ Bar chart долей quadrant_type по creative_segment построен.")
print(f"quad_segment shape: {quad_segment.shape}")
print(f"Plotly HTML: {html_path}")
print(f"Plotly PNG: {png_path} | {png_status}")
print(f"Matplotlib PNG: {mpl_path}")
print(f"Данные графика: {quad_segment_data_path}")
print(f"Таблица: {thesis_quad_segment_path}")

fig.show()

In [ ]:
# ============================================================
# ЯЧЕЙКА 29. Rolling convergence медианы semantic_balance
# ============================================================
# Смысл: показать, стабилизируется ли оценка медианы semantic_balance
# по мере увеличения размера выборки.

# Автономность ячейки
if "df_final" not in globals():
    df_final = pd.read_parquet(DATA_DIR / "09_final_indices.parquet")

if "save_plotly_figure" not in globals():
    raise RuntimeError("Сначала запустите ячейку 21 с helper-функциями для визуализаций.")

rolling_var = "semantic_balance"

rolling_base = (
    df_final[["vacancy_id", rolling_var]]
    .dropna()
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

# Чтобы график не был слишком тяжёлым, считаем точки каждые 25 наблюдений
step = 25
sample_sizes = list(range(step, len(rolling_base) + 1, step))

if sample_sizes[-1] != len(rolling_base):
    sample_sizes.append(len(rolling_base))

rolling_rows = []

for n in tqdm(sample_sizes, desc="Rolling convergence"):
    sample = rolling_base.iloc[:n]
    rolling_rows.append({
        "sample_size": n,
        "rolling_median": float(sample[rolling_var].median()),
        "rolling_mean": float(sample[rolling_var].mean()),
    })

rolling_df = pd.DataFrame(rolling_rows)

full_median = float(rolling_base[rolling_var].median())
full_mean = float(rolling_base[rolling_var].mean())

# ------------------------------------------------------------
# Plotly
# ------------------------------------------------------------

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=rolling_df["sample_size"],
    y=rolling_df["rolling_median"],
    mode="lines",
    name="Rolling median",
    line=dict(width=3),
))

fig.add_trace(go.Scatter(
    x=rolling_df["sample_size"],
    y=rolling_df["rolling_mean"],
    mode="lines",
    name="Rolling mean",
    line=dict(width=2, dash="dot"),
))

fig.add_hline(
    y=full_median,
    line_dash="dash",
    line_width=2,
    annotation_text=f"Full-sample median = {full_median:.3f}",
    annotation_position="bottom right",
)

fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Rolling convergence: медиана semantic_balance по мере роста выборки",
    xaxis_title="Размер накопленной выборки",
    yaxis_title="semantic_balance",
    height=700,
)

html_path, png_path, png_status = save_plotly_figure(
    fig,
    "09_rolling_convergence_semantic_balance_median",
    width=1200,
    height=750,
)

# ------------------------------------------------------------
# Matplotlib PNG-дубль
# ------------------------------------------------------------

fig_m, ax = plt.subplots(figsize=(10, 6))

ax.plot(
    rolling_df["sample_size"],
    rolling_df["rolling_median"],
    linewidth=2.5,
    label="Rolling median",
)

ax.plot(
    rolling_df["sample_size"],
    rolling_df["rolling_mean"],
    linewidth=2,
    linestyle=":",
    label="Rolling mean",
)

ax.axhline(full_median, linestyle="--", linewidth=2, label=f"Full median = {full_median:.3f}")

ax.set_title("Rolling convergence: медиана semantic_balance")
ax.set_xlabel("Размер накопленной выборки")
ax.set_ylabel("semantic_balance")
ax.legend()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

mpl_path = save_matplotlib_figure(
    fig_m,
    "09_rolling_convergence_semantic_balance_median",
)

# ------------------------------------------------------------
# Сохранение данных
# ------------------------------------------------------------

rolling_data_path = FIGURES_DIR / "09_rolling_convergence_semantic_balance_median_data.xlsx"

with pd.ExcelWriter(rolling_data_path, engine="openpyxl") as writer:
    rolling_df.to_excel(writer, sheet_name="rolling_convergence", index=False)
    pd.DataFrame({
        "metric": ["full_median", "full_mean", "n"],
        "value": [full_median, full_mean, len(rolling_base)],
    }).to_excel(writer, sheet_name="full_sample", index=False)

print("✅ Rolling convergence построен.")
print(f"rolling_df shape: {rolling_df.shape}")
print(f"Full-sample median: {full_median:.4f}")
print(f"Full-sample mean: {full_mean:.4f}")
print(f"Plotly HTML: {html_path}")
print(f"Plotly PNG: {png_path} | {png_status}")
print(f"Matplotlib PNG: {mpl_path}")
print(f"Данные графика: {rolling_data_path}")

fig.show()

In [ ]:
# ============================================================
# ЯЧЕЙКА 30. Bootstrap distribution медианы semantic_balance
# ============================================================

# Автономность ячейки
if "df_final" not in globals():
    df_final = pd.read_parquet(DATA_DIR / "09_final_indices.parquet")

if "save_plotly_figure" not in globals():
    raise RuntimeError("Сначала запустите ячейку 21 с helper-функциями для визуализаций.")

bootstrap_var = "semantic_balance"
BOOTSTRAP_MEDIAN_PATH = DATA_DIR / "15_bootstrap_distribution_semantic_balance_median.parquet"

x = pd.to_numeric(df_final[bootstrap_var], errors="coerce").dropna().values

if BOOTSTRAP_MEDIAN_PATH.exists():
    boot_df = pd.read_parquet(BOOTSTRAP_MEDIAN_PATH)
    print("✅ Найден готовый bootstrap-файл. Повторный расчёт не выполнялся.")
else:
    rng = np.random.default_rng(RANDOM_STATE)
    boot_medians = np.empty(BOOTSTRAP_B)

    for b in tqdm(range(BOOTSTRAP_B), desc="Bootstrap median semantic_balance"):
        sample = rng.choice(x, size=len(x), replace=True)
        boot_medians[b] = np.median(sample)

    boot_df = pd.DataFrame({
        "bootstrap_iteration": np.arange(1, BOOTSTRAP_B + 1),
        "median": boot_medians,
    })

    boot_df.to_parquet(BOOTSTRAP_MEDIAN_PATH, index=False)

median_point = float(np.median(x))
ci_low, ci_high = np.percentile(boot_df["median"], [2.5, 97.5])

# ------------------------------------------------------------
# Plotly
# ------------------------------------------------------------

fig = px.histogram(
    boot_df,
    x="median",
    nbins=50,
    histnorm="probability density",
    template=PLOTLY_TEMPLATE,
    title="Bootstrap distribution медианы semantic_balance",
    labels={
        "median": "Bootstrap-медиана semantic_balance",
        "count": "Плотность",
    },
)

fig.add_vline(
    x=median_point,
    line_width=2,
    line_dash="dash",
    annotation_text=f"Observed median = {median_point:.3f}",
    annotation_position="top right",
)

fig.add_vline(
    x=ci_low,
    line_width=1.5,
    line_dash="dot",
    annotation_text=f"2.5% = {ci_low:.3f}",
    annotation_position="bottom left",
)

fig.add_vline(
    x=ci_high,
    line_width=1.5,
    line_dash="dot",
    annotation_text=f"97.5% = {ci_high:.3f}",
    annotation_position="bottom right",
)

fig.update_layout(
    height=700,
    bargap=0.02,
)

html_path, png_path, png_status = save_plotly_figure(
    fig,
    "10_bootstrap_distribution_semantic_balance_median",
    width=1200,
    height=750,
)

# ------------------------------------------------------------
# Matplotlib PNG-дубль
# ------------------------------------------------------------

fig_m, ax = plt.subplots(figsize=(10, 6))

ax.hist(
    boot_df["median"],
    bins=50,
    density=True,
    alpha=0.75,
    edgecolor="black",
    linewidth=0.3,
)

ax.axvline(median_point, linestyle="--", linewidth=2, label=f"Observed median = {median_point:.3f}")
ax.axvline(ci_low, linestyle=":", linewidth=2, label=f"2.5% = {ci_low:.3f}")
ax.axvline(ci_high, linestyle=":", linewidth=2, label=f"97.5% = {ci_high:.3f}")

ax.set_title("Bootstrap distribution медианы semantic_balance")
ax.set_xlabel("Bootstrap-медиана semantic_balance")
ax.set_ylabel("Плотность")
ax.legend()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

mpl_path = save_matplotlib_figure(
    fig_m,
    "10_bootstrap_distribution_semantic_balance_median",
)

# ------------------------------------------------------------
# Сохранение данных
# ------------------------------------------------------------

bootstrap_data_path = FIGURES_DIR / "10_bootstrap_distribution_semantic_balance_median_data.xlsx"

bootstrap_summary = pd.DataFrame({
    "metric": [
        "observed_median",
        "bootstrap_mean_median",
        "bootstrap_std_median",
        "ci_low_2_5",
        "ci_high_97_5",
        "B",
        "n",
    ],
    "value": [
        median_point,
        float(boot_df["median"].mean()),
        float(boot_df["median"].std(ddof=1)),
        float(ci_low),
        float(ci_high),
        int(BOOTSTRAP_B),
        int(len(x)),
    ],
})

with pd.ExcelWriter(bootstrap_data_path, engine="openpyxl") as writer:
    bootstrap_summary.to_excel(writer, sheet_name="summary", index=False)
    boot_df.to_excel(writer, sheet_name="bootstrap_distribution", index=False)

print("✅ Bootstrap distribution медианы semantic_balance построен.")
print(f"boot_df shape: {boot_df.shape}")
print(f"Observed median: {median_point:.4f}")
print(f"95% CI: [{ci_low:.4f}; {ci_high:.4f}]")
print(f"Plotly HTML: {html_path}")
print(f"Plotly PNG: {png_path} | {png_status}")
print(f"Matplotlib PNG: {mpl_path}")
print(f"Parquet: {BOOTSTRAP_MEDIAN_PATH}")
print(f"Данные графика: {bootstrap_data_path}")

fig.show()

In [ ]:
# ============================================================
# ЯЧЕЙКА 31. Correlation heatmap 11 × 11 для эмбеддинговых индексов
# ============================================================

# Автономность ячейки
if "df_final" not in globals():
    df_final = pd.read_parquet(DATA_DIR / "09_final_indices.parquet")

if "save_plotly_figure" not in globals():
    raise RuntimeError("Сначала запустите ячейку 21 с helper-функциями для визуализаций.")

corr_vars = [f"{c}_emb" for c in ALL_CONSTRUCTS]

corr_data = df_final[corr_vars].apply(pd.to_numeric, errors="coerce")

spearman_corr = corr_data.corr(method="spearman")

# Переименовываем для графика
corr_labels = [
    construct_labels_ru.get(c.replace("_emb", ""), c)
    for c in spearman_corr.columns
]

spearman_corr_ru = spearman_corr.copy()
spearman_corr_ru.index = corr_labels
spearman_corr_ru.columns = corr_labels

# ------------------------------------------------------------
# Plotly
# ------------------------------------------------------------

fig = px.imshow(
    spearman_corr_ru,
    zmin=-1,
    zmax=1,
    text_auto=".2f",
    aspect="auto",
    template=PLOTLY_TEMPLATE,
    title="Spearman correlation heatmap: 11 эмбеддинговых индексов",
    labels=dict(x="Индекс", y="Индекс", color="ρ Spearman"),
)

fig.update_layout(
    height=850,
    width=950,
    xaxis_tickangle=-35,
)

html_path, png_path, png_status = save_plotly_figure(
    fig,
    "11_correlation_heatmap_11_embedding_indices",
    width=1200,
    height=1000,
)

# ------------------------------------------------------------
# Matplotlib PNG-дубль
# ------------------------------------------------------------

fig_m, ax = plt.subplots(figsize=(10, 9))

im = ax.imshow(
    spearman_corr_ru.values,
    vmin=-1,
    vmax=1,
    aspect="auto",
)

ax.set_xticks(np.arange(len(corr_labels)))
ax.set_yticks(np.arange(len(corr_labels)))
ax.set_xticklabels(corr_labels, rotation=35, ha="right")
ax.set_yticklabels(corr_labels)

for i in range(len(corr_labels)):
    for j in range(len(corr_labels)):
        ax.text(
            j,
            i,
            f"{spearman_corr_ru.values[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=8,
        )

ax.set_title("Spearman correlation heatmap: 11 эмбеддинговых индексов")
fig_m.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

mpl_path = save_matplotlib_figure(
    fig_m,
    "11_correlation_heatmap_11_embedding_indices",
)

# ------------------------------------------------------------
# Сохранение данных
# ------------------------------------------------------------

corr_heatmap_data_path = FIGURES_DIR / "11_correlation_heatmap_11_embedding_indices_data.xlsx"

corr_long = (
    spearman_corr
    .reset_index()
    .melt(id_vars="index", var_name="variable_2", value_name="spearman_rho")
    .rename(columns={"index": "variable_1"})
)

with pd.ExcelWriter(corr_heatmap_data_path, engine="openpyxl") as writer:
    spearman_corr.reset_index().rename(columns={"index": "variable"}).to_excel(
        writer,
        sheet_name="matrix_original",
        index=False,
    )
    spearman_corr_ru.reset_index().rename(columns={"index": "variable"}).to_excel(
        writer,
        sheet_name="matrix_ru",
        index=False,
    )
    corr_long.to_excel(writer, sheet_name="long", index=False)

thesis_corr_heatmap_path = TABLES_DIR / "table_correlation_matrix_11_indices.xlsx"
spearman_corr_ru.reset_index().rename(columns={"index": "variable"}).to_excel(
    thesis_corr_heatmap_path,
    index=False,
)

print("✅ Correlation heatmap 11 × 11 построен.")
print(f"spearman_corr shape: {spearman_corr.shape}")
print(f"Plotly HTML: {html_path}")
print(f"Plotly PNG: {png_path} | {png_status}")
print(f"Matplotlib PNG: {mpl_path}")
print(f"Данные графика: {corr_heatmap_data_path}")
print(f"Таблица: {thesis_corr_heatmap_path}")

fig.show()

In [ ]:
# ============================================================
# ЯЧЕЙКА 32. Экспорт финального аналитического датасета и компактных таблиц
# ============================================================

# Автономность ячейки
if "df_final" not in globals():
    df_final = pd.read_parquet(DATA_DIR / "09_final_indices.parquet")

# ------------------------------------------------------------
# 1. Полный parquet уже сохранён в ячейке 14.
# Здесь делаем компактный Excel/CSV без тяжёлых tokens и lemma_text.
# ------------------------------------------------------------

heavy_cols = [
    "tokens",
    "lemma_text",
    "description",
]

slim_cols_to_drop = [c for c in heavy_cols if c in df_final.columns]

df_export = df_final.drop(columns=slim_cols_to_drop).copy()

# Для Excel оставим preview описания, чтобы файл не был чрезмерно тяжёлым
df_export_excel = df_export.copy()
df_export_excel["description_preview_1000"] = (
    df_export_excel["description_original"]
    .astype(str)
    .str.slice(0, 1000)
)

if "description_original" in df_export_excel.columns:
    df_export_excel = df_export_excel.drop(columns=["description_original"])

final_dataset_csv_path = DATA_DIR / "final_analytical_dataset_slim.csv"
final_dataset_excel_path = TABLES_DIR / "final_analytical_dataset_slim.xlsx"

df_export.to_csv(final_dataset_csv_path, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(final_dataset_excel_path, engine="openpyxl") as writer:
    df_export_excel.to_excel(writer, sheet_name="final_dataset_slim", index=False)

# ------------------------------------------------------------
# 2. Компактная сводка по группам для главы эмпирики
# ------------------------------------------------------------

summary_group_vars = [
    "profession_group",
    "creative_segment",
    "city_group",
    "experience_text",
    "quadrant_type",
    "ai_mode",
]

summary_metrics = [
    "support_total",
    "control_total",
    "semantic_balance",
    "combined_intensity",
    "autonomy_emb",
    "competence_emb",
    "relatedness_emb",
    "meaning_emb",
    "authorship_emb",
    "output_control_emb",
    "behavior_control_emb",
    "intensity_emb",
    "selfbrand_emb",
    "ai_empowerment_emb",
    "ai_intensification_emb",
]

summary_tables = {}

for group_col in summary_group_vars:
    temp = df_final.dropna(subset=[group_col]).copy()

    table = (
        temp
        .groupby(group_col)
        .agg(
            n=("vacancy_id", "count"),
            ai_share=("ai_mention", "mean"),
            salary_mid_rub_n=("salary_mid", lambda s: int(s.notna().sum())),
        )
        .reset_index()
    )

    for metric in summary_metrics:
        metric_table = (
            temp
            .groupby(group_col)[metric]
            .agg(
                **{
                    f"{metric}_median": "median",
                    f"{metric}_mean": "mean",
                }
            )
            .reset_index()
        )
        table = table.merge(metric_table, on=group_col, how="left")

    summary_tables[group_col] = table

compact_tables_path = TABLES_DIR / "compact_group_summaries_for_thesis.xlsx"

with pd.ExcelWriter(compact_tables_path, engine="openpyxl") as writer:
    for group_col, table in summary_tables.items():
        table.to_excel(writer, sheet_name=group_col[:31], index=False)

print("✅ Финальный аналитический датасет и компактные таблицы экспортированы.")
print(f"df_export shape: {df_export.shape}")
print(f"CSV: {final_dataset_csv_path}")
print(f"Excel: {final_dataset_excel_path}")
print(f"Компактные групповые таблицы: {compact_tables_path}")

In [ ]:
# ============================================================
# ЯЧЕЙКА 33. README.md + архив output.zip
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import shutil
from datetime import datetime

# ------------------------------------------------------------
# Пути
# ------------------------------------------------------------

if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = Path("/content/output")
if "DATA_DIR" not in globals():
    DATA_DIR = OUTPUT_DIR / "data"
if "STATS_DIR" not in globals():
    STATS_DIR = OUTPUT_DIR / "stats"
if "VALIDATION_DIR" not in globals():
    VALIDATION_DIR = OUTPUT_DIR / "validation"
if "FIGURES_DIR" not in globals():
    FIGURES_DIR = OUTPUT_DIR / "figures"
if "TABLES_DIR" not in globals():
    TABLES_DIR = OUTPUT_DIR / "tables_for_thesis"
if "REFERENCE_DIR" not in globals():
    REFERENCE_DIR = OUTPUT_DIR / "reference"

for folder in [OUTPUT_DIR, DATA_DIR, STATS_DIR, VALIDATION_DIR, FIGURES_DIR, TABLES_DIR, REFERENCE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Загрузка финального датасета
# ------------------------------------------------------------

if "df_final" not in globals():
    final_parquet_path = DATA_DIR / "09_final_indices.parquet"
    if not final_parquet_path.exists():
        raise FileNotFoundError("Не найден /content/output/data/09_final_indices.parquet. Сначала запустите ячейки до 14.")
    df_final = pd.read_parquet(final_parquet_path)

# ------------------------------------------------------------
# Вспомогательные функции
# ------------------------------------------------------------

def safe_value(x, digits=4):
    if pd.isna(x):
        return "NA"
    if isinstance(x, (float, np.floating)):
        return f"{x:.{digits}f}"
    return str(x)

def counts_table(df, col, top_n=None):
    if col not in df.columns:
        return pd.DataFrame({"value": ["column_missing"], "n": [0], "share": [0.0]})

    out = (
        df[col]
        .fillna("не указано")
        .astype(str)
        .value_counts(dropna=False)
        .rename_axis(col)
        .reset_index(name="n")
    )
    out["share"] = out["n"] / len(df)

    if top_n is not None:
        out = out.head(top_n)

    return out

def df_to_markdown_table(df, max_rows=30):
    """
    Простой markdown без зависимости от tabulate.
    """
    if df is None or df.empty:
        return "_Нет данных._"

    df_show = df.head(max_rows).copy()

    for col in df_show.columns:
        if pd.api.types.is_float_dtype(df_show[col]):
            df_show[col] = df_show[col].map(lambda x: safe_value(x, 4))
        else:
            df_show[col] = df_show[col].astype(str)

    headers = list(df_show.columns)
    lines = []
    lines.append("| " + " | ".join(headers) + " |")
    lines.append("| " + " | ".join(["---"] * len(headers)) + " |")

    for _, row in df_show.iterrows():
        values = [str(row[col]).replace("\n", " ") for col in headers]
        lines.append("| " + " | ".join(values) + " |")

    return "\n".join(lines)

def file_exists_note(path):
    path = Path(path)
    return "есть" if path.exists() else "не найден"

# ------------------------------------------------------------
# Ключевые числа
# ------------------------------------------------------------

n_rows = len(df_final)
n_unique_vacancy_id = df_final["vacancy_id"].nunique(dropna=True) if "vacancy_id" in df_final.columns else "NA"

salary_text_non_missing = int(df_final["salary_text"].notna().sum()) if "salary_text" in df_final.columns else 0
salary_text_share = salary_text_non_missing / n_rows if n_rows else np.nan

salary_mid_rub_n = int(((df_final.get("salary_currency", pd.Series(index=df_final.index)) == "RUB") & df_final.get("salary_mid", pd.Series(index=df_final.index)).notna()).sum()) if "salary_mid" in df_final.columns else 0

ai_mention_n = int(df_final["ai_mention"].sum()) if "ai_mention" in df_final.columns else 0
ai_mention_share = ai_mention_n / n_rows if n_rows else np.nan

semantic_balance_median = df_final["semantic_balance"].median() if "semantic_balance" in df_final.columns else np.nan
support_median = df_final["support_total"].median() if "support_total" in df_final.columns else np.nan
control_median = df_final["control_total"].median() if "control_total" in df_final.columns else np.nan

# ------------------------------------------------------------
# Таблицы распределений
# ------------------------------------------------------------

segment_counts_md = counts_table(df_final, "creative_segment")
profession_counts_md = counts_table(df_final, "profession_group")
city_counts_md = counts_table(df_final, "city_group")
quadrant_counts_md = counts_table(df_final, "quadrant_type")
ai_mode_counts_md = counts_table(df_final, "ai_mode")

# ------------------------------------------------------------
# Проверка ключевых файлов
# ------------------------------------------------------------

key_files = pd.DataFrame([
    {
        "file": "data/09_final_indices.parquet",
        "status": file_exists_note(DATA_DIR / "09_final_indices.parquet"),
        "description": "Финальный датасет с лексиконными, эмбеддинговыми и составными индексами.",
    },
    {
        "file": "stats/03_descriptive_statistics.xlsx",
        "status": file_exists_note(STATS_DIR / "03_descriptive_statistics.xlsx"),
        "description": "Описательная статистика и bootstrap CI.",
    },
    {
        "file": "stats/04_inferential_statistics.xlsx",
        "status": file_exists_note(STATS_DIR / "04_inferential_statistics.xlsx"),
        "description": "Mann–Whitney, Kruskal–Wallis, Dunn, Spearman.",
    },
    {
        "file": "validation/top_bottom_20_by_indices.xlsx",
        "status": file_exists_note(VALIDATION_DIR / "top_bottom_20_by_indices.xlsx"),
        "description": "Топ-20 вакансий на полюсах индексов.",
    },
    {
        "file": "validation/lexicon_embedding_correlation_matrix.xlsx",
        "status": file_exists_note(VALIDATION_DIR / "lexicon_embedding_correlation_matrix.xlsx"),
        "description": "Корреляции лексиконных и эмбеддинговых индексов.",
    },
    {
        "file": "reference/lexicons.json",
        "status": file_exists_note(REFERENCE_DIR / "lexicons.json"),
        "description": "Редактируемые лексиконы.",
    },
    {
        "file": "reference/anchor_phrases.json",
        "status": file_exists_note(REFERENCE_DIR / "anchor_phrases.json"),
        "description": "Якорные фразы для эмбеддинговых центроидов.",
    },
])

# ------------------------------------------------------------
# README без рискованных тройных f-string внутри кода
# ------------------------------------------------------------

readme_parts = []

readme_parts.append("# Анализ мотивационного предложения в вакансиях креативных работников\n")

readme_parts.append(
    "Этот каталог содержит результаты эмпирического анализа базы вакансий hh.ru "
    "для курсовой работы по теме **«Мотивация креативных работников в фирмах: вызовы и возможности»**.\n"
)

readme_parts.append(
    "Важно: анализируется не реальная мотивация работников, а **мотивационное предложение работодателя** — "
    "то есть сигналы, которые работодатель транслирует кандидату в тексте вакансии.\n"
)

readme_parts.append("## 1. Ключевые параметры датасета\n")

readme_parts.append(f"- Финальное число вакансий после очистки: **{n_rows}**.\n")
readme_parts.append(f"- Уникальных `vacancy_id`: **{n_unique_vacancy_id}**.\n")
readme_parts.append(f"- Вакансий с заполненным `salary_text`: **{salary_text_non_missing}** ({safe_value(salary_text_share * 100, 2)}%).\n")
readme_parts.append(f"- Вакансий с рассчитанным `salary_mid` в RUB: **{salary_mid_rub_n}**.\n")
readme_parts.append(f"- Вакансий с упоминанием AI/ИИ: **{ai_mention_n}** ({safe_value(ai_mention_share * 100, 2)}%).\n")
readme_parts.append(f"- Медиана `support_total`: **{safe_value(support_median)}**.\n")
readme_parts.append(f"- Медиана `control_total`: **{safe_value(control_median)}**.\n")
readme_parts.append(f"- Медиана `semantic_balance`: **{safe_value(semantic_balance_median)}**.\n")

readme_parts.append("\n## 2. Методологическая оговорка\n")

readme_parts.append(
    "Индексы не измеряют внутреннее психологическое состояние работников. "
    "Они отражают структуру вакансии как текста: какие типы мотивационных обещаний, требований, ограничений "
    "и сигналов контроля работодатель включает в описание работы.\n"
)

readme_parts.append(
    "Зарплатный анализ является разведочным, потому что поле `salary_text` заполнено только у малой доли вакансий. "
    "Поэтому зарплатные результаты нельзя интерпретировать как репрезентативное описание рынка.\n"
)

readme_parts.append("\n## 3. Индексы\n")

readme_parts.append(
    "Используются 11 содержательных конструктов:\n\n"
    "1. `autonomy` — автономия и свобода действий.\n"
    "2. `competence` — развитие, обучение, профессиональный рост.\n"
    "3. `relatedness` — команда, поддержка, культура взаимодействия.\n"
    "4. `meaning` — смысл, миссия, вклад, социальная или культурная значимость.\n"
    "5. `authorship` — авторство, портфолио, признание, индивидуальный стиль.\n"
    "6. `output_control` — KPI, метрики, результативность, отчётность.\n"
    "7. `behavior_control` — ТЗ, брендбук, гайдлайны, регламенты, дедлайны.\n"
    "8. `intensity` — высокий темп, многозадачность, нагрузка, стрессоустойчивость.\n"
    "9. `selfbrand` — личный бренд, публичность, соцсети, экспертная медийность.\n"
    "10. `ai_empowerment` — AI как расширение возможностей специалиста.\n"
    "11. `ai_intensification` — AI как средство ускорения, масштабирования и интенсификации труда.\n"
)

readme_parts.append("\n## 4. Основные составные переменные\n")

readme_parts.append(
    "- `support_total` — сумма z-score по supportive-индексам: autonomy, competence, relatedness, meaning, authorship.\n"
    "- `control_total` — сумма z-score по control-индексам: output_control, behavior_control, intensity.\n"
    "- `semantic_balance` — разница между support_total и control_total.\n"
    "- `combined_intensity` — объединённый показатель интенсивности по лексиконному и эмбеддинговому сигналам.\n"
    "- `quadrant_type` — тип вакансии в квадрантной схеме support × control.\n"
    "- `ai_mode` — способ включения AI: empowerment, intensification, mixed, no_ai.\n"
)

readme_parts.append("\n## 5. Распределение по creative_segment\n\n")
readme_parts.append(df_to_markdown_table(segment_counts_md))

readme_parts.append("\n\n## 6. Распределение по profession_group\n\n")
readme_parts.append(df_to_markdown_table(profession_counts_md))

readme_parts.append("\n\n## 7. Распределение по city_group\n\n")
readme_parts.append(df_to_markdown_table(city_counts_md))

readme_parts.append("\n\n## 8. Распределение по quadrant_type\n\n")
readme_parts.append(df_to_markdown_table(quadrant_counts_md))

readme_parts.append("\n\n## 9. Распределение по ai_mode\n\n")
readme_parts.append(df_to_markdown_table(ai_mode_counts_md))

readme_parts.append("\n\n## 10. Структура папок\n")

readme_parts.append(
    "```text\n"
    "output/\n"
    "├── data/               # parquet/csv с промежуточными и финальными данными\n"
    "├── stats/              # описательная и инференциальная статистика\n"
    "├── validation/         # проверки качества индексов и устойчивости\n"
    "├── figures/            # графики в HTML и PNG\n"
    "├── tables_for_thesis/  # компактные таблицы\n"
    "└── reference/          # лексиконы, якорные фразы, маппинг профессий\n"
    "```\n"
)

readme_parts.append("\n## 11. Ключевые файлы\n\n")
readme_parts.append(df_to_markdown_table(key_files))

readme_parts.append("\n\n## 12. Как интерпретировать результаты\n")

readme_parts.append(
    "1. Сначала смотреть описательную статистику и медианы индексов по группам.\n"
    "2. Затем проверять инференциальные результаты только вместе с эффект-сайзами.\n"
    "3. Не делать выводы только по p-value: статистическая значимость при большом n может возникать даже при слабых эффектах.\n"
    "4. Для содержательной интерпретации использовать validation-файлы с top/bottom вакансиями по каждому индексу.\n"
    "5. Зарплатный блок использовать только как дополнительное наблюдение, а не как центральный результат.\n"
)

readme_parts.append("\n## 13. Дата генерации\n")
readme_parts.append(f"\nREADME сгенерирован: **{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}**.\n")

readme_text = "\n".join(readme_parts)

# ------------------------------------------------------------
# Сохранение README
# ------------------------------------------------------------

readme_path = OUTPUT_DIR / "README.md"

with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme_text)

# ------------------------------------------------------------
# Создание zip-архива
# ------------------------------------------------------------

zip_base = "/content/output"
zip_path = "/content/output.zip"

if Path(zip_path).exists():
    Path(zip_path).unlink()

shutil.make_archive(zip_base, "zip", OUTPUT_DIR)

print("✅ README.md создан.")
print(f"README: {readme_path}")
print("✅ Архив output.zip создан.")
print(f"ZIP: {zip_path}")
print(f"df_final shape: {df_final.shape}")
print(f"Папка output: {OUTPUT_DIR}")

✅ README.md создан.
README: /content/output/README.md
✅ Архив output.zip создан.
ZIP: /content/output.zip
df_final shape: (3121, 110)
Папка output: /content/output


In [ ]:
# ============================================================
# ЯЧЕЙКА 34. Финальная проверка результатов и скачивание output.zip
# ============================================================

from pathlib import Path
import shutil
import pandas as pd

# ------------------------------------------------------------
# Пути
# ------------------------------------------------------------

OUTPUT_DIR = Path("/content/output")
DATA_DIR = OUTPUT_DIR / "data"
STATS_DIR = OUTPUT_DIR / "stats"
VALIDATION_DIR = OUTPUT_DIR / "validation"
FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR = OUTPUT_DIR / "tables_for_thesis"
REFERENCE_DIR = OUTPUT_DIR / "reference"

ZIP_PATH = Path("/content/output.zip")

# ------------------------------------------------------------
# Проверка ключевых файлов
# ------------------------------------------------------------

key_files = [
    OUTPUT_DIR / "README.md",
    DATA_DIR / "09_final_indices.parquet",
    STATS_DIR / "03_descriptive_statistics.xlsx",
    STATS_DIR / "04_inferential_statistics.xlsx",
    REFERENCE_DIR / "lexicons.json",
    REFERENCE_DIR / "anchor_phrases.json",
    REFERENCE_DIR / "profession_mapping.csv",
]

check_rows = []

for path in key_files:
    check_rows.append({
        "path": str(path),
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / 1024 / 1024, 3) if path.exists() else None,
    })

check_df = pd.DataFrame(check_rows)

# ------------------------------------------------------------
# Подсчёт файлов по папкам
# ------------------------------------------------------------

folder_rows = []

for folder in [DATA_DIR, STATS_DIR, VALIDATION_DIR, FIGURES_DIR, TABLES_DIR, REFERENCE_DIR]:
    if folder.exists():
        files = [p for p in folder.rglob("*") if p.is_file()]
        total_size_mb = sum(p.stat().st_size for p in files) / 1024 / 1024
        folder_rows.append({
            "folder": str(folder),
            "n_files": len(files),
            "total_size_mb": round(total_size_mb, 2),
        })
    else:
        folder_rows.append({
            "folder": str(folder),
            "n_files": 0,
            "total_size_mb": 0,
        })

folder_df = pd.DataFrame(folder_rows)

# ------------------------------------------------------------
# Пересоздание архива
# ------------------------------------------------------------

if not OUTPUT_DIR.exists():
    raise FileNotFoundError("Папка /content/output не найдена. Сначала запустите предыдущие ячейки.")

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

shutil.make_archive("/content/output", "zip", OUTPUT_DIR)

# ------------------------------------------------------------
# Итоговый отчёт
# ------------------------------------------------------------

print("✅ Финальная проверка выполнена.")
print(f"Папка output: {OUTPUT_DIR}")
print(f"Архив создан: {ZIP_PATH}")
print(f"Размер архива: {round(ZIP_PATH.stat().st_size / 1024 / 1024, 2)} MB")

print("\nПроверка ключевых файлов:")
display(check_df)

print("\nСводка по папкам:")
display(folder_df)

missing = check_df[check_df["exists"] == False]

if len(missing) > 0:
    print("⚠️ Внимание: некоторые ключевые файлы не найдены.")
    display(missing)
else:
    print("✅ Все ключевые файлы найдены.")

# ------------------------------------------------------------
# Автоматическое скачивание архива
# ------------------------------------------------------------
# Если скачивание не началось автоматически, скачайте output.zip вручную через левую панель Files.

try:
    from google.colab import files
    files.download(str(ZIP_PATH))
    print("✅ Скачивание output.zip запущено.")
except Exception as e:
    print("⚠️ Автоматическое скачивание не запустилось.")
    print("Скачайте файл вручную из левой панели Colab: /content/output.zip")
    print(f"Причина: {repr(e)}")

✅ Финальная проверка выполнена.
Папка output: /content/output
Архив создан: /content/output.zip
Размер архива: 174.74 MB

Проверка ключевых файлов:


,path,exists,size_mb
0,/content/output/README.md,True,0.008
1,/content/output/data/09_final_indices.parquet,True,19.903
2,/content/output/stats/03_descriptive_statistic...,True,0.128
3,/content/output/stats/04_inferential_statistic...,True,0.137
4,/content/output/reference/lexicons.json,True,0.007
5,/content/output/reference/anchor_phrases.json,True,0.028
6,/content/output/reference/profession_mapping.csv,True,0.003



Сводка по папкам:


,folder,n_files,total_size_mb
0,/content/output/data,22,186.25
1,/content/output/stats,4,0.28
2,/content/output/validation,11,1.27
3,/content/output/figures,31,57.08
4,/content/output/tables_for_thesis,10,7.65
5,/content/output/reference,8,0.07


✅ Все ключевые файлы найдены.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Скачивание output.zip запущено.


In [ ]:
# ============================================================
# ЯЧЕЙКА 35. Разведочный зарплатный анализ
# ============================================================
# Важно: salary_text заполнен только у небольшой доли вакансий.
# Поэтому этот блок является дополнительным / разведочным, а не центральным доказательством.

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

# ------------------------------------------------------------
# Пути
# ------------------------------------------------------------

OUTPUT_DIR = Path("/content/output")
DATA_DIR = OUTPUT_DIR / "data"
STATS_DIR = OUTPUT_DIR / "stats"
FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR = OUTPUT_DIR / "tables_for_thesis"

for folder in [OUTPUT_DIR, DATA_DIR, STATS_DIR, FIGURES_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Загрузка финального датасета
# ------------------------------------------------------------

if "df_final" not in globals():
    final_parquet_path = DATA_DIR / "09_final_indices.parquet"
    if not final_parquet_path.exists():
        raise FileNotFoundError("Не найден /content/output/data/09_final_indices.parquet. Сначала запустите ячейки до 14.")
    df_final = pd.read_parquet(final_parquet_path)

# ------------------------------------------------------------
# Подготовка зарплатного поднабора
# ------------------------------------------------------------

required_cols = [
    "vacancy_id",
    "query_keyword",
    "profession_group",
    "creative_segment",
    "city_group",
    "experience_text",
    "salary_text",
    "salary_from",
    "salary_to",
    "salary_currency",
    "salary_gross",
    "salary_mid",
    "salary_parse_note",
    "semantic_balance",
    "support_total",
    "control_total",
]

missing_cols = [c for c in required_cols if c not in df_final.columns]
if missing_cols:
    raise ValueError(f"В df_final отсутствуют нужные колонки: {missing_cols}")

salary_all = df_final[required_cols].copy()

salary_all["salary_mid"] = pd.to_numeric(salary_all["salary_mid"], errors="coerce")
salary_all["salary_from"] = pd.to_numeric(salary_all["salary_from"], errors="coerce")
salary_all["salary_to"] = pd.to_numeric(salary_all["salary_to"], errors="coerce")

# Основной зарплатный анализ — только RUB и только там, где salary_mid рассчитан по двум границам
salary_df = salary_all[
    (salary_all["salary_currency"] == "RUB") &
    (salary_all["salary_mid"].notna())
].copy()

# Чистим категориальные поля
for col in ["profession_group", "creative_segment", "city_group", "experience_text", "query_keyword"]:
    salary_df[col] = salary_df[col].astype("object").where(salary_df[col].notna(), "не указано").astype(str)

# ------------------------------------------------------------
# Общая сводка
# ------------------------------------------------------------

salary_summary = pd.DataFrame({
    "metric": [
        "rows_total_final_dataset",
        "salary_text_non_missing",
        "salary_text_non_missing_share",
        "salary_mid_rub_available",
        "salary_mid_rub_available_share",
        "salary_mid_rub_median",
        "salary_mid_rub_mean",
        "salary_mid_rub_q1",
        "salary_mid_rub_q3",
        "salary_mid_rub_min",
        "salary_mid_rub_max",
    ],
    "value": [
        len(df_final),
        int(df_final["salary_text"].notna().sum()),
        float(df_final["salary_text"].notna().mean()),
        len(salary_df),
        len(salary_df) / len(df_final) if len(df_final) else np.nan,
        float(salary_df["salary_mid"].median()) if len(salary_df) else np.nan,
        float(salary_df["salary_mid"].mean()) if len(salary_df) else np.nan,
        float(salary_df["salary_mid"].quantile(0.25)) if len(salary_df) else np.nan,
        float(salary_df["salary_mid"].quantile(0.75)) if len(salary_df) else np.nan,
        float(salary_df["salary_mid"].min()) if len(salary_df) else np.nan,
        float(salary_df["salary_mid"].max()) if len(salary_df) else np.nan,
    ],
})

salary_warning = pd.DataFrame({
    "warning": [
        "Зарплатный анализ является разведочным.",
        "salary_text заполнен только у небольшой доли вакансий.",
        "salary_mid рассчитывается только при наличии обеих границ salary_from и salary_to.",
        "Групповые сравнения показываются только для групп с n >= 20 зарплатных наблюдений.",
        "salary_relative_to_city_median не рассчитывается для city_group с числом зарплатных наблюдений меньше 20.",
    ]
})

# ------------------------------------------------------------
# Функция групповой зарплатной статистики
# ------------------------------------------------------------

def salary_group_stats(data, group_col, min_n=20):
    if data.empty:
        return pd.DataFrame()

    out = (
        data
        .groupby(group_col, dropna=False)
        .agg(
            n=("salary_mid", "count"),
            salary_mid_median=("salary_mid", "median"),
            salary_mid_mean=("salary_mid", "mean"),
            salary_mid_q1=("salary_mid", lambda x: x.quantile(0.25)),
            salary_mid_q3=("salary_mid", lambda x: x.quantile(0.75)),
            salary_mid_min=("salary_mid", "min"),
            salary_mid_max=("salary_mid", "max"),
            semantic_balance_median=("semantic_balance", "median"),
            support_total_median=("support_total", "median"),
            control_total_median=("control_total", "median"),
        )
        .reset_index()
    )

    out["iqr"] = out["salary_mid_q3"] - out["salary_mid_q1"]
    out["included_in_interpretation"] = out["n"] >= min_n

    return out.sort_values(["included_in_interpretation", "n", "salary_mid_median"], ascending=[False, False, False])

salary_by_profession = salary_group_stats(salary_df, "profession_group", min_n=20)
salary_by_segment = salary_group_stats(salary_df, "creative_segment", min_n=20)
salary_by_city = salary_group_stats(salary_df, "city_group", min_n=20)
salary_by_experience = salary_group_stats(salary_df, "experience_text", min_n=20)

# ------------------------------------------------------------
# salary_relative_to_city_median только для city_group с n >= 20
# ------------------------------------------------------------

city_salary_counts = salary_df["city_group"].value_counts()
valid_salary_cities = city_salary_counts[city_salary_counts >= 20].index.tolist()

salary_df["city_salary_group_valid"] = salary_df["city_group"].isin(valid_salary_cities)

city_medians = (
    salary_df[salary_df["city_salary_group_valid"]]
    .groupby("city_group")["salary_mid"]
    .median()
    .to_dict()
)

salary_df["city_salary_median"] = salary_df["city_group"].map(city_medians)
salary_df["salary_relative_to_city_median"] = np.where(
    salary_df["city_salary_group_valid"],
    salary_df["salary_mid"] / salary_df["city_salary_median"],
    np.nan
)

# ------------------------------------------------------------
# Сохранение Excel
# ------------------------------------------------------------

salary_output_path = STATS_DIR / "05_salary_exploratory_analysis.xlsx"

with pd.ExcelWriter(salary_output_path, engine="openpyxl") as writer:
    salary_warning.to_excel(writer, sheet_name="warning", index=False)
    salary_summary.to_excel(writer, sheet_name="overall_summary", index=False)
    salary_df.to_excel(writer, sheet_name="salary_observations", index=False)
    salary_by_profession.to_excel(writer, sheet_name="by_profession", index=False)
    salary_by_segment.to_excel(writer, sheet_name="by_segment", index=False)
    salary_by_city.to_excel(writer, sheet_name="by_city_group", index=False)
    salary_by_experience.to_excel(writer, sheet_name="by_experience", index=False)

# Компактная таблица
salary_thesis_path = TABLES_DIR / "appendix_salary_exploratory_tables.xlsx"

with pd.ExcelWriter(salary_thesis_path, engine="openpyxl") as writer:
    salary_summary.to_excel(writer, sheet_name="salary_summary", index=False)
    salary_by_profession.query("included_in_interpretation == True").to_excel(
        writer,
        sheet_name="profession_n20",
        index=False,
    )
    salary_by_segment.query("included_in_interpretation == True").to_excel(
        writer,
        sheet_name="segment_n20",
        index=False,
    )
    salary_by_city.query("included_in_interpretation == True").to_excel(
        writer,
        sheet_name="city_n20",
        index=False,
    )

# ------------------------------------------------------------
# Графики
# ------------------------------------------------------------

# 1. Histogram salary_mid
if len(salary_df) > 0:
    fig_hist = px.histogram(
        salary_df,
        x="salary_mid",
        nbins=30,
        title="Распределение salary_mid, RUB: разведочный анализ",
        labels={"salary_mid": "salary_mid, RUB"},
        template="plotly_white",
    )

    fig_hist.add_vline(
        x=salary_df["salary_mid"].median(),
        line_dash="dash",
        annotation_text="median",
        annotation_position="top",
    )

    html_hist_path = FIGURES_DIR / "salary_histogram_rub.html"
    png_hist_path = FIGURES_DIR / "salary_histogram_rub.png"

    fig_hist.write_html(str(html_hist_path))

    try:
        fig_hist.write_image(str(png_hist_path), width=1100, height=700, scale=2)
        png_hist_status = "saved"
    except Exception as e:
        png_hist_status = f"png_not_saved: {repr(e)}"

    # Matplotlib duplicate
    mpl_hist_path = FIGURES_DIR / "salary_histogram_rub_matplotlib.png"

    plt.figure(figsize=(10, 6))
    plt.hist(salary_df["salary_mid"].dropna(), bins=30)
    plt.axvline(salary_df["salary_mid"].median(), linestyle="--")
    plt.xlabel("salary_mid, RUB")
    plt.ylabel("Количество вакансий")
    plt.title("Распределение salary_mid, RUB: разведочный анализ")
    plt.tight_layout()
    plt.savefig(mpl_hist_path, dpi=300, bbox_inches="tight")
    plt.close()
else:
    html_hist_path = None
    png_hist_path = None
    png_hist_status = "no_salary_data"
    mpl_hist_path = None

# 2. Bar chart median salary by profession_group, только n >= 20
salary_prof_plot = salary_by_profession.query("included_in_interpretation == True").copy()

if len(salary_prof_plot) > 0:
    fig_prof = px.bar(
        salary_prof_plot,
        x="salary_mid_median",
        y="profession_group",
        orientation="h",
        text="n",
        title="Медианная salary_mid по profession_group, только группы n ≥ 20",
        labels={
            "salary_mid_median": "Медианная salary_mid, RUB",
            "profession_group": "Profession group",
            "n": "n",
        },
        template="plotly_white",
    )

    fig_prof.update_layout(height=max(500, 40 * len(salary_prof_plot)))

    html_prof_path = FIGURES_DIR / "salary_median_by_profession_n20.html"
    png_prof_path = FIGURES_DIR / "salary_median_by_profession_n20.png"

    fig_prof.write_html(str(html_prof_path))

    try:
        fig_prof.write_image(str(png_prof_path), width=1100, height=max(600, 50 * len(salary_prof_plot)), scale=2)
        png_prof_status = "saved"
    except Exception as e:
        png_prof_status = f"png_not_saved: {repr(e)}"

    mpl_prof_path = FIGURES_DIR / "salary_median_by_profession_n20_matplotlib.png"

    plt.figure(figsize=(10, max(5, 0.45 * len(salary_prof_plot))))
    plt.barh(salary_prof_plot["profession_group"], salary_prof_plot["salary_mid_median"])
    plt.xlabel("Медианная salary_mid, RUB")
    plt.ylabel("Profession group")
    plt.title("Медианная salary_mid по profession_group, только группы n ≥ 20")
    plt.tight_layout()
    plt.savefig(mpl_prof_path, dpi=300, bbox_inches="tight")
    plt.close()
else:
    html_prof_path = None
    png_prof_path = None
    png_prof_status = "no_groups_with_n20"
    mpl_prof_path = None

# ------------------------------------------------------------
# Вывод
# ------------------------------------------------------------

print("✅ Разведочный зарплатный анализ выполнен.")
print(f"Всего вакансий в финальном датасете: {len(df_final)}")
print(f"salary_text заполнен: {int(df_final['salary_text'].notna().sum())} из {len(df_final)} ({df_final['salary_text'].notna().mean():.2%})")
print(f"salary_mid RUB доступен: {len(salary_df)} из {len(df_final)} ({len(salary_df) / len(df_final):.2%})")
print("⚠️ Интерпретация: зарплатный блок использовать только как приложение / разведочный анализ.")
print(f"Excel сохранён: {salary_output_path}")
print(f"Таблица для приложения сохранена: {salary_thesis_path}")
print(f"Histogram HTML: {html_hist_path}")
print(f"Histogram PNG: {png_hist_path} | {png_hist_status}")
print(f"Profession salary HTML: {html_prof_path}")
print(f"Profession salary PNG: {png_prof_path} | {png_prof_status}")

display(salary_summary)
display(salary_by_profession.head(20))

✅ Разведочный зарплатный анализ выполнен.
Всего вакансий в финальном датасете: 3121
salary_text заполнен: 126 из 3121 (4.04%)
salary_mid RUB доступен: 58 из 3121 (1.86%)
⚠️ Интерпретация: зарплатный блок использовать только как приложение / разведочный анализ.
Excel сохранён: /content/output/stats/05_salary_exploratory_analysis.xlsx
Таблица для приложения сохранена: /content/output/tables_for_thesis/appendix_salary_exploratory_tables.xlsx
Histogram HTML: /content/output/figures/salary_histogram_rub.html
Histogram PNG: /content/output/figures/salary_histogram_rub.png | png_not_saved: ValueError('\nImage export using the "kaleido" engine requires the kaleido package,\nwhich can be installed using pip:\n    $ pip install -U kaleido\n')
Profession salary HTML: /content/output/figures/salary_median_by_profession_n20.html
Profession salary PNG: /content/output/figures/salary_median_by_profession_n20.png | png_not_saved: ValueError('\nImage export using the "kaleido" engine requires the kalei

,metric,value
0,rows_total_final_dataset,3121.000000
1,salary_text_non_missing,126.000000
2,salary_text_non_missing_share,0.040372
3,salary_mid_rub_available,58.000000
4,salary_mid_rub_available_share,0.018584
5,salary_mid_rub_median,80000.000000
6,salary_mid_rub_mean,85347.500000
7,salary_mid_rub_q1,62625.000000
8,salary_mid_rub_q3,100000.000000
9,salary_mid_rub_min,105.000000


,profession_group,n,salary_mid_median,salary_mid_mean,salary_mid_q1,salary_mid_q3,salary_mid_min,salary_mid_max,semantic_balance_median,support_total_median,control_total_median,iqr,included_in_interpretation
0,design_visual,58,80000.0,85347.5,62625.0,100000.0,105.0,175000.0,0.602885,1.482978,0.595821,37375.0,True
